<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [42]</a>'.</span>

In [1]:
Month = 2
Year = 2569
session = 'g'

name_series = "SeriesG2569-1_.xlsx"
excel_file_path = 'ไฟล์ราคากลาง ชุด บ_ก.พ. 69.xlsx' #ไฟล์ราคากลาง ชุด บ_พ.ย. 68
right_price = 'บันทึกราคาถูกต้อง.xlsx'
masterF = 'Real_Master_CPI.xlsx'

Web_Data_SsoTpso = 1 #อยากให้รันข้อมูลสินค้นบนเว็บ ให้กดหนึ่ง ให้กดศูนย์
run_cpi_month = 1 # อยากให้รัน ไฟล์cpi_month ให้กดหนึ่ง ให้กดศูนย์
run_master_price = 1 #อยากให้รันราคากลาง ให้กดหนึ่ง ให้กดศูนย์
time_serie = 1 # อยากให้รัน ไฟล์time_serie ให้กดหนึ่ง ให้กดศูนย์
Web_Shop_12C = 1 #ร้านกรุงเทพ ให้กดหนึ่ง ให้กดศูนย์

User = ['All'] #All #['เบส','เป๋าฮื้อ'] 

#### Detail

In [2]:
import os
import re
import numpy as np
import pandas as pd

# OpenPyXL Imports (สำหรับการจัดการ Excel ขั้นสูง)
import openpyxl
from openpyxl import Workbook
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl.utils import get_column_letter, column_index_from_string
from openpyxl.styles import PatternFill, Font, Border, Side, Alignment
from openpyxl.worksheet.filters import AutoFilter

# XlsxWriter (เผื่อใช้ แต่ Logic หลักในนี้จะใช้ OpenPyXL ตาม Code เดิม)
import xlsxwriter

# กำหนด Path ปัจจุบัน
download_dir = os.getcwd()

In [3]:
if session == 'g':
    CPI_file = "cpig_month_"+str(Year)+"-"+str(Month)+".xlsx"
elif session == 'u':
    CPI_file = "cpiu_month_"+str(Year)+"-"+str(Month)+".xlsx"
time = CPI_file.split('_')[2].replace('.xlsx','')
time = '_'+time

filename = CPI_file.replace('.xlsx', '')
filename 

'cpig_month_2569-2'

#### WEB

In [4]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
import os, time

if Web_Data_SsoTpso==1:
    # ===== ฟังก์ชันรอจนไฟล์โหลดเสร็จ =====
    def wait_for_downloads(folder_path, timeout=120):
        """รอจนไม่มีไฟล์ .crdownload ในโฟลเดอร์"""
        start_time = time.time()
        while True:
            # ถ้ายังมีไฟล์ .crdownload แสดงว่ายังโหลดไม่เสร็จ
            downloading = any(f.endswith('.crdownload') for f in os.listdir(folder_path))
            if not downloading:
                time.sleep(15)
                break
            if time.time() - start_time > timeout:
                raise TimeoutError("⏰ โหลดไฟล์ไม่เสร็จภายในเวลาที่กำหนด")
            time.sleep(1)

    # ===== ตั้งค่า Chrome ให้โหลดไฟล์มาที่โฟลเดอร์ปัจจุบัน =====
    download_dir = os.getcwd()
    chrome_options = webdriver.ChromeOptions()
    prefs = {
        "download.default_directory": download_dir,
        "download.prompt_for_download": False,
        "download.directory_upgrade": True,
        "safebrowsing.enabled": True
    }
    chrome_options.add_experimental_option("prefs", prefs)

    # ===== ลบไฟล์ Excel เดิมทั้งหมดก่อนเริ่มดาวน์โหลด =====
    for f in os.listdir(download_dir):
        # เช็คชื่อหน้า: ขึ้นต้นด้วย "cpi" หรือ "Unconfirmed"
        # เช็คชื่อหลัง: ลงท้ายด้วย ".xlsx" หรือ ".tmp" หรือ ".crdownload"
        if f.startswith(("cpi", "Unconfirmed")) and f.endswith((".xlsx", ".tmp", ".crdownload")):
            try:
                os.remove(os.path.join(download_dir, f))
                print(f"ลบไฟล์เก่า: {f}")
            except Exception as e:
                print(f"ไม่สามารถลบ {f}: {e}")

    # ===== เริ่มเปิดเบราว์เซอร์ =====
    driver = webdriver.Chrome(options=chrome_options)
    driver.get("https://sso-tpso.moc.go.th/Login?ReturnUrl=%2F")

    # ===== เข้าระบบ =====
    driver.find_element(By.ID, "Username").send_keys("tanaponp")
    driver.find_element(By.ID, "Password").send_keys("&5#t2B3&")
    driver.find_element(By.ID, "Password").send_keys(Keys.RETURN)
    time.sleep(2)

    # ===== ไปหน้าที่มีปุ่ม cpig month =====
    driver.get("https://dev-tpso.moc.go.th/export12c/cpi")
    time.sleep(2)

    driver.execute_script(f"document.getElementById('ddlMonth').value = '{Month}';")
    driver.execute_script(f"document.getElementById('ddlYear').value = '{Year}';")

    # ===== คลิกปุ่ม cpig month =====
    if session == 'g':
        driver.find_element(By.ID, "btnCpigMonth").click()
        print("📂 กำลังดาวน์โหลดไฟล์ cpig_month_2568-10.xlsx ...")
    elif session == 'u':
        driver.find_element(By.ID, "btnCpiuMonth").click()
        print("📂 กำลังดาวน์โหลดไฟล์ cpig_month_2568-10.xlsx ...")


    # ✅ แทน time.sleep(50)
    # wait_for_downloads(download_dir, timeout=180)
    time.sleep(35)

    # ===== ตรวจสอบว่ามีไฟล์อยู่ในโฟลเดอร์ =====
    files = [f for f in os.listdir(download_dir) if f.endswith(".xlsx")]
    print("✅ ไฟล์ที่โหลดแล้ว:", files)

    driver.quit()


ลบไฟล์เก่า: cpig_month_2569-2.xlsx
ลบไฟล์เก่า: cpig_month_2569-2_ExWeb_Final_Clean.xlsx
ลบไฟล์เก่า: cpig_month_2569-2_MainBefore_exweb.xlsx
ลบไฟล์เก่า: cpig_month_2569-2_multi_sheet.xlsx
ลบไฟล์เก่า: cpig_month_2569-2_ราคากลาง_คุณต้น.xlsx
ลบไฟล์เก่า: cpig_month_2569-2_ราคากลาง_ส่วนกลาง.xlsx
ลบไฟล์เก่า: cpig_month_2569-2_สินค้ายอดนิยม.xlsx
ลบไฟล์เก่า: cpig_month_25696902_สินค้าหมด.xlsx
📂 กำลังดาวน์โหลดไฟล์ cpig_month_2568-10.xlsx ...
✅ ไฟล์ที่โหลดแล้ว: ['cpig_month_2569-2.xlsx', 'Final_Master_Report.xlsx', 'Final_Master_Report_Borders.xlsx', 'FULL.xlsx', 'Real_Master_CPI.xlsx', 'SeriesG2568-10_.xlsx.xlsx', 'SeriesG2568-11.xlsx', 'SeriesG2568-11_.xlsx.xlsx', 'SeriesG2569-1_.xlsx', 'SeriesU2568-10_.xlsx.xlsx', 'shop_data_20260210_165356.xlsx', 'TimeSeries_FULL.xlsx', 'Training_Data.xlsx', 'บันทึกราคาถูกต้อง.xlsx', 'รายการคำนวณและไม่คำนวณทุกชุด.xlsx', 'ไฟล์ราคากลาง ชุด บ_ก.พ. 69.xlsx', 'ไฟล์ราคากลาง ชุด บ_ม.ค. 69.xlsx']


In [5]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import os, time

if Web_Shop_12C == 1:
    # ===== ตั้งค่า Chrome ให้โหลดไฟล์ลง workspace ปัจจุบัน =====
    download_dir = os.getcwd()
    chrome_options = webdriver.ChromeOptions()
    prefs = {
        "download.default_directory": download_dir,
        "download.prompt_for_download": False,
        "download.directory_upgrade": True,
        "safebrowsing.enabled": True
    }
    chrome_options.add_experimental_option("prefs", prefs)

    # ===== ลบไฟล์ เดิมทั้งหมดก่อนเริ่มดาวน์โหลด =====
    for f in os.listdir(download_dir):
        if f.startswith("shop_data") & f.endswith(".xlsx"):
            try:
                os.remove(os.path.join(download_dir, f))
                print(f"ลบไฟล์เก่า: {f}")
            except Exception as e:
                print(f"ไม่สามารถลบ {f}: {e}")

    driver = webdriver.Chrome(options=chrome_options)

    # ===== เปิดหน้า login =====
    driver.get("https://cpi.moc.go.th/cpi/login.aspx")

    # ===== รอจนฟิลด์ username โหลดขึ้นมา =====
    WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.ID, "txtUser"))
    )

    # ===== กรอก username / password =====
    driver.find_element(By.ID, "txtUser").send_keys("index_bkk027")
    driver.find_element(By.ID, "txtPwd").send_keys("Tpsocpi1234")

    # ===== คลิกปุ่ม login =====
    driver.find_element(By.ID, "btnLogin").click()

    # ===== รอจนเข้าสู่ระบบสำเร็จ (หน้า login จะ redirect อัตโนมัติ) =====
    # ตรวจว่าหลุดจากหน้า login แล้ว
    WebDriverWait(driver, 20).until_not(
        EC.url_contains("login.aspx")
    )

    # ===== เข้าไปหน้าจัดการร้านค้า =====
    driver.get("https://cpi.moc.go.th/cpi/shopMgt.aspx")

    # ===== รอให้ปุ่ม Excel ปรากฏ =====
    WebDriverWait(driver, 15).until(
        EC.element_to_be_clickable((By.ID, "ContentPlaceHolder1_btnExcel"))
    )

    # ===== คลิกปุ่มเพื่อดาวน์โหลดไฟล์ Excel =====
    driver.find_element(By.ID, "ContentPlaceHolder1_btnExcel").click()

    print("📂 กำลังดาวน์โหลดไฟล์ร้านกรุงเทพ...")

    # ===== รอให้ไฟล์โหลดเสร็จ ===== 
    time.sleep(15)

    # ===== ตรวจสอบว่ามีไฟล์อยู่ในโฟลเดอร์ =====
    files = [f for f in os.listdir(download_dir) if f.endswith(".xlsx")]
    print("ไฟล์ที่โหลดแล้ว:", files)

    driver.quit()


ลบไฟล์เก่า: shop_data_20260210_165356.xlsx
📂 กำลังดาวน์โหลดไฟล์ร้านกรุงเทพ...
ไฟล์ที่โหลดแล้ว: ['cpig_month_2569-2.xlsx', 'Final_Master_Report.xlsx', 'Final_Master_Report_Borders.xlsx', 'FULL.xlsx', 'Real_Master_CPI.xlsx', 'SeriesG2568-10_.xlsx.xlsx', 'SeriesG2568-11.xlsx', 'SeriesG2568-11_.xlsx.xlsx', 'SeriesG2569-1_.xlsx', 'SeriesU2568-10_.xlsx.xlsx', 'shop_data_20260218_135233.xlsx', 'TimeSeries_FULL.xlsx', 'Training_Data.xlsx', 'บันทึกราคาถูกต้อง.xlsx', 'รายการคำนวณและไม่คำนวณทุกชุด.xlsx', 'ไฟล์ราคากลาง ชุด บ_ก.พ. 69.xlsx', 'ไฟล์ราคากลาง ชุด บ_ม.ค. 69.xlsx']


### FUNCTION

#### function1

In [6]:
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Alignment, Border, Side, PatternFill, Font
from openpyxl.utils import get_column_letter

def Excel_Condition2(df, filename,
                     columns_to_hide_by_name=None):
    """
    สร้างไฟล์ Excel พร้อมจัดรูปแบบ และซ่อนคอลัมน์ด้วย 'ชื่อคอลัมน์'
    - df: pandas.DataFrame
    - filename: ชื่อไฟล์ .xlsx
    - columns_to_hide_by_name: list[str] ของชื่อคอลัมน์ที่จะซ่อน (optional)
    """

    # # ----- เตรียม workbook / sheet -----
    wb = Workbook()
    ws = wb.active
    # ws = wb.create_sheet(title=sheet_name)
    # ws = wb.create_sheet(title=sheet_name)

    # ----- เขียนหัวตาราง -----
    headers = list(df.columns)
    ws.append(headers)

    # ----- เขียนข้อมูลทีละแถว (เริ่มที่แถว 2 เพราะแถว 1 เป็น header) -----
    for row_vals in df.itertuples(index=False, name=None):
        ws.append(row_vals)

    # ----- ฟังก์ชันช่วย: หา index คอลัมน์จากชื่อ (คืนค่าเป็น 1-based หรือ None) -----
    def col_idx(col_name: str):
        return headers.index(col_name) + 1 if col_name in headers else None

    # ----- ซ่อนคอลัมน์ด้วย 'ชื่อคอลัมน์' หลังจากที่มี header แล้ว -----
    if columns_to_hide_by_name is None:
        columns_to_hide_by_name = [
        'ผลการตรวจสอบModernTrade','ผลการตรวจสอบร้านอื่นๆ','CountAVG','AVGtail_stable','Report_Z>=2','Report_IQR','Z_Score_Local', 'Z_Score_Global','Sum_Z-Score','รายการ','จำนวนร้าน(ประเภทร้านค้า)',
        'PROVINCE_NAME','กลุ่ม', 'C_FLAG', 'G_FLAG','GROUP_KEY','P_FLAG','USERID',
       'L_FLAG', 'N_FLAG', 'R_FLAG', 'NR_FLAG', 'TYPE_CODE', 'ข้อความ0', 'ข้อความ1',
        'จำนวนสินค้า', 'Rank', 'ขนาดร้านค้า',
       'จำนวนร้าน(ขนาดร้านค้า)', 'PROVINCE_NAME', '[ภาค]', 
       'นับรายการเก็บต่อจังหวัด', 'ราคาที่นิยมเฉพาะร้าน',
       'ผลต่าง_ราคาที่นิยมเฉพาะร้าน', 'ฐานนิยม(ในสินค้า)',
       'ฐานนิยม(ในขนาดร้านค้า)', 'ฐานนิยม(ในประเภทร้านค้า)', 'ฐานนิยม(รวมค่า)',
       'ราคาอยู่ในช่วง', 'Z-Score_ทั้งสินค้า', 'Z-Score_เฉพาะแหล่งเดียวกัน',
       'Sum_Z-Score', 'Report Z >=2',  'ตีความComment1', 'check_max_ฐานนิยม(ในประเภทร้านค้า)', 'max', 'min',
       'รายงานจังหวัดที่ขาดส่ง', 'ราคาปกติ', 'ราคาโปร',  
       'นับการการแก้ไขตามจังหวัด',  'temp_diff',
       'AVG_Max', 'AVG_Min', 'AVG_maxdiff', 'AVG_mindiff',
       'Rel_Max', 'Rel_Min', 'Reltrail_eq100', 'ต่อเนื่องเกิน12เดือน',
        'MAX_DIFF', 'C diff A',
        'สรุปผลกระโดด','นับ_สรุปผลกระโดด','สรุปผลเกินขอบ','นับ_สรุปผลเกินขอบ','สรุปผลREL','นับ_สรุปผลREL','สรุปผล',
        'ราคากระโดดไกล','ราคาเกินช่วงสูงต่ำ','สถานะ','สรุปผล_จัดหมวด','AVG_ฐานนิยม','REL_ฐานนิยม','Bandรายละเอียด','การบรรจุ','ผลต่างราคากับราคากลาง'
        ,'TYPE','ผลต่างAVG','ผลต่างREL'
        ]        


    for name in columns_to_hide_by_name:
        idx = col_idx(name)
        if idx:
            ws.column_dimensions[get_column_letter(idx)].hidden = True

    # ----- จัดให้บางคอลัมน์ชิดกลาง (ถ้ามี) -----
    center_columns = ['LAST_PRICE','AVG','COMPARE_PRICE','LINK', 'REL', 'CODE_7_DIGITS', 'ราคากลาง', 'นับตัวซ้ำ']
    center_indices = [col_idx(c) for c in center_columns if col_idx(c)]
    if center_indices:
        for r in ws.iter_rows(min_row=2, max_row=ws.max_row):
            for ci in center_indices:
                r[ci-1].alignment = Alignment(horizontal='center')

    # ----- เส้นขอบแดช/บาง/หนา ตามเงื่อนไข -----
    dash_border  = Border(bottom=Side(style='dashed', color='FF6666'))
    thin_border  = Border(bottom=Side(style='thin'))
    thick_border = Border(bottom=Side(style='medium'))

    col_desc_th = col_idx('ประเภทร้านค้า')
    col_desc_en = col_idx('DESCRIPTION')
    col_code7   = col_idx('CODE_7_DIGITS')

    prev_desc_th = None
    prev_desc_en = None
    prev_code7   = None

    # เดินทีละแถวข้อมูลจริง (เริ่มที่แถว 2)
    for r in range(2, ws.max_row + 1):
        # ค่าปัจจุบันที่ใช้เปรียบเทียบ
        cur_desc_th = ws.cell(row=r, column=col_desc_th).value if col_desc_th else None
        cur_desc_en = ws.cell(row=r, column=col_desc_en).value if col_desc_en else None
        cur_code7   = ws.cell(row=r, column=col_code7).value   if col_code7   else None

        # ถ้า 'ประเภทร้านค้า' เปลี่ยน—ขีดเส้น dashed ทั้งแถว (หรือจะเลือกบางช่วงก็ได้)
        if col_desc_th and prev_desc_th is not None and cur_desc_th != prev_desc_th:
            for c in range(1, len(headers) + 1):
                ws.cell(row=r-1, column=c).border = dash_border

        # ถ้า DESCRIPTION เปลี่ยน—ขีดเส้น thin ทั้งแถว
        if col_desc_en and prev_desc_en is not None and cur_desc_en != prev_desc_en:
            for c in range(1, len(headers) + 1):
                ws.cell(row=r-1, column=c).border = thin_border

        # ถ้า CODE_7_DIGITS เปลี่ยน—ขีดเส้นหนา (medium) ทั้งแถว
        if col_code7 and prev_code7 is not None and cur_code7 != prev_code7:
            for c in range(1, len(headers) + 1):
                ws.cell(row=r-1, column=c).border = thick_border

        prev_desc_th = cur_desc_th
        prev_desc_en = cur_desc_en
        prev_code7   = cur_code7

    # ----- ไฮไลต์ทั้งแถวถ้า "ขนาดร้านค้า" = ร้านขนาดใหญ่ -----
    # ----- ไฮไลต์บางคอลัมน์ถ้า "ขนาดร้านค้า" = ร้านขนาดใหญ่ -----
    col_size = col_idx('ขนาดร้านค้า')
    if col_size:
        # ใช้สีฟ้าอ่อนแทนเขียว
        highlight_fill = PatternFill(start_color="A6F1F1", end_color="A6F1F1", fill_type="solid")
        
        for r in range(2, ws.max_row + 1):
            if ws.cell(row=r, column=col_size).value == "ร้านขนาดใหญ่":
                # ระบายเฉพาะคอลัมน์ที่ต้องการ (เช่น 24 ถึง 25)
                for c in range(26, 30):
                    ws.cell(row=r, column=c).fill = highlight_fill


    # ----- หัวตารางตัวหนา + พื้นหลังฟ้า -----
    bold_font = Font(bold=True)
    blue_fill = PatternFill(start_color="87CEEB", end_color="87CEEB", fill_type="solid")
    for c in range(1, len(headers) + 1):
        cell = ws.cell(row=1, column=c)
        cell.font = bold_font
        cell.fill = blue_fill

    # ----- ฟอนต์สีแดงที่ AVG และสีเขียวที่ ราคาแนะนำ (ถ้ามีคอลัมน์) -----
    red_font   = Font(color="FF0000")
    green_font = Font(color="00B050")
    blue_font = Font(color="0000FF")
    c_avg  = col_idx('AVG')
    c_sug  = col_idx('ราคาแนะนำ')
    c_result = col_idx('LINK')
    c_middle = col_idx('ราคากลาง')
    if c_avg:
        for r in range(2, ws.max_row + 1):
            ws.cell(row=r, column=c_avg).font = red_font
    if c_sug:
        for r in range(2, ws.max_row + 1):
            ws.cell(row=r, column=c_sug).font = green_font
    if c_middle:
        for r in range(2, ws.max_row + 1):
            ws.cell(row=r, column=c_middle).font = green_font            
    if c_result is not None:
        for r in range(2, ws.max_row + 1):
            ws.cell(row=r, column=c_result).font = blue_font

    # ----- เติมสีตาม 'สถานะ' และ 'REL' -----
    c_status = col_idx('สถานะ')
    if c_status:
        for r in range(2, ws.max_row + 1):
            v = ws.cell(row=r, column=c_status).value
            s = str(v)
            if "น่าสงสัย" in s:
                ws.cell(row=r, column=c_status).fill = PatternFill(start_color="FFCCCC", end_color="FFCCCC", fill_type="solid")
            if "ผิดปกติ" in s:
                ws.cell(row=r, column=c_status).fill = PatternFill(start_color="FF6667", end_color="FF6667", fill_type="solid")
            if "ไม่มีราคา" in s or v == 0:
                ws.cell(row=r, column=c_status).fill = PatternFill(start_color="D3D3D3", end_color="D3D3D3", fill_type="solid")

    c_rel = col_idx('REL')
    if c_rel:
        for r in range(2, ws.max_row + 1):
            v = ws.cell(row=r, column=c_rel).value
            if v is None or v == '' or (isinstance(v, float) and pd.isna(v)):
                ws.cell(row=r, column=c_rel).fill = PatternFill(start_color="D3D3D3", end_color="D3D3D3", fill_type="solid")
            else:
                try:
                    x = float(v)
                    if x == 0:
                        ws.cell(row=r, column=c_rel).fill = PatternFill(start_color="D3D3D3", end_color="D3D3D3", fill_type="solid")
                    elif x < 100:
                        ws.cell(row=r, column=c_rel).fill = PatternFill(start_color="FFFACD", end_color="FFFACD", fill_type="solid")
                    elif x > 100:
                        ws.cell(row=r, column=c_rel).fill = PatternFill(start_color="CCFFCC", end_color="CCFFCC", fill_type="solid")
                except Exception:
                    # ถ้าแปลงเลขไม่ได้ ไม่ทำอะไร
                    pass
                
        c_avg_target = col_idx('AVG')                    # คอลัมน์เป้าหมาย (ระบายสีที่นี่)
        c_check_other = col_idx('ผลการตรวจสอบร้านอื่นๆ') # เงื่อนไข 1 (สีเหลือง)
        c_check_modern = col_idx('ผลการตรวจสอบModernTrade') # เงื่อนไข 2 (สีแดงตามรูป)
        
        # ตรวจสอบว่ามีคอลัมน์เป้าหมายอยู่จริง
        if c_avg_target:
            # เตรียมสี
            yellow_fill = PatternFill(start_color="FFFF00", end_color="FFFF00", fill_type="solid") # เหลือง
            red_fill = PatternFill(start_color="FF9999", end_color="FF9999", fill_type="solid")    # แดงอ่อน/ชมพู (ตามรูป)

            # for r in range(2, ws.max_row + 1):
            #     # --- 1. เช็คเงื่อนไข ร้านอื่นๆ (สีเหลือง) ---
            #     if c_check_other:
            #         val_other = ws.cell(row=r, column=c_check_other).value
            #         if val_other and "ให้ตรวจสอบราคา" in str(val_other):
            #             ws.cell(row=r, column=c_avg_target).fill = yellow_fill 
            target_texts = ["ให้ตรวจสอบราคา", "ให้ตรวจสอบการเปลี่ยนแปลงราคาในอดีต"]

            for r in range(2, ws.max_row + 1):
                if c_check_other:
                    val_other = str(ws.cell(row=r, column=c_check_other).value or "")
                    # ถ้าข้อความใน Cell ตรงกับคำใดคำหนึ่งใน list
                    if any(text in val_other for text in target_texts):
                        ws.cell(row=r, column=c_avg_target).fill = yellow_fill  
                                              

                # --- 2. เช็คเงื่อนไข Modern Trade (สีแดง) ---
                # (เขียนทับสีเหลืองถ้ามีทั้งคู่ หรือถ้าต้องการลำดับความสำคัญอื่น ให้สลับตำแหน่ง if)
                if c_check_modern:
                    val_modern = ws.cell(row=r, column=c_check_modern).value
                    # ตรวจสอบคำว่า "ให้ตรวจสอบ" (ครอบคลุม 'ให้ตรวจสอบ REL...')
                    if val_modern and "ให้ตรวจสอบ" in str(val_modern):
                        ws.cell(row=r, column=c_avg_target).fill = red_fill 

    # ----- AutoFilter & Freeze -----
    ws.auto_filter.ref = ws.dimensions
    ws.freeze_panes = "A2"

    c_comkey = col_idx('ComKeyBasic')
    if c_comkey:
        # 1. สร้างสีเทา (PatternFill)
        gray_fill = PatternFill(start_color="D3D3D3", end_color="D3D3D3", fill_type="solid")
        
        # 2. วนลูปตั้งแต่แถว 2 (ข้าม Header) จนถึงแถวสุดท้าย
        for r in range(2, ws.max_row + 1):
            ws.cell(row=r, column=c_comkey).fill = gray_fill


    # ----- ตัวอย่าง extend column width (ถ้ามีฟังก์ชันนี้ในโค้ดคุณ) -----
    # ถ้าไม่มีฟังก์ชัน extend_column_width ให้คอมเมนต์บรรทัดเหล่านี้
    try:
        def _ext(name, width):
            ci = col_idx(name)
            if ci:
                ws.column_dimensions[get_column_letter(ci)].width = width
        _ext('ข้อความ2', 2)
        _ext('LINK', 7)
        _ext('นับตัวซ้ำ', 4)
        _ext('Rank', 8)
        _ext('REL', 6)
        # _ext('รายงานจังหวัดที่ขาดส่ง', 30)
        _ext('DESCRIPTION', 30)
        _ext('COMMODITY_CODE', 17)
        _ext('CODE_7_DIGITS', 8)
        _ext('DILLER_CODE', 11)
        _ext('ราคาแนะนำ', 14)
        _ext('LAST_PRICE',5)
        _ext('COMPARE_PRICE',6)
        _ext('AVG',8)
        _ext('จำนวนร้าน(ประเภทร้านค้า)',3)
        _ext('นับตัวซ้ำ_',6)
        _ext('สรุปผล_จัดหมวด',25)
        _ext('แสดงผล',5)
        _ext('ราคากลาง',9)
        _ext('ใช้คำนวณ',3)
        _ext('TYPE',4)
        _ext('MONTH',4)
        _ext('YEAR',5)
        _ext('จำนวนต่อหน่วยบรรจุ',3)
        _ext('ปริมาณ',6)
        _ext('หน่วย',6)
        _ext('ราคากลาง',11)
        _ext('TOTAL_FLAGS',7)
        _ext('ผู้ดูแล',5)        
        _ext('ผลการตรวจสอบ',30)
        _ext('ภาค',5)
        


        
    except NameError:
        # ไม่มี extend_column_width ในสโคปนี้ ข้ามไปได้
        pass

    # ----- บันทึกไฟล์ -----
    # (ไม่จำเป็นต้อง drop_duplicates ใน df ตอนบันทึก เว้นแต่คุณตั้งใจให้ตัดซ้ำจริง ๆ)
    wb.save(filename)
    print(f"ไฟล์_{filename}_ถูกสร้างเรียบร้อยแล้ว")


#### function2

In [7]:
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Alignment, Border, Side, PatternFill, Font
from openpyxl.utils import get_column_letter

# def Excel_Condition2(df, filename,
#                      columns_to_hide_by_name=None):
#     """
#     สร้างไฟล์ Excel พร้อมจัดรูปแบบ และซ่อนคอลัมน์ด้วย 'ชื่อคอลัมน์'
#     - df: pandas.DataFrame
#     - filename: ชื่อไฟล์ .xlsx
#     - columns_to_hide_by_name: list[str] ของชื่อคอลัมน์ที่จะซ่อน (optional)
#     """

#     # # ----- เตรียม workbook / sheet -----
#     wb = Workbook()
#     ws = wb.active
#     # ws = wb.create_sheet(title=sheet_name)
#     # ws = wb.create_sheet(title=sheet_name)
def Excel_Condition3(df, wb, sheet_name, columns_to_hide_by_name=None):
    """
    สร้างชีตใน workbook พร้อมจัดรูปแบบ และซ่อนคอลัมน์ด้วย 'ชื่อคอลัมน์'
    - df        : pandas.DataFrame
    - wb        : openpyxl.Workbook (สร้างจากข้างนอก)
    - sheet_name: ชื่อชีตในไฟล์
    """

    # ----- เตรียม sheet จาก workbook -----
    ws = wb.create_sheet(title=sheet_name)
    # ----- เขียนหัวตาราง -----
    headers = list(df.columns)
    ws.append(headers)

    # ----- เขียนข้อมูลทีละแถว (เริ่มที่แถว 2 เพราะแถว 1 เป็น header) -----
    for row_vals in df.itertuples(index=False, name=None):
        ws.append(row_vals)

    # ----- ฟังก์ชันช่วย: หา index คอลัมน์จากชื่อ (คืนค่าเป็น 1-based หรือ None) -----
    def col_idx(col_name: str):
        return headers.index(col_name) + 1 if col_name in headers else None

    # ----- ซ่อนคอลัมน์ด้วย 'ชื่อคอลัมน์' หลังจากที่มี header แล้ว -----
    if columns_to_hide_by_name is None:
        columns_to_hide_by_name = [
       'ผลการตรวจสอบModernTrade','ผลการตรวจสอบร้านอื่นๆ','CountAVG','AVGtail_stable','Report_Z>=2','Report_IQR', 'Z_Score_Local','Z_Score_Global','Sum_Z-Score','รายการ','PROVINCE_NAME','กลุ่ม', 'C_FLAG', 'G_FLAG','GROUP_KEY','P_FLAG','USERID',
       'L_FLAG', 'N_FLAG', 'R_FLAG', 'NR_FLAG', 'TYPE_CODE', 'ข้อความ0', 'ข้อความ1',
        'จำนวนสินค้า', 'Rank', 'ขนาดร้านค้า',
       'จำนวนร้าน(ขนาดร้านค้า)', 'PROVINCE_NAME', '[ภาค]', 
       'นับรายการเก็บต่อจังหวัด', 'ราคาที่นิยมเฉพาะร้าน',
       'ผลต่าง_ราคาที่นิยมเฉพาะร้าน', 'ฐานนิยม(ในสินค้า)',
       'ฐานนิยม(ในขนาดร้านค้า)', 'ฐานนิยม(ในประเภทร้านค้า)', 'ฐานนิยม(รวมค่า)',
       'ราคาอยู่ในช่วง', 'Z-Score_ทั้งสินค้า', 'Z-Score_เฉพาะแหล่งเดียวกัน',
       'Sum_Z-Score', 'Report Z >=2',  'ตีความComment1', 'check_max_ฐานนิยม(ในประเภทร้านค้า)', 'max', 'min',
       'รายงานจังหวัดที่ขาดส่ง', 'ราคาปกติ', 'ราคาโปร',  
       'นับการการแก้ไขตามจังหวัด',  'temp_diff',
       'AVG_Max', 'AVG_Min', 'AVG_maxdiff', 'AVG_mindiff',
       'Rel_Max', 'Rel_Min', 'Reltrail_eq100', 'ต่อเนื่องเกิน12เดือน',
        'MAX_DIFF', 'C diff A',
        'สรุปผลกระโดด','นับ_สรุปผลกระโดด','สรุปผลเกินขอบ','นับ_สรุปผลเกินขอบ','สรุปผลREL','นับ_สรุปผลREL','สรุปผล',
        'ราคากระโดดไกล','ราคาเกินช่วงสูงต่ำ','สถานะ','สรุปผล_จัดหมวด','AVG_ฐานนิยม','REL_ฐานนิยม','Bandรายละเอียด','การบรรจุ','ผลต่างราคากับราคากลาง'
        ,'TYPE','ผลต่างAVG','ผลต่างREL','CODE9','CODE7','ComKeyAdv'

        ]        


    for name in columns_to_hide_by_name:
        idx = col_idx(name)
        if idx:
            ws.column_dimensions[get_column_letter(idx)].hidden = True

    # ----- จัดให้บางคอลัมน์ชิดกลาง (ถ้ามี) -----
    center_columns = ['LAST_PRICE','AVG','COMPARE_PRICE','LINK', 'REL', 'CODE_7_DIGITS', 'ราคากลาง', 'นับตัวซ้ำ']
    center_indices = [col_idx(c) for c in center_columns if col_idx(c)]
    if center_indices:
        for r in ws.iter_rows(min_row=2, max_row=ws.max_row):
            for ci in center_indices:
                r[ci-1].alignment = Alignment(horizontal='center')

    # ----- เส้นขอบแดช/บาง/หนา ตามเงื่อนไข -----
    dash_border  = Border(bottom=Side(style='dashed', color='FF6666'))
    thin_border  = Border(bottom=Side(style='thin'))
    thick_border = Border(bottom=Side(style='medium'))

    col_desc_th = col_idx('ประเภทร้านค้า')
    col_desc_en = col_idx('DESCRIPTION')
    col_code7   = col_idx('CODE_7_DIGITS')

    prev_desc_th = None
    prev_desc_en = None
    prev_code7   = None

    # เดินทีละแถวข้อมูลจริง (เริ่มที่แถว 2)
    for r in range(2, ws.max_row + 1):
        # ค่าปัจจุบันที่ใช้เปรียบเทียบ
        cur_desc_th = ws.cell(row=r, column=col_desc_th).value if col_desc_th else None
        cur_desc_en = ws.cell(row=r, column=col_desc_en).value if col_desc_en else None
        cur_code7   = ws.cell(row=r, column=col_code7).value   if col_code7   else None

        # ถ้า 'ประเภทร้านค้า' เปลี่ยน—ขีดเส้น dashed ทั้งแถว (หรือจะเลือกบางช่วงก็ได้)
        if col_desc_th and prev_desc_th is not None and cur_desc_th != prev_desc_th:
            for c in range(1, len(headers) + 1):
                ws.cell(row=r-1, column=c).border = dash_border

        # ถ้า DESCRIPTION เปลี่ยน—ขีดเส้น thin ทั้งแถว
        if col_desc_en and prev_desc_en is not None and cur_desc_en != prev_desc_en:
            for c in range(1, len(headers) + 1):
                ws.cell(row=r-1, column=c).border = thin_border

        # ถ้า CODE_7_DIGITS เปลี่ยน—ขีดเส้นหนา (medium) ทั้งแถว
        if col_code7 and prev_code7 is not None and cur_code7 != prev_code7:
            for c in range(1, len(headers) + 1):
                ws.cell(row=r-1, column=c).border = thick_border

        prev_desc_th = cur_desc_th
        prev_desc_en = cur_desc_en
        prev_code7   = cur_code7

    # ----- ไฮไลต์ทั้งแถวถ้า "ขนาดร้านค้า" = ร้านขนาดใหญ่ -----
    # ----- ไฮไลต์บางคอลัมน์ถ้า "ขนาดร้านค้า" = ร้านขนาดใหญ่ -----
    col_size = col_idx('ขนาดร้านค้า')
    if col_size:
        # ใช้สีฟ้าอ่อนแทนเขียว
        highlight_fill = PatternFill(start_color="A6F1F1", end_color="A6F1F1", fill_type="solid")
        
        for r in range(2, ws.max_row + 1):
            if ws.cell(row=r, column=col_size).value == "ร้านขนาดใหญ่":
                # ระบายเฉพาะคอลัมน์ที่ต้องการ (เช่น 24 ถึง 25)
                for c in range(26, 30):
                    ws.cell(row=r, column=c).fill = highlight_fill


    # ----- หัวตารางตัวหนา + พื้นหลังฟ้า -----
    bold_font = Font(bold=True)
    blue_fill = PatternFill(start_color="87CEEB", end_color="87CEEB", fill_type="solid")
    for c in range(1, len(headers) + 1):
        cell = ws.cell(row=1, column=c)
        cell.font = bold_font
        cell.fill = blue_fill

    # ----- ฟอนต์สีแดงที่ AVG และสีเขียวที่ ราคาแนะนำ (ถ้ามีคอลัมน์) -----
    red_font   = Font(color="FF0000")
    green_font = Font(color="00B050")
    blue_font = Font(color="0000FF")

    red_bold_font   = Font(color="FF0000", bold=True)  # แดงเข้ม (ตัวหนา)
    black_bold_font = Font(color="000000", bold=True)  # ดำเข้ม (ตัวหนา)

    c_avg  = col_idx('AVG')
    c_rel = col_idx('REL')
    c_sug  = col_idx('ราคาแนะนำ')
    c_result = col_idx('LINK')
    c_middle = col_idx('ราคากลาง')    
    c_middle_String_online  = col_idx('Online')
    c_reason0 = col_idx('ผลการตรวจสอบ')
    c_reason1 = col_idx('ผลการตรวจสอบ AVG')
    c_reason2 = col_idx('ผลการตรวจสอบ REL')
    	

    if c_avg:
        for r in range(2, ws.max_row + 1):
            ws.cell(row=r, column=c_avg).font = red_bold_font
    if c_rel:
        for r in range(2, ws.max_row + 1):
            ws.cell(row=r, column=c_rel).font = black_bold_font

    if c_sug:
        for r in range(2, ws.max_row + 1):
            ws.cell(row=r, column=c_sug).font = green_font
    if c_middle:
        for r in range(2, ws.max_row + 1):
            ws.cell(row=r, column=c_middle).font = green_font 
            ws.cell(row=r, column=c_middle_String_online).font = green_font            
    if c_result is not None:
        for r in range(2, ws.max_row + 1):
            ws.cell(row=r, column=c_result).font = blue_font
    if c_reason0 is not None:
        for r in range(2, ws.max_row + 1):
            ws.cell(row=r, column=c_reason0).font = red_font
    if c_reason1 is not None:
        for r in range(2, ws.max_row + 1):
            ws.cell(row=r, column=c_reason1).font = red_font
    if c_reason2 is not None:
        for r in range(2, ws.max_row + 1):
            ws.cell(row=r, column=c_reason2).font = red_font

    # ----- เติมสีตาม 'สถานะ' และ 'REL' -----
    c_status = col_idx('สถานะ')
    if c_status:
        for r in range(2, ws.max_row + 1):
            v = ws.cell(row=r, column=c_status).value
            s = str(v)
            if "น่าสงสัย" in s:
                ws.cell(row=r, column=c_status).fill = PatternFill(start_color="FFCCCC", end_color="FFCCCC", fill_type="solid")
            if "ผิดปกติ" in s:
                ws.cell(row=r, column=c_status).fill = PatternFill(start_color="FF6667", end_color="FF6667", fill_type="solid")
            if "ไม่มีราคา" in s or v == 0:
                ws.cell(row=r, column=c_status).fill = PatternFill(start_color="D3D3D3", end_color="D3D3D3", fill_type="solid")

    c_rel = col_idx('REL')
    if c_rel:
        for r in range(2, ws.max_row + 1):
            v = ws.cell(row=r, column=c_rel).value
            if v is None or v == '' or (isinstance(v, float) and pd.isna(v)):
                ws.cell(row=r, column=c_rel).fill = PatternFill(start_color="D3D3D3", end_color="D3D3D3", fill_type="solid")
            else:
                try:
                    x = float(v)
                    if x == 0:
                        ws.cell(row=r, column=c_rel).fill = PatternFill(start_color="D3D3D3", end_color="D3D3D3", fill_type="solid")
                    elif x < 100:
                        ws.cell(row=r, column=c_rel).fill = PatternFill(start_color="FFFACD", end_color="FFFACD", fill_type="solid")
                    elif x > 100:
                        ws.cell(row=r, column=c_rel).fill = PatternFill(start_color="CCFFCC", end_color="CCFFCC", fill_type="solid")
                except Exception:
                    # ถ้าแปลงเลขไม่ได้ ไม่ทำอะไร
                    pass
        c_check_other = col_idx('ผลการตรวจสอบร้านอื่นๆ') # คอลัมน์เงื่อนไข
        c_avg_target = col_idx('AVG')                    # คอลัมน์เป้าหมายที่จะระบายสี
        
        # # ต้องเช็คว่ามีทั้งสองคอลัมน์นี้อยู่จริงในไฟล์ถึงจะทำงาน
        # if c_check_other and c_avg_target:
        #     yellow_fill_check = PatternFill(start_color="FFFF00", end_color="FFFF00", fill_type="solid")
            
        #     for r in range(2, ws.max_row + 1):
        #         # อ่านค่าจากคอลัมน์ 'ผลการตรวจสอบร้านอื่นๆ'
        #         val_condition = ws.cell(row=r, column=c_check_other).value
                
        #         # ถ้ามีค่า และมีคำว่า "ให้ตรวจสอบราคา"
        #         if val_condition and "ให้ตรวจสอบราคา" in str(val_condition):
        #             # สั่งระบายสีที่คอลัมน์ AVG (c_avg_target)
        #             ws.cell(row=r, column=c_avg_target).fill = yellow_fill_check
    # -----------------------------------------------------------
        # [NEW] จัดการสีที่คอลัมน์ 'AVG' ตามเงื่อนไขต่างๆ
        # -----------------------------------------------------------
        c_avg_target = col_idx('AVG')                    # คอลัมน์เป้าหมาย (ระบายสีที่นี่)
        c_rel_target = col_idx('REL')                    # คอลัมน์เป้าหมาย (ระบายสีที่นี่)
        c_check_other = col_idx('ผลการตรวจสอบร้านอื่นๆ') # เงื่อนไข 1 (สีเหลือง)
        c_check_modern = col_idx('ผลการตรวจสอบModernTrade') # เงื่อนไข 2 (สีแดงตามรูป)
        
        # ตรวจสอบว่ามีคอลัมน์เป้าหมายอยู่จริง
        if c_avg_target:
            # เตรียมสี
            yellow_fill = PatternFill(start_color="FFFF00", end_color="FFFF00", fill_type="solid") # เหลือง
            red_fill = PatternFill(start_color="FF9999", end_color="FF9999", fill_type="solid")    # แดงอ่อน/ชมพู (ตามรูป)

            for r in range(2, ws.max_row + 1):
                # --- 1. เช็คเงื่อนไข ร้านอื่นๆ (สีเหลือง) ---
                if c_check_other:
                    val_other = ws.cell(row=r, column=c_check_other).value
                    if val_other and "ให้ตรวจสอบราคา" in str(val_other):
                        ws.cell(row=r, column=c_avg_target).fill = yellow_fill

                # --- 2. เช็คเงื่อนไข Modern Trade (สีแดง) ---
                # (เขียนทับสีเหลืองถ้ามีทั้งคู่ หรือถ้าต้องการลำดับความสำคัญอื่น ให้สลับตำแหน่ง if)
                if c_check_modern:
                    val_modern = ws.cell(row=r, column=c_check_modern).value
                    # ตรวจสอบคำว่า "ให้ตรวจสอบ" (ครอบคลุม 'ให้ตรวจสอบ REL...')
                    if val_modern and "ให้ตรวจสอบ" in str(val_modern):
                        ws.cell(row=r, column=c_avg_target).fill = red_fill  
                        ws.cell(row=r, column=c_rel_target).fill = red_fill  
        
    # ----- AutoFilter & Freeze -----
    ws.auto_filter.ref = ws.dimensions
    ws.freeze_panes = "A2"
    c_comkey = col_idx('ComKeyBasic')
    if c_comkey:
        # 1. สร้างสีเทา (PatternFill)
        gray_fill = PatternFill(start_color="D3D3D3", end_color="D3D3D3", fill_type="solid")
        
        # 2. วนลูปตั้งแต่แถว 2 (ข้าม Header) จนถึงแถวสุดท้าย
        for r in range(2, ws.max_row + 1):
            ws.cell(row=r, column=c_comkey).fill = gray_fill
    # ----- ตัวอย่าง extend column width (ถ้ามีฟังก์ชันนี้ในโค้ดคุณ) -----
    # ถ้าไม่มีฟังก์ชัน extend_column_width ให้คอมเมนต์บรรทัดเหล่านี้
    try:
        def _ext(name, width):
            ci = col_idx(name)
            if ci:
                ws.column_dimensions[get_column_letter(ci)].width = width
        _ext('ข้อความ2', 2)
        _ext('LINK', 7)
        _ext('นับตัวซ้ำ', 4)
        _ext('Rank', 8)
        _ext('REL', 6)
        # _ext('รายงานจังหวัดที่ขาดส่ง', 30)
        _ext('DESCRIPTION', 30)
        _ext('COMMODITY_CODE', 17)
        _ext('CODE_7_DIGITS', 8)
        _ext('DILLER_CODE', 11)
        _ext('ราคาแนะนำ', 14)
        _ext('LAST_PRICE',5)
        _ext('COMPARE_PRICE',6)
        _ext('AVG',8)
        _ext('จำนวนร้าน(ประเภทร้านค้า)',3)
        _ext('นับตัวซ้ำ_',6)
        _ext('สรุปผล_จัดหมวด',25)
        _ext('แสดงผล',5)
        _ext('ราคากลาง',9)
        _ext('ใช้คำนวณ',3)
        _ext('TYPE',4)
        _ext('MONTH',3)
        _ext('YEAR',5)
        _ext('จำนวนต่อหน่วยบรรจุ',3)
        _ext('ปริมาณ',6)
        _ext('หน่วย',6)
        _ext('ราคากลาง',11)
        _ext('TOTAL_FLAGS',7)
        _ext('ผู้ดูแล',5.5)        
        _ext('ผลการตรวจสอบ',30)
        _ext('ภาค',5)
        _ext('ผลการตรวจสอบ AVG',25)
        _ext('ผลการตรวจสอบ REL',25)
        _ext('ComKeyBasic',3.4)





        
    except NameError:
        # ไม่มี extend_column_width ในสโคปนี้ ข้ามไปได้
        pass

    # ----- บันทึกไฟล์ -----
    # (ไม่จำเป็นต้อง drop_duplicates ใน df ตอนบันทึก เว้นแต่คุณตั้งใจให้ตัดซ้ำจริง ๆ)
    # wb.save(filename)
    # print(f"ไฟล์_{filename}_ถูกสร้างเรียบร้อยแล้ว")


### ราคากลาง

In [8]:
import pandas as pd

# ราคากลาง
if run_master_price == 1:
    try:
        # 1. อ่านไฟล์แบบไม่เอา Header (header=None) เพื่อจัดการหัวตารางเอง
        print(f"กำลังอ่านข้อมูลจากไฟล์ '{excel_file_path}'...")
        all_sheets_dict = pd.read_excel(excel_file_path, sheet_name=None, header=None)
        print(f"อ่านไฟล์เสร็จสิ้น พบทั้งหมด {len(all_sheets_dict)} ชีท")

    except Exception as e:
        print(f"เกิดข้อผิดพลาด: {e}")
        all_sheets_dict = {}

    def center(df_raw):
        # ลบแถวว่างทิ้ง
        df_raw = df_raw.dropna(how='all')
        
        # --- ส่วนสำคัญ: จัดการ Header ---
        
        # ดึงแถวที่ 0 (ชื่อร้าน: Lotus, BigC) และ แถวที่ 1 (ชื่อคอลัมน์: Link, ราคาปกติ...)
        # หมายเหตุ: .iloc[0] คือบรรทัดแรกสุดของ Excel
        row_store = df_raw.iloc[0].copy()
        row_sub_header = df_raw.iloc[1].copy()
        
        # *** KEY FIX: ถมชื่อร้านไปทางขวาให้เต็ม (Lotus, NaN, NaN -> Lotus, Lotus, Lotus) ***
        # เพื่อให้คอลัมน์ 'ราคาปกติ' รู้ว่าตัวเองอยู่ใต้ร้าน 'Lotus' ไม่ใช่ร้านว่างเปล่า
        row_store = row_store.ffill()
        
        # ข้อมูลจริงเริ่มตั้งแต่แถวที่ 2 เป็นต้นไป
        df_data = df_raw.iloc[2:].reset_index(drop=True)
        
        # ระบุคอลัมน์หลัก (A และ B)
        # 0 = รหัสรายการ, 1 = ชื่อรายการ
        col_code_idx = 0
        col_name_idx = 1
        
        # สร้าง list ไว้เก็บ DataFrame ของแต่ละร้าน
        dfs_list = []
        
        # หาชื่อร้านค้าทั้งหมด (เริ่มดูจากคอลัมน์ 2 เป็นต้นไป เพื่อข้าม รหัส/ชื่อรายการ)
        # dropna() เพื่อกันช่องว่าง, unique() เพื่อเอารายชื่อร้านที่ไม่ซ้ำ
        unique_stores = row_store[2:].dropna().unique()
        
        for store_name in unique_stores:
            # ข้ามถ้าเป็น header ของวันที่ (เช่น 'ธ.ค.-68') ที่อาจติดมา
            if "ธ.ค." in str(store_name): continue
            
            # หา index ของคอลัมน์ทั้งหมดที่เป็นของร้านนี้
            store_indices = row_store[row_store == store_name].index
            
            # ในกลุ่มคอลัมน์ของร้านนี้ หาตำแหน่งของ 'ราคาปกติ', 'ราคาโปร', 'ภาวะ'
            # โดยดูจาก row_sub_header
            idx_normal = None
            idx_promo = None
            idx_cond = None
            
            for idx in store_indices:
                col_name = str(row_sub_header[idx])
                if "ราคาปกติ" in col_name:
                    idx_normal = idx
                elif "ราคาโปร" in col_name:
                    idx_promo = idx
                elif "ภาวะ" in col_name:
                    idx_cond = idx
            
            # ถ้าเจอคอลัมน์ราคาปกติ แสดงว่าเป็นข้อมูลร้านค้าที่ถูกต้อง
            if idx_normal is not None:
                temp_df = pd.DataFrame()
                
                # ดึงข้อมูลใส่ DataFrame ใหม่
                temp_df['รหัสรายการ\nCOMMODITY_CODE\n'] = df_data[col_code_idx]
                temp_df['ชื่อรายการ'] = df_data[col_name_idx]
                temp_df['ประเภทร้านค้า'] = store_name # ใส่ชื่อร้านที่ถูกต้อง (Lotus, BigC, Tops)
                
                # ใส่ราคา (ถ้าไม่มี index ให้ใส่ NaN)
                temp_df['ราคาปกติ'] = df_data[idx_normal]
                temp_df['ราคาโปร'] = df_data[idx_promo] if idx_promo is not None else None
                temp_df['ภาวะ'] = df_data[idx_cond] if idx_cond is not None else None
                
                dfs_list.append(temp_df)
                
        if not dfs_list:
            return pd.DataFrame()

        # รวมทุกร้านเข้าด้วยกัน
        df_final = pd.concat(dfs_list, ignore_index=True)
        
        # Clean ข้อมูล: ลบทิ้งถ้า ราคาปกติ, ราคาโปร, ภาวะ เป็นว่างทั้งหมด
        df_final = df_final.dropna(subset=['ราคาปกติ', 'ราคาโปร', 'ภาวะ'], how='all')
        
        return df_final

    # Loop รวมข้อมูลจากทุก Sheet
    list_of_dfs = []
    for name, df in all_sheets_dict.items():
        # print(f"Processing sheet: {name}")
        dcen = center(df)
        if not dcen.empty:
            list_of_dfs.append(dcen)

    if list_of_dfs:
        allcenter = pd.concat(list_of_dfs, ignore_index=True)
        # display(allcenter)
        print(f"ประมวลผลเสร็จสิ้น ได้ข้อมูล {len(allcenter)} แถว")
    else:
        allcenter = pd.DataFrame()
        print("ไม่พบข้อมูล")

กำลังอ่านข้อมูลจากไฟล์ 'ไฟล์ราคากลาง ชุด บ_ก.พ. 69.xlsx'...
อ่านไฟล์เสร็จสิ้น พบทั้งหมด 62 ชีท
ประมวลผลเสร็จสิ้น ได้ข้อมูล 1257 แถว


In [9]:
dname = pd.read_excel(excel_file_path)
print(dname.columns[0])
dname = dname.columns[0].strftime("%y%m")
dname

2569-02-01 00:00:00


'6902'

In [10]:
# หมด 1
allcenter_clean = allcenter.copy()
allcenter_clean['ประเภทร้านค้า'] = allcenter_clean['ประเภทร้านค้า'].replace({'BigC':'Big C','Lotus':'Tesco Lotus','7 - Eleven':'Seven Eleven','แม็คโคร':'Makro','CP Freshmart':'CP Freshmart'})

allcenter_clean = allcenter_clean.rename(columns={'รหัสรายการ\nCOMMODITY_CODE\n':'COMMODITY_CODE','ชื่อรายการ':'DESCRIPTION'})
allcenter_clean = allcenter_clean[['COMMODITY_CODE','DESCRIPTION', 'ประเภทร้านค้า', 'ราคาปกติ','ราคาโปร','ภาวะ']]
allcenter_clean['COMMODITY_CODE'] = allcenter_clean['COMMODITY_CODE'].astype(str).str.zfill(16)
type(allcenter_clean['COMMODITY_CODE'][1])


# Step 1: Convert price columns to numeric, coercing errors to NaN
allcenter_clean['ราคาปกติ'] = pd.to_numeric(
    allcenter_clean['ราคาปกติ'], 
    errors='coerce'
)
allcenter_clean['ราคาโปร'] = pd.to_numeric(
    allcenter_clean['ราคาโปร'], 
    errors='coerce'
)
# Element-wise minimum for each row (axis=1)
allcenter_clean['ราคากลาง'] = allcenter_clean[['ราคาปกติ', 'ราคาโปร']].min(axis=1)
allcenter_clean = allcenter_clean[['COMMODITY_CODE','DESCRIPTION','ประเภทร้านค้า','ราคากลาง','ภาวะ']]

# allcenter_clean_outstock-----------------------------------------------------------------
allcenter_clean_outstock = allcenter_clean[allcenter_clean['ภาวะ']=='หมด'].copy()
allcenter_clean_outstock['ราคาปกติ'] = ""
allcenter_clean_outstock['ราคาโปร'] = ""
allcenter_clean_outstock = allcenter_clean_outstock[['COMMODITY_CODE','DESCRIPTION','ประเภทร้านค้า','ราคาปกติ', 'ราคาโปร','ภาวะ']]

# output_filename = filename[:-2] + dname +'_สินค้าหมด.xlsx'

# # กำหนดความกว้างแบบ Manual (อยากได้กว้างเท่าไหร่ใส่ตรงนี้)
# # หน่วยเป็นจำนวนตัวอักษรโดยประมาณ
# custom_widths = {
#     'COMMODITY_CODE': 20,    # รหัส 16 หลัก เผื่อไว้นิดหน่อย
#     'DESCRIPTION': 40,       # ชื่อสินค้าอาจจะยาว
#     'ประเภทร้านค้า': 15,
#     'ราคาปกติ': 12,
#     'ราคาโปร': 12,
#     # คอลัมน์ไหนไม่ใส่ในนี้ จะถูกคำนวณ Auto
# }
# with pd.ExcelWriter(output_filename, engine='xlsxwriter') as writer:    
#     # 1. เขียนข้อมูลลง Sheet
#     sheet_name = 'Sheet1'
#     allcenter_clean_outstock.to_excel(writer, sheet_name=sheet_name, index=False)    
#     # 2. เข้าถึง object ของ worksheet
#     worksheet = writer.sheets[sheet_name]    
#     # 3. วนลูปทุกคอลัมน์
#     for i, col in enumerate(allcenter_clean_outstock.columns):        
#         # --- เช็คก่อนว่าคอลัมน์นี้เรากำหนดความกว้างเองไว้ไหม? ---
#         if col in custom_widths:
#             # ถ้ามีใน dict ให้ใช้ค่าที่กำหนดเลย
#             col_width = custom_widths[col]            
#         else:
#             # --- ถ้าไม่มี ให้คำนวณ Auto-fit ตามข้อมูลจริง ---            
#             # หาความยาวสูงสุดของข้อมูล
#             max_data_len = allcenter_clean_outstock[col].astype(str).map(len).max()
#             if pd.isna(max_data_len): 
#                 max_data_len = 0            
#             # หาความยาวของ Header
#             header_len = len(str(col))            
#             # เลือกค่ามากสุด + padding นิดหน่อย
#             col_width = max(max_data_len, header_len) + 2        
#         # สั่ง set ความกว้าง (XlsxWriter ใช้ set_column(first_col, last_col, width))
#         worksheet.set_column(i, i, col_width)
# # allcenter_clean_outstock-----------------------------------------------------------------

allcenter_clean
# allcenter[allcenter['ภาวะ']=='หมด']
output_file2 = filename[:-2] + dname + '_checkราคากลาง.xlsx'
allcenter_clean['Online'] = 'Online'
# allcenter_clean.to_excel(output_file2)
allcenter_clean

,COMMODITY_CODE,DESCRIPTION,ประเภทร้านค้า,ราคากลาง,ภาวะ,Online
0,1112104111020180,แป้งทอดกรอบ ขนาดบรรจุถุง น้ำหนัก 150 กรัม ตรา...,Tesco Lotus,16.5,NaN,Online
1,1112104111030401,แป้งทอดกรอบ ขนาดบรรจุถุง น้ำหนัก 500 กรัม ตร...,Tesco Lotus,35.0,โปรจาก 37,Online
2,1112104111010180,แป้งทอดกรอบ ขนาดบรรจุถุง น้ำหนัก 1 ก.ก. ตราครั...,Tesco Lotus,58.0,NaN,Online
3,1112104111030101,แป้งทอดกรอบ ขนาดบรรจุถุง น้ำหนัก 500 กรัม ตรา...,Tesco Lotus,44.0,NaN,Online
4,1112104111050101,แป้งทอดกรอบ ขนาดบรรจุถุง น้ำหนัก 120 กรัม ตรา...,Tesco Lotus,18.0,NaN,Online
...,...,...,...,...,...,...
1252,6125001131020101,อาหารสุนัข สำหรับสุนัขอายุโตเต็มวัย\ 3.0 กก....,Tops,NaN,หมด,Online
1253,6125001211020103,อาหารแมว สำหรับแมวทั่วไป อายุ 1 ปีขึ้นไป บรรจุ...,Tops,135.0,โปรจาก 161,Online
1254,6125001211020104,อาหารแมว สำหรับแมวทั่วไป อายุ 1 ปีขึ้นไป บรรจุ...,Tops,161.0,NaN,Online
1255,6125001221030303,อาหารแมว สำหรับแมวทั่วไป บรรจุถุง\1.2 กก. ร...,Tops,129.0,โปรจาก 150,Online


In [11]:
import pandas as pd

# --- ส่วนเตรียมข้อมูลเดิมของคุณ ---
report_master_price = allcenter_clean.copy()
report_master_price['ปกติ'] = ''
report_master_price['โปร'] = ''
report_master_price = report_master_price[['COMMODITY_CODE', 'DESCRIPTION', 'ประเภทร้านค้า', 'ปกติ', 'โปร', 'ภาวะ']]
report_master_price = report_master_price[report_master_price['ภาวะ']=='หมด']
report_master_price = report_master_price.drop_duplicates()

# --- ส่วนการ Export ให้สวยงาม ---
# --- ส่วนการ Export ---
output_file = filename[:-2] + dname + '_สินค้าหมด.xlsx'
custom_desc_width = 50 

with pd.ExcelWriter(output_file, engine='xlsxwriter') as writer:
    # 1. เขียนข้อมูล
    report_master_price.to_excel(writer, sheet_name='Sheet1', index=False, startrow=1, header=False)

    workbook  = writer.book
    worksheet = writer.sheets['Sheet1']

    # 2. กำหนด Style
    header_fmt = workbook.add_format({
        'bold': True, 'fg_color': '#4472C4', 'font_color': '#FFFFFF',
        'border': 1, 'align': 'center', 'valign': 'vcenter'
    })

    body_fmt = workbook.add_format({
        'border': 1, 'valign': 'vcenter'
    })
    
    # Style สำหรับช่อง 'ภาวะ' (ต้องใส่ Border ด้วยเพื่อความชัวร์)
    status_fmt = workbook.add_format({
        'border': 1, 'font_color': '#C00000', 'bold': True,
        'align': 'center', 'valign': 'vcenter'
    })

    # เตรียมตัวแปรเก็บตำแหน่งคอลัมน์ 'ภาวะ'
    status_col_idx = None 

    # 3. เขียน Header และจัดความกว้างคอลัมน์
    for col_num, value in enumerate(report_master_price.columns.values):
        worksheet.write(0, col_num, value, header_fmt)
        
        # คำนวณความกว้าง
        if value == 'DESCRIPTION':
            column_len = custom_desc_width
        else:
            column_len = max(
                report_master_price[value].astype(str).map(len).max(),
                len(str(value))
            ) + 2

        # --- [จุดที่แก้] ใส่แค่ความกว้าง (ไม่ต้องใส่ fmt ใน set_column) ---
        worksheet.set_column(col_num, col_num, column_len)
        
        # จำตำแหน่งคอลัมน์ 'ภาวะ' ไว้ใช้ข้างล่าง
        if value == 'ภาวะ':
            status_col_idx = col_num

    # 4. ใส่ Format ให้ข้อมูล (ใช้ conditional_format แทน เพื่อไม่ให้เกินบรรทัด)
    (max_row, max_col) = report_master_price.shape
    
    # 4.1 ใส่เส้นขอบธรรมดาให้ 'ทุกช่อง' ก่อน
    worksheet.conditional_format(1, 0, max_row, max_col-1, {
        'type': 'formula', 
        'criteria': '=TRUE', 
        'format': body_fmt
    })

    # 4.2 ใส่สีแดง+หนา เฉพาะคอลัมน์ 'ภาวะ' (ทับลงไปในช่วงข้อมูลที่มีเท่านั้น)
    if status_col_idx is not None:
        worksheet.conditional_format(1, status_col_idx, max_row, status_col_idx, {
            'type': 'formula',
            'criteria': '=TRUE', # ใช้สูตร True เพื่อบังคับใส่ Format
            'format': status_fmt
        })

print(f"✅ บันทึกไฟล์เรียบร้อย (แก้ตารางเกินแล้ว): {output_file}")

✅ บันทึกไฟล์เรียบร้อย (แก้ตารางเกินแล้ว): cpig_month_25696902_สินค้าหมด.xlsx


### RUN

In [12]:
# ds_ = pd.read_excel('shop_data_20251117_134054.xlsx',skiprows= 2)
import pandas as pd
import glob
# ค้นหาไฟล์ทั้งหมดที่ขึ้นต้นด้วย shop_data และลงท้ายด้วย .xlsx
files_shop = glob.glob("shop_data*.xlsx")
ds_ = pd.read_excel(files_shop [0], skiprows=2, engine='calamine')

In [13]:
dfr = pd.read_excel(right_price)
Real_master = pd.read_excel(masterF)
if run_cpi_month == 1:
    Main_data = pd.read_excel("cpi"+str(session)+"_month_"+str(Year)+"-"+str(Month)+".xlsx", engine='calamine')

In [14]:
if time_serie == 1:
    dfs_ = pd.read_excel(name_series, engine='calamine') 

In [15]:
Main_data.to_clipboard()

#### เตรียมข้อมูล

In [16]:
# ==========================================
# 1. เตรียมข้อมูลและจัด Format เบื้องต้น
# ==========================================
df = Main_data.copy()

# จัด Format รหัส
df["COMMODITY_CODE"] = df["COMMODITY_CODE"].astype(str).str.zfill(16)
df["DILLER_CODE"]    = df["DILLER_CODE"].astype(str).str.zfill(10)
df["CODE_7_DIGITS"]  = df["COMMODITY_CODE"].astype(str).str[:7]
df["CODE7"] = df["COMMODITY_CODE"].str[0:7]
df["CODE9"] = df["COMMODITY_CODE"].str[7:16]

# คำนวณ Flag
flag_cols = ['C_FLAG', 'G_FLAG', 'L_FLAG', 'R_FLAG']
df['TOTAL_FLAGS'] = np.where(df[flag_cols].sum(axis=1) > 0, 'คำนวณ', 'ไม่คำนวณ')

# แปลงเป็น str และแทนค่าว่างด้วย '' (สตริงเปล่า) หรือ '0' ตามความเหมาะสม
col1 = df['COMMODITY_CODE'].fillna('').astype(str)
col2 = df['DILLER_CODE'].fillna('').astype(str)
df['ComKeyBasic'] = col1 + col2

# แปลงเป็น str และแทนค่าว่างด้วย '' (สตริงเปล่า) หรือ '0' ตามความเหมาะสม
col1 = df['COMMODITY_CODE'].fillna('').astype(str)
col2 = df['DILLER_CODE'].fillna('').astype(str)
col3 = df['AVG'].fillna(0).astype(str) 
col4 = df['COMPARE_PRICE'].fillna(0).astype(str)
df['ComKeyAdv'] = col1 + col2 + col3 + col4

# Merge Real_master (RM)
RM = Real_master[['รหัส', 'รายการ','ผู้ดูแล']].rename(columns={'รหัส': 'CODE_7_DIGITS'})
RM["CODE_7_DIGITS"] = RM["CODE_7_DIGITS"].astype(str).str.zfill(7)
df = df.merge(RM, on=['CODE_7_DIGITS'], how='left')

# กรอง User
if User != ['All']:
    df = df.loc[df['ผู้ดูแล'].isin(User)]

# นับตัวซ้ำ
df['นับตัวซ้ำ'] = df.groupby(['DILLER_CODE', 'COMMODITY_CODE'])['DILLER_CODE'].transform('count')

# ==========================================
# 2. จัดการประเภทและขนาดร้านค้า
# ==========================================

def classify_shop_type(x):
    if not isinstance(x, str): return "ร้านอื่นๆ"
    s = x 
    if any(k in s for k in ["โลตัส", "tesco", "Tesco", "โลตัล", "Lotus"]): return "Tesco Lotus"
    if any(k in s for k in ["บิ๊กซี", "บิกซี", "Big C", "Big-C"]): return "Big C"
    if any(k in s for k in ["อีเลฟเว่น", "ELEVEN", "7 - 11", "7-11", "7-ELEVEN", "เซเว่น", "7-Eleven", "seven", "SEVEN", "Eleven", "เซ่เว่น"]): return "Seven Eleven"
    if any(k in s for k in ["CJ", "ซี เจ", "ซีเจ", "Cj Express", "Cj MORE"]): return "CJ EXPRESS"
    if any(k in s for k in ["ท็อปส์ เดลี่", "ท็อปส์ มาร์เก็ต", "ท็อปส์มาร์เก็ต", "Tops", "ห้างท็อปส์", "TOPS"]): return "Tops"
    if any(k in s for k in ["Maxvalue", "แม็กซ์แวลู", "Max Valu"]): return "Maxvalue"
    # if "เดอะมอลล์" in s: return "The Mall"
    # if "แม็คโค" in s: return "Makro"
    if any(k in s for k in ["แม็คโค", "เเม็คโค"]): return "Makro"
    if any(k in s for k in ["กูรเมต์", "กูร์เมต์","กูร์เมต์","กรูเม่"]): return "Gourmet Market"
    if any(k in s for k in ["เดอะมอลล์"]): return "The Mall"
    if any(k in s for k in ["เซนทรัล", "CENTRAL", "เซ็นทรัล"]): return "Central"
    if any(k in s for k in ["วัตสัน", "Watson"]): return "Watson"
    if any(k in s for k in ["เฟรชมาร์ท", "Freshmat"]): return "CP Freshmart"
    if "โฮมโปร" in s: return "HomePro"
    if "ร้านเอเซ่เว่น" in s: return "ร้านอื่นๆ"
    return "ร้านอื่นๆ"

df['ประเภทร้านค้า'] = df['SHOP_NAME'].apply(classify_shop_type)

# แก้ไขร้านค้ากรณีพิเศษ
mask_other = df['SHOP_NAME'].str.contains("ร้านหน้าโลตัส|Watson|วัตสัน|ปลาผา|WASH|Amazon|เอเมซอน|คาเฟ่|Cafe|ศูนย์อาหาร|อาหาร|ร้านยา|ขายยา|ข้าว|DRUG", case=False, na=False)
df.loc[mask_other, 'ประเภทร้านค้า'] = "ร้านอื่นๆ"
df.loc[df['SHOP_NAME'].str.contains("โฮมโปร", case=False, na=False), 'ประเภทร้านค้า'] = "HomePro"

# แก้ไขตามรหัส
df.loc[df['CODE_7_DIGITS'].astype(str).str.startswith("4111", na=False), 'ประเภทร้านค้า'] = "ร้านอื่นๆ"
df.loc[df['CODE_7_DIGITS'].astype(str).isin(["1250001", "1250002", "1250008", "1160015"]), 'ประเภทร้านค้า'] = "ร้านอื่นๆ"
df.loc[df['SHOP_NAME'] == 'ท็อปน์ (เซ็นทรัล นนทบุรี)', 'ประเภทร้านค้า'] = "Tops"
df.loc[df['SHOP_NAME'] == 'ร้านเอเซ่เว่น', 'ประเภทร้านค้า'] = "ร้านอื่นๆ"

# ..ห้างบิ๊กซีมินิ..... บิ๊กซีมาร์เก็ต และโลตัสเอ็กเพรส ไม่สามารถใช้ราคากลางได้
# 1. Big C Mini
mask_bigc_mini = (df['ประเภทร้านค้า'] == 'Big C') & (df['SHOP_NAME'].str.contains('มินิ|Mini', case=False, na=False))
df.loc[mask_bigc_mini, 'ประเภทร้านค้า'] = 'BigC_mini'
# # 2. Big C Market
# mask_bigc_mkt = (df['ประเภทร้านค้า'] == 'Big C') & (df['SHOP_NAME'].str.contains('มาร์เก็ต|มาเก็ต|เกต|มาเกต|Market', case=False, na=False))
# df.loc[mask_bigc_mkt, 'ประเภทร้านค้า'] = 'BigC_market'
# 3. Lotus Express
mask_lotus_exp = (df['ประเภทร้านค้า'] == 'Tesco Lotus') & (df['SHOP_NAME'].str.contains('เฟรซ|เฟรช|โก|exp|gofresh', case=False, na=False))
df.loc[mask_lotus_exp, 'ประเภทร้านค้า'] = 'Lotus_gofresh'
mask_lotus_exp = (df['ประเภทร้านค้า'] == 'Lotus_gofresh') & (df['SHOP_NAME'].str.contains('โก้', case=False, na=False))
df.loc[mask_lotus_exp, 'ประเภทร้านค้า'] = 'Tesco Lotus'

mask_Tops_day = (df['ประเภทร้านค้า'] == 'Tops') & (df['SHOP_NAME'].str.contains('เดลี่', case=False, na=False))
df.loc[mask_Tops_day, 'ประเภทร้านค้า'] = 'Tops_daily'



# คำนวณขนาดและจำนวน
df['ขนาดร้านค้า'] = df['ประเภทร้านค้า'].apply(lambda x: 'ร้านขนาดเล็ก' if x == 'ร้านอื่นๆ' else 'ร้านขนาดใหญ่')
for col in ['ขนาดร้านค้า', 'ประเภทร้านค้า']:
    df.insert(df.columns.get_loc(col)+1, f'จำนวนร้าน({col})', df.groupby(['DESCRIPTION', col])[col].transform('count'))
df.insert(df.columns.get_loc('DESCRIPTION')+1, 'จำนวนสินค้า', df.groupby(['DESCRIPTION'])['DESCRIPTION'].transform('count')) 

df = df.sort_values(by=['COMMODITY_CODE', 'DESCRIPTION', 'ประเภทร้านค้า', 'AVG'], ascending=[True, True, True, False])

# ==========================================
# 3. จัดการภาค (Region)
# ==========================================
region_map = {
    'กลาง': ["CU", "CC"], 'เหนือ': ["NU", "NN"], 
    'ใต้': ["SU", "SS"], 'ตะวันออกเฉียงเหนือ': ["EU", "EE"]
}
df['ภาค'] = None
for region, codes in region_map.items():
    df.loc[df['TYPE'].isin(codes), 'ภาค'] = region

target_provinces = ['นนทบุรี', 'สมุทรปราการ', 'ปทุมธานี']
df.loc[df['PROVINCE_NAME'].isin(target_provinces), 'ภาค'] = 'ปริมณฑล'
df['ภาค'] = df['ภาค'].fillna('กทม.')
df['PROVINCE_NAME'] = df['PROVINCE_NAME'].replace('กรุงเทพมหานคร', 'กทม.')

# ==========================================
# 4. Merge ข้อมูลภายนอก (Center Stock & Dealer Group)
# ==========================================
# Center Stock
# center_stock = allcenter_clean[allcenter_clean['ภาวะ'] != 'หมด'][['COMMODITY_CODE', 'ประเภทร้านค้า','Online','ภาวะ','ราคากลาง']].copy()
center_stock = allcenter_clean[['COMMODITY_CODE', 'ประเภทร้านค้า','Online','ภาวะ','ราคากลาง']].copy()
df = df.merge(center_stock, on=['COMMODITY_CODE', 'ประเภทร้านค้า'], how='left')
df['ผลต่างราคากับราคากลาง'] = abs(df['ราคากลาง'] - df['AVG'])
df.loc[df['ภาวะ'] == 'หมด', 'ราคากลาง'] = np.nan
df.loc[df['ภาวะ'] == 'หมด', 'ผลต่างราคากับราคากลาง'] = np.nan
df.drop(columns=['ภาวะ'], inplace=True)

# Dealer Group
ds = ds_[["รหัสร้าน", "กลุ่ม"]].rename(columns={"รหัสร้าน": "DILLER_CODE"})
ds["DILLER_CODE"] = ds["DILLER_CODE"].astype(str).str.zfill(10)
ds["กลุ่ม"] = ds["กลุ่ม"].replace(["-", " "], "", regex=True)
df = df.merge(ds, on=["DILLER_CODE"], how="left")

# สร้าง Column GROUP
df['GROUP'] = df['PROVINCE_NAME'].fillna('').astype(str) + df['กลุ่ม'].fillna('').astype(str)

# ==========================================
# 5. คำนวณค่าฐานนิยม (Mode)
# ==========================================
def get_mode(series):
    modes = series.mode()
    if modes.empty: return None
    return modes.tolist() if len(modes) > 1 else modes.iloc[0]

def clean_fmt(x):    
    if x is None or (isinstance(x, float) and np.isnan(x)) or x == "": return ""    
    try:
        val = float(x)      
        return str(int(val)) if val.is_integer() else str(val)
    except: return str(x)

# คำนวณ Mode และ Merge
for col in ['AVG', 'REL']:
    mode_df = df.groupby(['COMMODITY_CODE', 'ประเภทร้านค้า'])[col].agg(get_mode).reset_index().rename(columns={col: f'{col}_ฐานนิยม'})
    df = df.merge(mode_df, on=['COMMODITY_CODE', 'ประเภทร้านค้า'], how='left')
    df[f'{col}_ฐานนิยม'] = df[f'{col}_ฐานนิยม'].apply(lambda x: ', '.join([clean_fmt(v) for v in x]) if isinstance(x, list) else clean_fmt(x))

# ให้ราคากลาง เป็นแทนที่ฐานนิยมร้าน
df.loc[(df['ราคากลาง']>0),'AVG_ฐานนิยม'] = df['ราคากลาง']



# คำนวณผลต่าง (ใช้ Temp columns แล้วลบ)
df['AVG_ฐานนิยม_temp'] = pd.to_numeric(df['AVG_ฐานนิยม'], errors='coerce')
df['ผลต่างAVG'] = abs(df['AVG'] - df['AVG_ฐานนิยม_temp'])

df['REL_ฐานนิยม_temp'] = pd.to_numeric(df['REL_ฐานนิยม'], errors='coerce')
df['ผลต่างREL'] = abs(df['REL'] - df['REL_ฐานนิยม_temp'])

df = df.drop(columns=['AVG_ฐานนิยม_temp', 'REL_ฐานนิยม_temp'])

# ==========================================
# 6. รายงานจังหวัดที่ขาดส่ง
# ==========================================
all_groups_set = set(df['GROUP'].unique()) 

def get_missing_groups(current_groups):
    missing = all_groups_set - set(current_groups)
    exclude_words = {
        'กทม.ซีเจมอร์oneprice', 'กรุงเทพมหานครกลุ่มเขตพิเศษ',
        'กรุงเทพมหานครบิ๊กซีoneprice', 'กรุงเทพมหานครโลตัสoneprice',
        'กรุงเทพมหานครซีเจมอร์oneprice', 'กรุงเทพมหานครกลุ่มประตูน้ำ'
    }
    filtered = [str(x) for x in missing if str(x).strip() not in exclude_words]
    return ', '.join(filtered) if filtered else ""

missing_df = df.groupby('CODE_7_DIGITS')['GROUP'].apply(get_missing_groups).reset_index(name='รายงานจังหวัดที่ขาดส่ง')
df = df.merge(missing_df, on='CODE_7_DIGITS', how='left')

# ==========================================
# 7. Merge Stats & Calculate Rank
# ==========================================
# 7.1 Merge Stats
stats_cols = [
    'COMMODITY_CODE', 'DILLER_CODE', 'CountAVG',
    'AVGtail_stable', 'AVG_Max', 'AVG_Min', 'AVG_maxdiff', 'AVG_mindiff', 
    'Rel_Max', 'Rel_Min', 'Reltrail_eq100'
]
stats_df = dfs_[stats_cols].copy()
stats_df['COMMODITY_CODE'] = stats_df['COMMODITY_CODE'].astype(str).str.zfill(16)
stats_df['DILLER_CODE']    = stats_df['DILLER_CODE'].astype(str).str.zfill(10)
stats_df = stats_df.drop_duplicates(subset=['COMMODITY_CODE', 'DILLER_CODE'])
df = df.merge(stats_df, on=['COMMODITY_CODE', 'DILLER_CODE'], how='left')

# 7.2 Calculate Rank (ลงใน Main DF เลย)
# สร้างตารางช่วยคำนวณ Rank (ระดับสินค้า)
rank_ref = df[['CODE_7_DIGITS', 'COMMODITY_CODE', 'จำนวนสินค้า']].drop_duplicates()
rank_ref['Rank'] = rank_ref.groupby('CODE_7_DIGITS')['จำนวนสินค้า'].rank(method='first', ascending=False).astype(int)
# Merge Rank กลับเข้า df หลัก
df = df.merge(rank_ref[['COMMODITY_CODE', 'Rank']], on='COMMODITY_CODE', how='left')

# ==========================================
# 8. Export Excel (สินค้ายอดนิยม)
# ==========================================
def export_popularity_excel(main_df, file_name):
    # เตรียมข้อมูลสำหรับ Excel (Unique Items + Rank)
    dfpop = main_df[['CODE_7_DIGITS', 'COMMODITY_CODE', 'DESCRIPTION', 'จำนวนสินค้า', 'Rank']].copy()
    dfpop = dfpop.drop_duplicates().sort_values(['CODE_7_DIGITS', 'Rank'], ascending=[True, True])
    
    wb = Workbook()
    ws = wb.active
    ws.title = f"{file_name}_ยอดนิยม"

    # Write Data
    for r in dataframe_to_rows(dfpop, index=False, header=True):
        ws.append(r)

    # Styles
    green_fill = PatternFill(start_color="C6EFCE", end_color="C6EFCE", fill_type="solid")
    blue_fill = PatternFill(start_color="87CEEB", end_color="87CEEB", fill_type="solid")
    bold_font = Font(bold=True)
    thick_border_bottom = Border(bottom=Side(style='medium'))

    # Header Style
    for cell in ws[1]:
        cell.font = bold_font
        cell.fill = blue_fill

    # Conditional Formatting & Borders
    max_row = ws.max_row
    max_col = ws.max_column
    
    for row_num in range(2, max_row + 1):
        # Highlight Top 5
        rank_val = ws.cell(row=row_num, column=5).value
        if rank_val and 1 <= rank_val <= 5:
            for col in range(1, max_col + 1):
                ws.cell(row=row_num, column=col).fill = green_fill
        
        # Border separation
        current_code = ws.cell(row=row_num, column=1).value
        next_code = ws.cell(row=row_num + 1, column=1).value if row_num < max_row else None
        
        if current_code != next_code:
            for col in range(1, max_col + 1):
                ws.cell(row=row_num, column=col).border = thick_border_bottom

    # Column Widths
    def set_width(col_name, width):
        try:
            col_idx = dfpop.columns.get_loc(col_name) + 1
            ws.column_dimensions[get_column_letter(col_idx)].width = width
        except: pass

    set_width('CODE_7_DIGITS', 8)
    set_width('COMMODITY_CODE', 18)
    set_width('DESCRIPTION', 60)
    set_width('จำนวนสินค้า', 9)
    set_width('Rank', 10)

    ws.auto_filter.ref = ws.dimensions
    ws.freeze_panes = "A2"
    
    output_file = f"{file_name}_สินค้ายอดนิยม.xlsx"
    wb.save(output_file)
    print(f"บันทึกไฟล์ Excel: {output_file} เรียบร้อยแล้ว")

# เรียกใช้ฟังก์ชัน Export Excel
# (ต้องมั่นใจว่าตัวแปร filename มีค่าอยู่แล้ว)
try:
    export_popularity_excel(df, filename)
except NameError:
    print("Warning: ไม่พบตัวแปร 'filename' ข้ามการ Export Excel")

# ==========================================
# 9. จัดเรียงคอลัมน์และส่งออกผลลัพธ์สุดท้าย
# ==========================================
# จัดลำดับคอลัมน์ให้ Rank ไปอยู่หลัง 'จำนวนสินค้า'
cols = list(df.columns)
if 'Rank' in cols and 'จำนวนสินค้า' in cols:
    cols.remove('Rank')
    insert_idx = cols.index('จำนวนสินค้า') + 1
    cols.insert(insert_idx, 'Rank')
    df = df[cols]

# Sort ตามความต้องการ
df = df.sort_values(by=['CODE_7_DIGITS', 'Rank', 'ประเภทร้านค้า', 'AVG'], ascending=[True, True, True, False])


# ==========================================
# 10. ตรวจสอบ Modern Trade (ส่วนที่แก้ไข)
# ==========================================
# [แก้ไข] สร้าง column มารอไว้ก่อน เพื่อป้องกัน Error
# df['ผลการตรวจสอบModernTrade'] = None 
df['ผลการตรวจสอบModernTrade'] = None

# พี่ ณรงค์ Logic
mask_mt_price = (df['Online'] == 'Online') & (df['LINK'] == 0) & (df['ประเภทร้านค้า'] != 'ร้านอื่นๆ') & (df['ผลต่างAVG'] > 0)
df.loc[mask_mt_price, 'ผลการตรวจสอบModernTrade'] = 'ให้ตรวจสอบ ราคา(ModernTrade)'

mask_mt_rel = (df['Online'] == 'Online') & (df['LINK'] == 0) & (df['ประเภทร้านค้า'] != 'ร้านอื่นๆ') & (df['ผลต่างREL'] > 0)
df.loc[mask_mt_rel, 'ผลการตรวจสอบModernTrade'] = 'ให้ตรวจสอบ REL(ModernTrade)'

# (Optional) ถ้าต้องการให้ Column นี้ไม่มี NaN แต่เป็นค่าว่างเปล่าๆ ให้เปิดบรรทัดนี้
# df['ผลการตรวจสอบModernTrade'] = df['ผลการตรวจสอบModernTrade'].fillna("")

df = df.reset_index(drop=True)


print("Process Completed: Main DataFrame copied to clipboard.")


บันทึกไฟล์ Excel: cpig_month_2569-2_สินค้ายอดนิยม.xlsx เรียบร้อยแล้ว
Process Completed: Main DataFrame copied to clipboard.


In [17]:
# df

#### ราคากลาง

In [18]:
sent_modern_trade_price = df.copy()
sent_modern_trade_price = sent_modern_trade_price[(sent_modern_trade_price['ผลต่างราคากับราคากลาง']>0) | (((sent_modern_trade_price['ราคากลาง']>0))&(sent_modern_trade_price['AVG'].isna()))].reset_index(drop=True)
sent_modern_trade_price = sent_modern_trade_price[['ผู้ดูแล','G_FLAG','L_FLAG','N_FLAG','TOTAL_FLAGS','CODE7','CODE9','COMMODITY_CODE','DILLER_CODE','DESCRIPTION','SHOP_NAME','ภาค','GROUP','ประเภทร้านค้า','LAST_PRICE','AVG','LINK','COMPARE_PRICE','REL','COMM1','COMM2','Online','ราคากลาง','ผลต่างราคากับราคากลาง']]

sent_modern_trade_price['SemiCompositeKey'] = sent_modern_trade_price['COMMODITY_CODE'] + sent_modern_trade_price['DILLER_CODE'] 
sent_modern_trade_price['FullCompositeKey'] = sent_modern_trade_price['COMMODITY_CODE'] + sent_modern_trade_price['DILLER_CODE'] + sent_modern_trade_price['DESCRIPTION']  + sent_modern_trade_price['SHOP_NAME']

# --- 1. ค้นหา DILLER_CODE ที่มี SHOP_NAME ไม่ซ้ำกันมากกว่า 1 ---
shop_count_by_diller = sent_modern_trade_price.groupby('DILLER_CODE')['SHOP_NAME'].nunique()
duplicate_diller_codes = shop_count_by_diller[shop_count_by_diller > 1].index.tolist()
sent_modern_trade_price['Codeซ้ำ หลายชื่อร้าน'] = 'No'
sent_modern_trade_price.loc[
    sent_modern_trade_price['DILLER_CODE'].isin(duplicate_diller_codes), 
    'Codeซ้ำ หลายชื่อร้าน'
] = 'Yes'
print("✅ DataFrame หลัก (sent_modern_trade_price) พร้อม Label ความซ้ำซ้อน:")

# จัดเรียงเพื่อให้เห็นรายการที่ 'Yes' อยู่ติดกัน (ทางเลือก)
df_display = sent_modern_trade_price.sort_values(
    by=['Codeซ้ำ หลายชื่อร้าน', 'DILLER_CODE'], 
    ascending=[False, True]
)

description_count_by_commodity = sent_modern_trade_price.groupby('COMMODITY_CODE')['DESCRIPTION'].nunique()
duplicate_commodity_codes = description_count_by_commodity[description_count_by_commodity > 1].index.tolist()

# --- 2. Label ข้อมูลใน DataFrame หลัก (sent_modern_trade_price) ---
sent_modern_trade_price['รหัสซ้ำ หลายชื่อรายการ'] = 'No'
sent_modern_trade_price.loc[
    sent_modern_trade_price['COMMODITY_CODE'].isin(duplicate_commodity_codes), 
    'รหัสซ้ำ หลายชื่อรายการ'
] = 'Yes'

# --- 3. แสดงผลลัพธ์ (แสดงทั้ง DataFrame หลัก) ---
print("✅ เพิ่ม Label 'รหัสซ้ำ หลายชื่อรายการ' ใน DataFrame เรียบร้อยแล้ว")
print("รายการที่เข้าเงื่อนไข COMMODITY_CODE ซ้ำ, DESCRIPTION ต่างกัน:")

# จัดเรียงเพื่อให้เห็นรายการที่ 'Yes' อยู่ติดกัน
df_display_commodity = sent_modern_trade_price.sort_values(
    by=['รหัสซ้ำ หลายชื่อรายการ', 'COMMODITY_CODE'], 
    ascending=[False, True]
)

sent_modern_trade_price = sent_modern_trade_price[
    (sent_modern_trade_price['Codeซ้ำ หลายชื่อร้าน'] == 'No') & 
    (sent_modern_trade_price['รหัสซ้ำ หลายชื่อรายการ'] == 'No')]

# sent_modern_trade_price = sent_modern_trade_price[sent_modern_trade_price['COMM1'].isna()].reset_index(drop=True)
# 4.2. เลือกเฉพาะคอลัมน์ที่ต้องการส่งออก (แก้ไข Syntax Error)

# sent_modern_trade_price = sent_modern_trade_price[sent_modern_trade_price['TOTAL_FLAGS']=='คำนวณ']

sent_modern_trade_price = sent_modern_trade_price[sent_modern_trade_price['LINK']!=3] #ไม่เอา link3 เพราะเดี๋ยวจะป่นกับ seris

sent_modern_trade_price = sent_modern_trade_price[['ผู้ดูแล','TOTAL_FLAGS','COMMODITY_CODE', 'DILLER_CODE', 'DESCRIPTION', 'SHOP_NAME','ภาค','GROUP','Online','ราคากลาง','AVG','COMM1','COMM2']]
sent_modern_trade_price  = sent_modern_trade_price.drop_duplicates().reset_index(drop=True)

import pandas as pd
from openpyxl.styles import PatternFill, Font, Alignment 
# ต้องแน่ใจว่าได้ติดตั้ง openpyxl แล้ว: pip install openpyxl

# สมมติว่า sent_modern_trade_price และ filename ได้ถูกกำหนดค่าแล้ว
try:
    filename
except NameError:
    filename = 'Data_Filtered' 

# -------------------------------------------------------------
# **โค้ดส่วน Label และกรองข้อมูลของคุณ (ย่อไว้เพื่อความกระชับ)**
# -------------------------------------------------------------

output_filename = filename + '_ราคากลาง_ส่วนกลาง.xlsx'
MAX_DESCRIPTION_WIDTH = 30 
MAX_SHOP_NAME_WIDTH = 15  

# 1. สร้าง ExcelWriter object และใช้ openpyxl engine
writer = pd.ExcelWriter(output_filename, engine='openpyxl')
sheet_name = 'รายการที่ผ่านการกรอง'

# 2. ส่งออก DataFrame ไปยัง Excel
sent_modern_trade_price.to_excel(writer, sheet_name=sheet_name, index=False)

# 3. เข้าถึง Workbook และ Worksheet เพื่อใช้ openpyxl จัดรูปแบบ
workbook  = writer.book
worksheet = writer.sheets[sheet_name]

# --- การตั้งค่า Workbook พิเศษ (Freeze, Filter, Widths) ---

# 4. Freeze Pane (ตรึงแถวแรก)
worksheet.freeze_panes = 'A2'

# 5. AutoFilter (เพิ่มปุ่ม Filter)
max_col_letter = worksheet.cell(row=1, column=len(sent_modern_trade_price.columns)).column_letter
worksheet.auto_filter.ref = f"A1:{max_col_letter}1"


# 6. กำหนดรูปแบบใหม่สำหรับ Header และ Alignment

header_fill = PatternFill(start_color='ADD8E6', end_color='ADD8E6', fill_type='solid')
header_font = Font(bold=True)
center_alignment = Alignment(horizontal='center')
right_alignment = Alignment(horizontal='right') # ใช้สำหรับราคากลาง

# 6.2. นำสไตล์ไปใช้กับแถว Header
header_row = worksheet[1] 
for cell in header_row:
    cell.fill = header_fill
    cell.font = header_font
    # ตรึงและใส่ Filter ให้กับตารางจะทำให้ตารางดูเป็นระเบียบขึ้น 


# 7. Auto-fit Column Width และกำหนดรูปแบบเฉพาะคอลัมน์ (สีแดง/เขียว)
header_names = sent_modern_trade_price.columns.tolist()

# เตรียมรูปแบบ Font ที่จะใช้
red_font = Font(color="FF0000")    # สีแดง สำหรับ AVG
green_font = Font(color="008000")  # สีเขียว สำหรับ ราคากลาง

for col_idx, column in enumerate(worksheet.columns, 1):
    max_length = 0
    column_letter = column[0].column_letter
    header_name = column[0].value 

    # 7.1 คำนวณความกว้าง (เหมือนเดิม)
    for cell in column:
        try:
            if cell.value and len(str(cell.value)) > max_length:
                max_length = len(str(cell.value))
        except:
            pass
    
    # --- ส่วนที่เพิ่มเติม: ใส่สีตามเงื่อนไข ---
    
    # กรณีคอลัมน์ 'AVG' -> ใส่ตัวอักษรสีแดง
    if header_name == 'AVG':
        for cell in column[1:]:  # [1:] เพื่อข้าม Header (แถวแรก) ไม่ให้โดนเปลี่ยนสี
            cell.font = red_font

    # กรณีคอลัมน์ 'ราคากลาง' -> ใส่ตัวอักษรสีเขียว และ จัดกึ่งกลาง
    elif header_name == 'ราคากลาง':
        for cell in column[1:]:  # [1:] ข้าม Header
            cell.font = green_font
            cell.alignment = center_alignment
        
        # อย่าลืมจัด Header ให้กึ่งกลางด้วย (ถ้าต้องการ)
        column[0].alignment = center_alignment

    # 7.2 กำหนดความกว้าง (Logic เดิม)
    if header_name == 'DESCRIPTION':
        adjusted_width = min(max_length + 2, MAX_DESCRIPTION_WIDTH) 
    elif header_name == 'SHOP_NAME':
        adjusted_width = min(max_length + 2, MAX_SHOP_NAME_WIDTH)
    else:
        adjusted_width = (max_length + 5)
    
    worksheet.column_dimensions[column_letter].width = adjusted_width

# 8. บันทึกและปิดไฟล์ Excel
writer.close()

print(f"\n✅ สร้างไฟล์ Excel พร้อมการจัดรูปแบบเสร็จสมบูรณ์:")
print(f"   - พื้นหลัง Header: สีฟ้าอ่อน")
print(f"   - คอลัมน์ 'AVG': ตัวหนังสือสีแดง")
print(f"   - คอลัมน์ 'ราคากลาง': ตัวหนังสือสีเขียว & จัดกึ่งกลาง")
print(f"ไฟล์ถูกบันทึกที่: {output_filename}")


✅ DataFrame หลัก (sent_modern_trade_price) พร้อม Label ความซ้ำซ้อน:
✅ เพิ่ม Label 'รหัสซ้ำ หลายชื่อรายการ' ใน DataFrame เรียบร้อยแล้ว
รายการที่เข้าเงื่อนไข COMMODITY_CODE ซ้ำ, DESCRIPTION ต่างกัน:

✅ สร้างไฟล์ Excel พร้อมการจัดรูปแบบเสร็จสมบูรณ์:
   - พื้นหลัง Header: สีฟ้าอ่อน
   - คอลัมน์ 'AVG': ตัวหนังสือสีแดง
   - คอลัมน์ 'ราคากลาง': ตัวหนังสือสีเขียว & จัดกึ่งกลาง
ไฟล์ถูกบันทึกที่: cpig_month_2569-2_ราคากลาง_ส่วนกลาง.xlsx


#### ส่งแทนที่ราคากลาง

In [19]:
sent_modern_trade_price = df.copy()
sent_modern_trade_price = sent_modern_trade_price[(sent_modern_trade_price['ผลต่างราคากับราคากลาง']>0) | (((sent_modern_trade_price['ราคากลาง']>0))&(sent_modern_trade_price['AVG'].isna()))].reset_index(drop=True)
sent_modern_trade_price = sent_modern_trade_price[['G_FLAG','L_FLAG','N_FLAG','CODE7','CODE9','COMMODITY_CODE','DILLER_CODE','DESCRIPTION','SHOP_NAME','ภาค','GROUP','ประเภทร้านค้า','LAST_PRICE','AVG','LINK','COMPARE_PRICE','REL','COMM1','COMM2','ราคากลาง','ผลต่างราคากับราคากลาง']]

sent_modern_trade_price['SemiCompositeKey'] = sent_modern_trade_price['COMMODITY_CODE'] + sent_modern_trade_price['DILLER_CODE'] 
sent_modern_trade_price['FullCompositeKey'] = sent_modern_trade_price['COMMODITY_CODE'] + sent_modern_trade_price['DILLER_CODE'] + sent_modern_trade_price['DESCRIPTION']  + sent_modern_trade_price['SHOP_NAME']

# --- 1. ค้นหา DILLER_CODE ที่มี SHOP_NAME ไม่ซ้ำกันมากกว่า 1 ---
sent_modern_trade_price
# จัดกลุ่ม DILLER_CODE และนับจำนวน SHOP_NAME ที่ไม่ซ้ำกัน (nunique)
shop_count_by_diller = sent_modern_trade_price.groupby('DILLER_CODE')['SHOP_NAME'].nunique()

# กรองเอาเฉพาะ DILLER_CODE ที่มี SHOP_NAME > 1 (คือ DILLER_CODE ที่ซ้ำกันในหลายร้าน)
duplicate_diller_codes = shop_count_by_diller[shop_count_by_diller > 1].index.tolist()

# --- 2. Label ข้อมูลใน DataFrame หลัก (sent_modern_trade_price) ---

# 2.1. สร้างคอลัมน์ใหม่ 'คนละชื่อเเต่ซ้ำกัน' ใน DataFrame หลัก
#      กำหนดค่าเริ่มต้นให้เป็น 'No' หรือว่างไว้ก่อน
sent_modern_trade_price['Codeซ้ำ หลายชื่อร้าน'] = 'No'

# 2.2. ใช้ .loc เพื่อกำหนดค่า 'Yes' ให้กับแถวที่ DILLER_CODE ตรงกับรายการที่ซ้ำกัน
sent_modern_trade_price.loc[
    sent_modern_trade_price['DILLER_CODE'].isin(duplicate_diller_codes), 
    'Codeซ้ำ หลายชื่อร้าน'
] = 'Yes'


# --- 3. แสดงผลลัพธ์ (แสดงทั้ง DataFrame หลัก) ---

print("✅ DataFrame หลัก (sent_modern_trade_price) พร้อม Label ความซ้ำซ้อน:")

# จัดเรียงเพื่อให้เห็นรายการที่ 'Yes' อยู่ติดกัน (ทางเลือก)
df_display = sent_modern_trade_price.sort_values(
    by=['Codeซ้ำ หลายชื่อร้าน', 'DILLER_CODE'], 
    ascending=[False, True]
)

# sent_modern_trade_price.to_clipboard(index=False)

# ใช้ .to_string() เพื่อแสดงทุกแถวในคอนโซลอย่างชัดเจน
# print(df_display[['DILLER_CODE', 'SHOP_NAME', 'DESCRIPTION', 'คนละชื่อเเต่ซ้ำกัน']].to_string())

# 4. คัดลอก DataFrame หลักที่เพิ่มคอลัมน์แล้วไปยัง Clipboard
# sent_modern_trade_price.to_excel(filename + '_คุณต้น.xlsx',index=False)

# สมมติว่า sent_modern_trade_price ถูกกำหนดค่าและมีข้อมูลที่เพิ่มแถวซ้ำแล้ว

# --- 1. ค้นหา COMMODITY_CODE ที่มี DESCRIPTION ไม่ซ้ำกันมากกว่า 1 ---

# จัดกลุ่มข้อมูล (Group By) ด้วย 'COMMODITY_CODE'
# และนับจำนวน 'DESCRIPTION' ที่ไม่ซ้ำกัน (nunique) สำหรับแต่ละ COMMODITY_CODE
description_count_by_commodity = sent_modern_trade_price.groupby('COMMODITY_CODE')['DESCRIPTION'].nunique()

# กรองเอาเฉพาะ COMMODITY_CODE ที่มีจำนวน DESCRIPTION ที่ไม่ซ้ำกัน (nunique) มากกว่า 1
# (นั่นคือ COMMODITY_CODE นั้นปรากฏในชื่อสินค้ามากกว่าหนึ่งชื่อ)
duplicate_commodity_codes = description_count_by_commodity[description_count_by_commodity > 1].index.tolist()

# --- 2. Label ข้อมูลใน DataFrame หลัก (sent_modern_trade_price) ---

# 2.1. สร้างคอลัมน์ใหม่ 'รหัสซ้ำ หลายชื่อรายการ' ใน DataFrame หลัก
#      กำหนดค่าเริ่มต้นให้เป็น 'No'
sent_modern_trade_price['รหัสซ้ำ หลายชื่อรายการ'] = 'No'

# 2.2. ใช้ .loc เพื่อกำหนดค่า 'Yes' ให้กับแถวที่ COMMODITY_CODE ตรงกับรายการที่ซ้ำกัน
sent_modern_trade_price.loc[
    sent_modern_trade_price['COMMODITY_CODE'].isin(duplicate_commodity_codes), 
    'รหัสซ้ำ หลายชื่อรายการ'
] = 'Yes'

# --- 3. แสดงผลลัพธ์ (แสดงทั้ง DataFrame หลัก) ---

print("✅ เพิ่ม Label 'รหัสซ้ำ หลายชื่อรายการ' ใน DataFrame เรียบร้อยแล้ว")
print("รายการที่เข้าเงื่อนไข COMMODITY_CODE ซ้ำ, DESCRIPTION ต่างกัน:")

# จัดเรียงเพื่อให้เห็นรายการที่ 'Yes' อยู่ติดกัน
df_display_commodity = sent_modern_trade_price.sort_values(
    by=['รหัสซ้ำ หลายชื่อรายการ', 'COMMODITY_CODE'], 
    ascending=[False, True]
)

sent_modern_trade_price = sent_modern_trade_price[
    (sent_modern_trade_price['Codeซ้ำ หลายชื่อร้าน'] == 'No') & 
    (sent_modern_trade_price['รหัสซ้ำ หลายชื่อรายการ'] == 'No')]

sent_modern_trade_price = sent_modern_trade_price[sent_modern_trade_price['COMM1'].isna()].reset_index(drop=True)

# 4.2. เลือกเฉพาะคอลัมน์ที่ต้องการส่งออก (แก้ไข Syntax Error)
sent_modern_trade_price = sent_modern_trade_price[['COMMODITY_CODE', 'DILLER_CODE', 'DESCRIPTION', 'SHOP_NAME', 'ราคากลาง']]

sent_modern_trade_price  = sent_modern_trade_price.drop_duplicates().reset_index(drop=True)
# sent_modern_trade_price.to_excel(filename + '_คุณต้น.xlsx', index=False)
# print(f"\nไฟล์ถูกบันทึกที่: {filename}_คุณต้น.xlsx")
sent_modern_trade_price.to_clipboard()


✅ DataFrame หลัก (sent_modern_trade_price) พร้อม Label ความซ้ำซ้อน:
✅ เพิ่ม Label 'รหัสซ้ำ หลายชื่อรายการ' ใน DataFrame เรียบร้อยแล้ว
รายการที่เข้าเงื่อนไข COMMODITY_CODE ซ้ำ, DESCRIPTION ต่างกัน:


In [20]:
import pandas as pd
from openpyxl.styles import PatternFill, Font, Alignment 
# ต้องแน่ใจว่าได้ติดตั้ง openpyxl แล้ว: pip install openpyxl

# สมมติว่า sent_modern_trade_price และ filename ได้ถูกกำหนดค่าแล้ว
try:
    filename
except NameError:
    filename = 'Data_Filtered' 

# -------------------------------------------------------------
# **โค้ดส่วน Label และกรองข้อมูลของคุณ (ย่อไว้เพื่อความกระชับ)**
# -------------------------------------------------------------

output_filename = filename + '_ราคากลาง_คุณต้น.xlsx'
MAX_DESCRIPTION_WIDTH = 30 
MAX_SHOP_NAME_WIDTH = 15  

# 1. สร้าง ExcelWriter object และใช้ openpyxl engine
writer = pd.ExcelWriter(output_filename, engine='openpyxl')
sheet_name = 'รายการที่ผ่านการกรอง'

# 2. ส่งออก DataFrame ไปยัง Excel
sent_modern_trade_price.to_excel(writer, sheet_name=sheet_name, index=False)

# 3. เข้าถึง Workbook และ Worksheet เพื่อใช้ openpyxl จัดรูปแบบ
workbook  = writer.book
worksheet = writer.sheets[sheet_name]

# --- การตั้งค่า Workbook พิเศษ (Freeze, Filter, Widths) ---

# 4. Freeze Pane (ตรึงแถวแรก)
worksheet.freeze_panes = 'A2'

# 5. AutoFilter (เพิ่มปุ่ม Filter)
max_col_letter = worksheet.cell(row=1, column=len(sent_modern_trade_price.columns)).column_letter
worksheet.auto_filter.ref = f"A1:{max_col_letter}1"


# 6. กำหนดรูปแบบใหม่สำหรับ Header และ Alignment

# 6.1. สร้างสไตล์สำหรับพื้นหลัง Header (สีฟ้าอ่อน)
# 'ADD8E6' คือรหัส HEX ของ Light Blue (สามารถปรับเปลี่ยนได้)
header_fill = PatternFill(start_color='ADD8E6', end_color='ADD8E6', fill_type='solid')
header_font = Font(bold=True)
center_alignment = Alignment(horizontal='center')
right_alignment = Alignment(horizontal='right') # ใช้สำหรับราคากลาง

# 6.2. นำสไตล์ไปใช้กับแถว Header
header_row = worksheet[1] 
for cell in header_row:
    cell.fill = header_fill
    cell.font = header_font
    # ตรึงและใส่ Filter ให้กับตารางจะทำให้ตารางดูเป็นระเบียบขึ้น 


# 7. Auto-fit Column Width และจำกัดความกว้าง (รวมถึงการจัด Alignment)
header_names = sent_modern_trade_price.columns.tolist()
price_col_index = header_names.index('ราคากลาง') + 1 # หาตำแหน่งคอลัมน์ราคากลาง (บวก 1 เพราะ openpyxl เริ่มที่ 1)

for col_idx, column in enumerate(worksheet.columns, 1):
    max_length = 0
    column_letter = column[0].column_letter
    header_name = column[0].value 

    # 7.1. คำนวณความยาวสูงสุด
    for cell in column:
        try:
            if cell.value and len(str(cell.value)) > max_length:
                max_length = len(str(cell.value))
        except:
            pass
    
    # 7.2. จัด Alignment สำหรับ 'ราคากลาง'
    if header_name == 'ราคากลาง':
        # จัดให้อยู่ตรงกลาง (Center)
        for cell in column:
            cell.alignment = center_alignment
    
    # 7.3. กำหนดความกว้างตามเงื่อนไข
    if header_name == 'DESCRIPTION':
        adjusted_width = min(max_length + 2, MAX_DESCRIPTION_WIDTH) 
    elif header_name == 'SHOP_NAME':
        adjusted_width = min(max_length + 2, MAX_SHOP_NAME_WIDTH)
    else:
        adjusted_width = (max_length + 5)
    
    worksheet.column_dimensions[column_letter].width = adjusted_width

# 8. บันทึกและปิดไฟล์ Excel
writer.close()

print(f"\n✅ สร้างไฟล์ Excel พร้อมการจัดรูปแบบและจำกัดความกว้างเสร็จสมบูรณ์:")
print(f"   - พื้นหลัง Header: สีฟ้าอ่อน")
print(f"   - คอลัมน์ 'ราคากลาง': จัดตำแหน่งกลาง")
print(f"ไฟล์ถูกบันทึกที่: {output_filename}")


✅ สร้างไฟล์ Excel พร้อมการจัดรูปแบบและจำกัดความกว้างเสร็จสมบูรณ์:
   - พื้นหลัง Header: สีฟ้าอ่อน
   - คอลัมน์ 'ราคากลาง': จัดตำแหน่งกลาง
ไฟล์ถูกบันทึกที่: cpig_month_2569-2_ราคากลาง_คุณต้น.xlsx


#### Report เบื้องต้น + Zscore + IQR

In [21]:
new_cols = [
    'นับตัวซ้ำ', 'ผู้ดูแล','USERID','MONTH', 'YEAR', 'TYPE', 
    'C_FLAG', 'G_FLAG', 'L_FLAG', 'N_FLAG', 'R_FLAG', 'NR_FLAG', 'TOTAL_FLAGS', 'ComKeyAdv',
    'ComKeyBasic', 'CODE_7_DIGITS','รายการ','CODE7','CODE9', 'COMMODITY_CODE', 'DILLER_CODE', 'TYPE_CODE', 
    'DESCRIPTION', 'จำนวนสินค้า', 'Rank', 'ขนาดร้านค้า', 'จำนวนร้าน(ขนาดร้านค้า)', 
    'ภาค', 'กลุ่ม', 'GROUP','SHOP_NAME', 'ประเภทร้านค้า', 'จำนวนร้าน(ประเภทร้านค้า)', 
    'PROVINCE_NAME',  'GROUP_KEY', 
    'LAST_PRICE', 'Online','ราคากลาง',  'ผลต่างราคากับราคากลาง','AVG','LINK', 'COMPARE_PRICE', 'REL', 
    'COMM1', 'COMM2', # <--- เติม comma ให้แล้ว
    'AVG_ฐานนิยม', 'REL_ฐานนิยม', 
    'ผลต่างAVG', 'ผลต่างREL', 'รายงานจังหวัดที่ขาดส่ง', 'CountAVG', 
    'AVGtail_stable', 'AVG_Max', 'AVG_Min', 'AVG_maxdiff', 'AVG_mindiff', 
    'Rel_Max', 'Rel_Min', 'Reltrail_eq100', 'ผลการตรวจสอบModernTrade'
    # หมายเหตุ: P_FLAG, TOTAL_FLAGS, 
].copy()

df_sorted = df.reindex(columns=new_cols)
# df_sorted = df_sorted[df_sorted['TOTAL_FLAGS']=='คำนวณ']
# # Excel_Condition2(df_sorted ,'test.xlsx')
# # df_sorted

# df[df['TOTAL_FLAGS']=='ไม่คำนวณ']


In [22]:
import pandas as pd
import numpy as np
import warnings

# ==========================================
# 1. (ตัดฟังก์ชัน Box-Cox ทิ้งไปเลย)
# ==========================================

# ==========================================
# 2. เตรียมข้อมูล
# ==========================================
mask_valid = (df_sorted['AVG'] > 0) & (df_sorted['AVG'].notna())
process_df = df_sorted[mask_valid].copy()
ignored_df = df_sorted[~mask_valid].copy()

# ==========================================
# 3. คำนวณ Robust Z-Score (Vectorized)
# ==========================================

# --- 3.1 คำนวณ Median และ MAD ต่อกลุ่ม ---
# กำหนด Grouper (คุณใช้อันไหนเป็นหลัก เลือกใช้อันนั้นได้เลยครับ)
# ในที่นี้ทำให้ดูทั้ง Global และ Local

cols_group = ['CODE_7_DIGITS', 'DESCRIPTION']
grouper = process_df.groupby(cols_group)['AVG']

# 1. หา Median (ตัวแทนค่ากลางที่ทนต่อความเบ้)
process_df['Median_Global'] = grouper.transform('median')

# 2. หาค่าเบี่ยงเบนสัมบูรณ์ (Absolute Deviation)
process_df['Abs_Dev'] = (process_df['AVG'] - process_df['Median_Global']).abs()

# 3. หา MAD (Median Absolute Deviation) ของกลุ่มนั้น
process_df['MAD_Global'] = process_df.groupby(cols_group)['Abs_Dev'].transform('median')

# --- 3.2 คำนวณ Modified Z-Score ---
# สูตร: 0.6745 * (X - Median) / MAD
# ค่า 0.6745 คือค่าคงที่เพื่อให้ MAD สอดคล้องกับ SD ใน Normal Dist.

# ป้องกันการหารด้วย 0 (กรณีสินค้าในกลุ่มราคาเท่ากันหมด MAD จะเป็น 0)
process_df['MAD_Global'] = process_df['MAD_Global'].replace(0, np.nan) 

process_df['Z_Score_Global'] = (0.6745 * (process_df['AVG'] - process_df['Median_Global'])) / process_df['MAD_Global']

# Fillna(0) สำหรับเคสที่ MAD=0 หรือข้อมูลไม่พอ
process_df['Sum_Z-Score'] = process_df['Z_Score_Global'].fillna(0).abs().round(2)

# ==========================================
# 4. ส่วนคำนวณอื่นๆ (คงเดิมเพราะเร็วอยู่แล้ว)
# ==========================================
# คำนวณสัดส่วน %
grouper_desc = process_df.groupby("DESCRIPTION")['AVG']
process_df['cnt_items'] = grouper_desc.transform('count')
process_df['pct_share'] = (process_df['AVG'] / grouper_desc.transform('sum') * 100).round(2)

# ==========================================
# 5. กฎการตรวจสอบ (Anomaly Detection)
# ==========================================
# ปรับ Threshold: ปกติ Modified Z-Score > 3.5 ถือว่าเป็น Outlier
# แต่ถ้าคุณใช้เกณฑ์เดิม (>2) ก็ใช้ได้ครับ แต่อาจจะเจอเยอะกว่าปกติเล็กน้อย

# --- Rule 1 ---
conditions = [ process_df['Sum_Z-Score'] > 3.5 ] # แนะนำที่ 3.5 สำหรับ Modified Z-Score
choices = [ 'ผิดปกติ(MZ)' ]

# --- Rule 2 (เงื่อนไขพิเศษ) ---
c_cnt = process_df['cnt_items']
c_pct = process_df['pct_share']
c_z   = process_df['Sum_Z-Score']

# ปรับ Threshold Z ใน Rule ย่อยตามความเหมาะสม
# เนื่องจาก Modified Z สเกลจะต่างจาก Standard Z เล็กน้อย 
# (ถ้าเดิมใช้ 1.0 อาจจะลองขยับเป็น 1.5 หรือทดสอบกับข้อมูลจริงดูครับ)
mask_rule3 = (c_cnt == 3) & ((c_pct < 19) | (c_pct > 51)) & (c_z > 1.5)
mask_rule4 = (c_cnt == 4) & ((c_pct < 14) | (c_pct > 41)) & (c_z > 2.0)
mask_rule5 = (c_cnt == 5) & ((c_pct < 12) | (c_pct > 32)) & (c_z > 2.5)

process_df['Flag_Special'] = np.select(
    [mask_rule3, mask_rule4, mask_rule5], 
    ['ผิดปกติ(MZ)', 'ผิดปกติ(MZ)', 'ผิดปกติ(MZ)'], 
    default=''
)

# --- Combine Flags ---
process_df['Report_Z>=2'] = np.select(conditions, choices, default='')
mask_empty_report = (process_df['Report_Z>=2'] == '')
process_df.loc[mask_empty_report, 'Report_Z>=2'] = process_df.loc[mask_empty_report, 'Flag_Special']

# ==========================================
# 6. Finalize (เหมือนเดิม)
# ==========================================
cols_to_drop = ['Median_Global', 'Abs_Dev', 'MAD_Global', 'Z_Score_Global', 
                'cnt_items', 'pct_share', 'Flag_Special']
process_df = process_df.drop(columns=cols_to_drop, errors='ignore')

df_sorted = pd.concat([process_df, ignored_df], ignore_index=True)

# Clean up
df_sorted.loc[df_sorted['ประเภทร้านค้า'] != 'ร้านอื่นๆ', 'Report_Z>=2'] = ''
dup_counts = df_sorted.groupby(['COMMODITY_CODE', 'AVG'])['AVG'].transform('count')
mask_dup_safe = (df_sorted['ประเภทร้านค้า'] == 'ร้านอื่นๆ') & (dup_counts >= 10)
df_sorted.loc[mask_dup_safe, 'Report_Z>=2'] = ''

df_sorted = df_sorted.sort_values(
    by=['COMMODITY_CODE', 'DESCRIPTION', 'ประเภทร้านค้า', 'AVG'], 
    ascending=[True, True, True, False]
).reset_index(drop=True)

# df_sorted.to_clipboard()
print("Process Completed with Robust Z-Score.")

Process Completed with Robust Z-Score.


In [23]:
# df_sorted.columns

In [24]:
import pandas as pd
import numpy as np

# ==============================================================================
# สร้าง Column ใหม่: Report_IQR (ใช้หลักการ Interquartile Range)
# ==============================================================================

# 1. เตรียมข้อมูลเฉพาะส่วนที่คำนวณได้ (AVG > 0)
mask_calc = (df_sorted['AVG'] > 0) & (df_sorted['AVG'].notna())

# สร้าง DataFrame ชั่วคราวเพื่อคำนวณ (ใช้ Index เดิมเพื่อ Mapping กลับ)
calc_df = df_sorted[mask_calc].copy()

# 2. คำนวณ Q1, Q3 และ IQR แยกตามกลุ่มสินค้า
# Group by CODE_7_DIGITS และ DESCRIPTION
grouper = calc_df.groupby(['CODE_7_DIGITS', 'DESCRIPTION'])['AVG']

# ใช้ transform เพื่อให้ค่ากลับมาเท่ากับจำนวนแถวเดิม
Q1 = grouper.transform(lambda x: x.quantile(0.25))
Q3 = grouper.transform(lambda x: x.quantile(0.75))
IQR = Q3 - Q1

# คำนวณขอบเขต (Bounds)
Lower_Bound = Q1 - (1.5 * IQR)
Upper_Bound = Q3 + (1.5 * IQR)

# 3. สร้างคอลัมน์ใหม่ใน df_sorted (ค่าเริ่มต้นเป็นว่าง)
df_sorted['Report_IQR'] = ''

# 4. ตรวจจับ Outlier
# เงื่อนไข: น้อยกว่าขอบล่าง OR มากกว่าขอบบน
mask_outlier_iqr = (calc_df['AVG'] < Lower_Bound) | (calc_df['AVG'] > Upper_Bound)

# Map ค่ากลับไปที่ df_sorted โดยอ้างอิง Index
# (เฉพาะแถวที่เข้าเงื่อนไข Outlier จะถูกกาว่า 'ผิดปกติ(IQR)')
df_sorted.loc[mask_outlier_iqr[mask_outlier_iqr].index, 'Report_IQR'] = 'ผิดปกติ(IQR)'

# ==============================================================================
# 5. ใส่กฎยกเว้น (Business Rules) *สำคัญ*
# ==============================================================================

# กฎที่ 1: ล้าง Flag ถ้าไม่ใช่ "ร้านอื่นๆ" (ร้านมาตรฐานไม่เช็ค)
df_sorted.loc[df_sorted['ประเภทร้านค้า'] != 'ร้านอื่นๆ', 'Report_IQR'] = ''

# กฎที่ 2: ล้าง Flag ถ้ามีราคาซ้ำกัน >= 2 ครั้ง (เฉพาะร้านอื่นๆ)
# (ถ้ามีคนขายราคานี้เหมือนกัน 2 ร้าน แสดงว่าเป็นราคาตลาดจริง ไม่ใช่ Outlier)
dup_counts = df_sorted.groupby(['COMMODITY_CODE', 'AVG'])['AVG'].transform('count')
mask_dup_safe = (df_sorted['ประเภทร้านค้า'] == 'ร้านอื่นๆ') & (dup_counts >= 10)
df_sorted.loc[mask_dup_safe, 'Report_IQR'] = ''

# ==============================================================================
# 6. จัดเรียงและส่งออก
# ==============================================================================
# จัดเรียงให้ดูง่าย
df_sorted = df_sorted.sort_values(
    by=['COMMODITY_CODE', 'DESCRIPTION', 'ประเภทร้านค้า', 'AVG'], 
    ascending=[True, True, True, False]
).reset_index(drop=True)

# ส่งออก
df_sorted.to_clipboard(index=False)
# print("เพิ่มคอลัมน์ Report_IQR เรียบร้อยแล้ว")
# print(df_sorted[['DESCRIPTION', 'AVG', 'Report>=2', 'Report_IQR']].head(10)) # โชว์เทียบกัน
# df_sorted.to_clipboard()

In [25]:
#  df_sorted['Report_Z>=2']

In [26]:
# 1. สร้าง Mask ตามเงื่อนไข
mask_shop = df_sorted['ประเภทร้านค้า'] == 'ร้านอื่นๆ'
mask_z = df_sorted['Report_Z>=2'].astype(str).str.contains('ผิดปกติ', na=False)
mask_iqr = df_sorted['Report_IQR'].astype(str).str.contains('ผิดปกติ', na=False)

final_mask = mask_shop & (mask_z | mask_iqr)
# -----------------------------------------------------------
# 3. สร้างคอลัมน์ 'ผลการตรวจสอบ' (รวม 2 คอลัมน์)
# -----------------------------------------------------------
# ใส่ค่า 'ให้ตรวจสอบราคา' (ตอนนี้คอลัมน์มีอยู่จริงแน่นอนแล้ว จะไม่ error)
for col in ['ผลการตรวจสอบร้านอื่นๆ', 'ผลการตรวจสอบModernTrade', 'ผลการตรวจสอบ']:
    if col not in df_sorted.columns:
        df_sorted[col] = None
df_sorted.loc[final_mask, 'ผลการตรวจสอบร้านอื่นๆ'] = 'ให้ตรวจสอบราคา'
# ใช้ fillna('') เพื่อกัน Error เวลาบวกค่าว่าง
df_sorted['ผลการตรวจสอบ'] = (
    df_sorted['ผลการตรวจสอบModernTrade'].fillna('').astype(str) + 
    df_sorted['ผลการตรวจสอบร้านอื่นๆ'].fillna('').astype(str)
)

df_sorted['count_price_series'] = df_sorted.groupby(['CODE_7_DIGITS', 'DESCRIPTION','AVG'])['AVG'].transform('count')

df_sorted['ผล_TimeSerie'] = None
df_sorted['MAX_DIFF_VALUE'] = np.maximum(df_sorted['AVG_maxdiff'].abs(), df_sorted['AVG_mindiff'].abs())
series_mask = df_sorted['MAX_DIFF_VALUE'] < abs(df_sorted['AVG']-df_sorted['LAST_PRICE'])
df_sorted.loc[series_mask,'ผล_TimeSerie'] = 'ให้ตรวจสอบการเปลี่ยนแปลงราคาในอดีต'

df_sorted.loc[(df_sorted['count_price_series'] == 1)&
              (df_sorted['ผล_TimeSerie'] == 'ให้ตรวจสอบการเปลี่ยนแปลงราคาในอดีต')&
              (df_sorted['REL'] != 100) &
              (df_sorted['AVG'].notna())
              ,'ผลการตรวจสอบ'] = 'ให้ตรวจสอบการเปลี่ยนแปลงราคาในอดีต'

# df_sorted['ผล_TimeSerie_Outlier'] = None
df_sorted.drop(columns=['count_price_series', 'ผล_TimeSerie', 'MAX_DIFF_VALUE'], inplace=True, errors='ignore')

# จัดการค่าว่าง: ถ้าผลลัพธ์เป็นข้อความว่างเปล่า ให้แก้กลับเป็น None 
# (เพื่อให้ใน Excel เป็นช่องว่างสวยๆ ไม่ใช่ "")
df_sorted.loc[df_sorted['ผลการตรวจสอบ'] == '', 'ผลการตรวจสอบ'] = None
# df_sorted.loc[(df_sorted['แสดงผล'] == 'ให้แก้ไข')  & (df_sorted['สถานะ'] == 'น่าสงสัย'), 'ผลการตรวจสอบ'] = 'ผิดปกติ'

# Excel_Condition2(df_sorted,'Test.xlsx') #น้อง

# df_sorted


#### ExWeb

In [27]:
# 1. ระบุคอลัมน์ที่ต้องการ (ตัดคอลัมน์ 'ราคากลาง' ที่ซ้ำออก)
columns_to_keep = [
    'G_FLAG', 'L_FLAG', 'N_FLAG', 'CODE7', 'CODE9', 'COMMODITY_CODE', 
    'DESCRIPTION', 'DILLER_CODE', 'SHOP_NAME', 'ภาค', 'GROUP', 
    'ประเภทร้านค้า', 'LAST_PRICE', 'AVG', 'LINK', 'COMPARE_PRICE', 
    'REL', 'COMM1', 'COMM2', 'Online','ราคากลาง', 'นับตัวซ้ำ', 'CountAVG', 
    'AVGtail_stable', 'ผลการตรวจสอบ', 'ผลต่างราคากับราคากลาง'
]
# 2. สร้าง DataFrame ใหม่โดยการคัดเลือกคอลัมน์
df_exweb = df_sorted[columns_to_keep].copy()

In [28]:
cal_list = pd.read_excel('รายการคำนวณและไม่คำนวณทุกชุด.xlsx')

# 2. หาค่า Max แนวนอนจากคอลัมน์ที่กำหนด (กทมG จนถึง เหนือN)
# เลือกคอลัมน์ตั้งแต่ 'กทมG' ถึง 'เหนือN' เพื่อหาค่าสูงสุด
area_columns = cal_list.columns[cal_list.columns.get_loc('กทมG'):cal_list.columns.get_loc('เหนือN')+1]
cal_list['max_val'] = abs(cal_list[area_columns].max(axis=1))
cal_list['CODE7'] = cal_list['CODE7'].astype(str).str.zfill(7) 

In [29]:
import pandas as pd
import re
import numpy as np

# ==========================================
# 1. การตั้งค่าและ Keywords
# ==========================================

# รายชื่อคอลัมน์
columns_to_keep = [
    'G_FLAG', 'L_FLAG', 'N_FLAG', 'R_FLAG', 'NR_FLAG', 'CODE7', 'CODE9', 'COMMODITY_CODE', 
    'DESCRIPTION', 'DILLER_CODE', 'SHOP_NAME', 'ภาค', 'GROUP', 
    'ประเภทร้านค้า', 'LAST_PRICE', 'AVG', 'LINK', 'COMPARE_PRICE', 
    'REL', 'COMM1', 'COMM2', 'Online','ราคากลาง', 'นับตัวซ้ำ', 
    'AVGtail_stable', 'ผลการตรวจสอบ', 'ผลต่างราคากับราคากลาง'
]

# Keywords
KEYWORDS_SPEC      = ["ขอแก้ไข","เปลี่ยนเป็น","สเปค","ขนาด","บรรจุ","ปรับปริมาณ","ตรา","รุ่น","ยี่ห้อ","ขวด","แพ็ค","เล่ม","www","com","ดื่ม","กาแฟ","แข็ง","เกลือ","วิตามิน"]
KEYWORDS_SHOP      = ["เปลี่ยนร้าน","เปลี่ยนเเหล่ง","เปลี่ยนแหล่ง","เปลี่ยนเป็นร้าน","ร้าน","ห้าง","เเหล่ง","แหล่ง","โลตัส","บิ๊กซี","lotus","ซีเจ","CJ","BigC","Tesco","Makro","แม็คโคร","แมคโคร"]

# [NEW] Keywords สำหรับเคสเก็บผิดสเปค
KEYWORDS_ERROR     = ["เก็บผิด", "ผิดสเปค", "ผิดรุ่น", "ราคาผิด", "ผิดสังเกต", "ผิดขนาด", "เก็บราคาผิด", "ผิดยี่ห้อ", "ผิดสเปก"]

KEYWORDS_NOSELL    = ["ไม่มีขาย", "ไม่มีสินค้า", "สินค้าขาดหาย", "ขาด", "หมด", "ไม่พบ", "ไม่มีจำหน่าย"]
KEYWORDS_SELL      = ["สินค้ามีจำหน่าย","สินค้าพบ","สินค้ามีขาย","สินค้ายังมีขาย"]
KEYWORDS_PROMOTION = ["หมดช่วง", "แถม", "โปร", "หมดโปร", "หมดโปรโมชั่น","ขอตรวจสอบหาร้าน","ราคาขึ้น","ราคาลง","ยืนยัน"]
KEYWORDS_DELETE    = ["ขอลบ", "ลบรายการ", "ลบสินค้า", "ลบออก", "ยกเลิกสินค้า","ขอยกเลิก","เอาออก","ยกเลิกรายการ","ยกเลิกสินค้า","ยกเลิก"]
KEYWORDS_INCREASE  = ["เพิ่มสินค้า", "เพิ่มรายการ","รายการเพิ่ม"]
REMOVE_WORDS       = ["พิจารณาเปลี่ยนสเปคหรือแหล่ง","เปลี่ยนสเปคให้แล้ว","ตราไม่มีตรา"]

# ==========================================
# 2. เตรียมข้อมูล
# ==========================================

df_exweb = df_sorted[columns_to_keep].copy()
df_exweb['Action_Log'] = None 

# โหลดข้อมูล cal_list
try:
    cal_list = pd.read_excel('รายการคำนวณและไม่คำนวณทุกชุด.xlsx')
    area_columns = cal_list.columns[cal_list.columns.get_loc('กทมG'):cal_list.columns.get_loc('เหนือN')+1]
    cal_list['max_val'] = abs(cal_list[area_columns].max(axis=1))
    cal_list['CODE7'] = cal_list['CODE7'].astype(str).str.zfill(7)
except Exception as e:
    print(f"Warning: ไม่สามารถอ่านไฟล์ 'รายการคำนวณและไม่คำนวณทุกชุด.xlsx' ({e}) - ข้ามขั้นตอน Flag")
    cal_list = pd.DataFrame()

# จัดการ Type ข้อมูล
cols_num = ['LAST_PRICE', 'AVG', 'COMPARE_PRICE', 'REL', 'LINK', 'ราคากลาง', 'ผลต่างราคากับราคากลาง']
for col in cols_num:
    df_exweb[col] = pd.to_numeric(df_exweb[col], errors='coerce').fillna(0)

df_exweb['COMM1'] = df_exweb['COMM1'].fillna('').astype(str)
df_exweb['COMM2'] = df_exweb['COMM2'].fillna('').astype(str)

# display(df_exweb[df_exweb['CODE7']=='1160004'])
# ==========================================
# 3. ฟังก์ชัน Logic หลัก
# ==========================================

def process_logic(row):
    # --- A. เตรียมข้อมูล ---
    comm1 = row['COMM1'].strip()
    comm2 = row['COMM2'].strip()
    
    # ล้างคำที่ไม่ต้องการ
    comm1_clean = comm1
    for word in REMOVE_WORDS:
        comm1_clean = comm1_clean.replace(word, "")
    
    # ดึงค่าตัวเลข
    last_price    = row['LAST_PRICE']
    avg_price     = row['AVG']
    compare_price = row['COMPARE_PRICE']
    link_val      = 0
    action_type   = None
    
    # ตัวแปรช่วยเช็คสถานะ
    has_last_price = (last_price != 0)
    has_avg_price  = (avg_price != 0)
    
    # --- B. ตรวจสอบ Keywords และเงื่อนไขพื้นฐาน ---
    
    # 1. c_value logic (ป้องกันไม่ให้ c-value ทับเก็บผิดสเปค)
    # จะทำก็ต่อเมื่อข้อความสั้นๆ (ป้องกันการจับผิดในประโยคยาว)
    new_c_value = ""
    if len(comm1) <= 50:
        match = re.search(r'c(\d+)', comm1_clean, re.IGNORECASE)
        if match:
            new_c_value = f"c{int(match.group(1)) + 1}" 
            comm2 = new_c_value

    if not comm2 and any(kw in comm1_clean for kw in KEYWORDS_NOSELL): comm2 = "c1"
    if any(kw in comm1_clean.lower() for kw in [f"c{i}" for i in range(3, 10)]): comm2 = "c4 พิจารณาเปลี่ยนสเปคหรือแหล่ง"
    if any(kw in comm1_clean for kw in KEYWORDS_SELL): comm2 = ""

    # 2. Link Logic
    found_spec = any(kw in comm1_clean for kw in KEYWORDS_SPEC)
    found_shop = any(kw in comm1_clean for kw in KEYWORDS_SHOP)
    
    if any(ex in comm1_clean for ex in ["ต้นทุน", "ผลิต", "ราคาตลาด", "ภาวะตลาด", "ประหยัด"]): found_spec = False
    if any(ex in comm1_clean for ex in ["ปรับราคา","ขอตรวจสอบหาร้านก่อน"]): found_shop = False
    if re.search(r'\d{8,}', comm1_clean): found_spec = True      

    if found_spec and found_shop: link_val = 3
    elif found_shop: link_val = 2
    elif found_spec: link_val = 1
    
    # Cleanup Link/Comm2
    # [FIX] เพิ่ม "เก็บผิดสเปค" ลงใน protected เพื่อป้องกันการ reset ผิดพลาด
    protected_values = ["เตรียมลบรายการ", "c1", "c4 พิจารณาเปลี่ยนสเปคหรือแหล่ง", "เก็บผิดสเปค", new_c_value]
    
    if link_val in [1, 2, 3]:
        # [FIX] ถ้าเป็น Link 1-3 อย่าลบ Comm2 ถ้ามันคือ "เก็บผิดสเปค"
        if not (comm2.startswith("เปลี่ยนให้เดือนหน้า") or comm2 == "เก็บผิดสเปค"): 
            comm2 = "" 
    elif comm2 in protected_values and comm2 != "":
        link_val = 0
        
    # Promotion Check: เช็คโปรโมชั่นเฉพาะเมื่อ Link ยังเป็น 0
    if link_val == 0:
        if any(kw in comm1_clean for kw in KEYWORDS_PROMOTION) and len(comm1_clean) <= 25:
            # อย่าลบถ้าเป็นเคสเก็บผิดสเปค
            if comm2 != "เก็บผิดสเปค":
                comm2 = ""
            link_val = 0
        
    if last_price == 0 and avg_price == 0 and compare_price == 0 and row['REL'] == 0:
        link_val = 0

    # --- C. ระบุ Action Log (รวมศูนย์ที่นี่) ---
    
    is_delete_kw = any(kw in comm1_clean for kw in KEYWORDS_DELETE)
    is_error_kw  = any(kw in comm1_clean for kw in KEYWORDS_ERROR) # Check เก็บผิด
    
    # Priority 1: ลบสินค้า
    if is_delete_kw:
        action_type = "เตรียมลบสินค้า"
        comm2 = "เตรียมลบรายการ"
        link_val = 0 
        
    # Priority 2: [NEW] เก็บผิดสเปค (ระบุใน Action Log ด้วย)
    elif is_error_kw:
        action_type = "เก็บผิดสเปค"  # <--- ระบุใน Action Log ตรงนี้
        link_val = 3                # <--- Link 3
        comm2 = "เก็บผิดสเปค"       # <--- ใส่ใน Comm2
        
    # Priority 3: เพิ่มสินค้า
    elif any(kw in comm1_clean for kw in KEYWORDS_INCREASE) or \
         (not has_last_price and (has_avg_price or compare_price != 0)):
        action_type = "เตรียมเพิ่มสินค้า"
        link_val = 3 
        if has_avg_price: 
            compare_price = avg_price
            
    # Priority 4: เปลี่ยนสเปค/ร้าน
    elif link_val == 3: action_type = "เปลี่ยนทั้งร้านสเปค"
    elif link_val == 2: action_type = "เปลี่ยนร้าน"
    elif link_val == 1: action_type = "เปลี่ยนสเปค"
    
    # Priority 5: เติมค่าต่างๆ (Filling Logic)
    else:
        if (row['ราคากลาง'] > 0) and (row['ผลต่างราคากับราคากลาง'] != 0) and (comm1 == ""):
            avg_price = row['ราคากลาง']
            action_type = "เติมราคากลาง"
        
    # เติม Compare/AVG (ถ้ายังไม่มี Action หรือเป็นเคสเติม)
    if (compare_price == 0) and has_last_price:
        compare_price = last_price
        action_type = "เติมCompare ด้วยLastPrice"
    elif (compare_price == 0) and has_avg_price:
        compare_price = avg_price
        action_type = "เติมCompare ด้วยAVG"
    elif (avg_price == 0) and (compare_price != 0):
        avg_price = compare_price
        action_type = "เติมAVG ด้วยCompare"

    return pd.Series([avg_price, link_val, compare_price, comm1, comm2, action_type])

# ==========================================
# 4. Apply Function
# ==========================================

df_exweb[['AVG', 'LINK', 'COMPARE_PRICE', 'COMM1', 'COMM2', 'Action_Log']] = df_exweb.apply(process_logic, axis=1)

# ==========================================
# 5. จัดการ Flag
# ==========================================
if not cal_list.empty:
    for i in cal_list['CODE7']:
        row_cal = cal_list[cal_list['CODE7'] == i].iloc[0]
        m_val = row_cal['max_val']
        if m_val in [0, 1]:
            for region, suffix in [('กทม.', 'กทม'), ('ปริมณฑล', 'ปริมณฑล'), ('กลาง', 'กลาง'), 
                                   ('ตะวันออกเฉียงเหนือ', 'ตะวันออกเฉียงเหนือ'), ('ใต้', 'ใต้'), ('เหนือ', 'เหนือ')]:
                mask = (df_exweb['CODE7'] == i) & (df_exweb['ภาค'] == region)
                if mask.any():
                    df_exweb.loc[mask, ['G_FLAG', 'L_FLAG', 'N_FLAG']] = \
                        [row_cal[f'{suffix}G'], row_cal[f'{suffix}L'], row_cal[f'{suffix}N']]

mask_special = (df_exweb['CODE7'] == '1160010')
df_exweb.loc[mask_special & (df_exweb['AVG'] > 30), ['G_FLAG', 'L_FLAG', 'N_FLAG']] = [1, 0, 0]
df_exweb.loc[mask_special & (df_exweb['AVG'] <= 30), ['G_FLAG', 'L_FLAG', 'N_FLAG']] = [1, 1, 0]

# df_exweb.loc[df_exweb['GROUP'].isin(['แม่ฮ่องสอน','พังงา','สกลนคร','นครนายก']), ['G_FLAG','L_FLAG','N_FLAG']] = [0,0,0]

# ==========================================
# 6. Final Formatting & Export
# ==========================================

df_exweb['REL2'] = (df_exweb['AVG'] / df_exweb['COMPARE_PRICE']) * 100
df_exweb['REL2'] = df_exweb['REL2'].round(2)

cols_to_blank = ['LAST_PRICE', 'AVG', 'COMPARE_PRICE', 'REL','REL2']
for col in cols_to_blank:
    df_exweb[col] = df_exweb[col].astype(object)
    df_exweb.loc[df_exweb[col] == 0, col] = None

df_exweb['LINK'] = df_exweb['LINK'].replace("N", 0)
df_exweb['ราคากลาง'] = df_exweb['ราคากลาง'].replace(0, "")
df_exweb['ผลต่างราคากับราคากลาง'] = df_exweb['ผลต่างราคากับราคากลาง'].replace(0, "")

# Logic เพิ่มเติมท้ายสุดสำหรับ Link 3 (ให้ Compare = AVG ด้วย เพื่อความชัวร์)
df_exweb.loc[df_exweb['LINK'].isin([1, 2, 3]), 'COMPARE_PRICE'] = df_exweb['AVG']

mask_no_comm = (
    (df_exweb['COMM1'].isna() | (df_exweb['COMM1'] == '')) & 
    (df_exweb['COMM2'].isna() | (df_exweb['COMM2'] == ''))
)

mask_check_price = (df_exweb['ผลการตรวจสอบ'] == 'ให้ตรวจสอบราคา')

# กรณีราคานิ่งเกิน 1 ปี
df_exweb.loc[
    mask_no_comm & mask_check_price & (df_exweb['AVGtail_stable'] > 12),
    ['เเจ้งเตือนการไม่เปลี่ยนเเปลง', 'COMM2']
] = 'ราคานิ่งเกินหนึ่งปีโปรดติดตามอย่างใกล้ชิด'

# กรณีราคานิ่งเกิน 2 ปี
df_exweb.loc[
    mask_no_comm & mask_check_price & (df_exweb['AVGtail_stable'] > 24),
    ['เเจ้งเตือนการไม่เปลี่ยนเเปลง', 'COMM2']
] = 'ราคานิ่งเกินสองปีโปรดติดตามอย่างใกล้ชิด'


df_exweb.loc[
    (df_exweb['CODE7'].isin(['1160004', '1160010', '1160015'])) &
    (df_exweb['COMM2'].astype(str).str.startswith('ราคานิ่งเกิน')),
    'COMM2'
] = None



val_to_insert = df_exweb['COMMODITY_CODE'].astype(str) + df_exweb['DILLER_CODE'].astype(str) # 2. แปลงเป็น String ก่อนเชื่อม (เพื่อกันมันบวกกันเป็นตัวเลข)
if 'ComKeyBasic' in df_exweb.columns: df_exweb.drop(columns=['ComKeyBasic'], inplace=True)
df_exweb['ComKeyBasic'] =  val_to_insert # 3. Insert ลงไปที่ตำแหน่งนั้น


if filename[3] == 'g':    
    columns_to_keep = [
        'G_FLAG', 'L_FLAG', 'N_FLAG', 'CODE7', 'CODE9','ภาค', 'GROUP', 
        'COMMODITY_CODE', 'DESCRIPTION', 'DILLER_CODE', 'SHOP_NAME', 
        'ประเภทร้านค้า', 'LAST_PRICE', 'AVG', 'LINK', 'COMPARE_PRICE', 
        'REL', 'COMM1', 'COMM2', 'Online','ราคากลาง', 'ผลต่างราคากับราคากลาง', 'นับตัวซ้ำ', 
        'AVGtail_stable', 'ผลการตรวจสอบ',
        'ComKeyBasic','Action_Log'  # <--- ใส่เพิ่มตรงนี้ด้วยครับ
    ]
    df_exweb = df_exweb[columns_to_keep]


if filename[3] == 'u':    
    columns_to_keep = [
        'R_FLAG', 'NR_FLAG', 'CODE7', 'CODE9','ภาค', 'GROUP', 
        'COMMODITY_CODE', 'DESCRIPTION', 'DILLER_CODE', 'SHOP_NAME', 
        'ประเภทร้านค้า', 'LAST_PRICE', 'AVG', 'LINK', 'COMPARE_PRICE', 
        'REL', 'COMM1', 'COMM2', 'Online','ราคากลาง', 'ผลต่างราคากับราคากลาง', 'นับตัวซ้ำ', 
        'AVGtail_stable', 'ผลการตรวจสอบ',
        'ComKeyBasic','Action_Log'  # <--- ใส่เพิ่มตรงนี้ด้วยครับ
    ]
    df_exweb = df_exweb[columns_to_keep]

# ==========================================
# 6. Final Formatting & Export (ส่วนแสดงผลสวยงาม)
# ==========================================
from xlsxwriter.utility import xl_col_to_name # ต้อง import อันนี้เพิ่มที่หัวไฟล์

# ... (Logic ส่วนเตรียม DataFrame ด้านบนเหมือนเดิม) ...

# ==========================================
# 6. Final Formatting & Export (ส่วนแสดงผลสวยงาม + เส้นขอบ)
# ==========================================

output_path = f"{filename}_ExWeb_Final_Clean.xlsx"

try:
    with pd.ExcelWriter(output_path, engine='xlsxwriter') as writer:
        sheet_name = 'Result'
        df_exweb.to_excel(writer, sheet_name=sheet_name, index=False)
        
        workbook  = writer.book
        worksheet = writer.sheets[sheet_name]
        
        # --- 1. กำหนด Formats พื้นฐาน ---
        header_fmt = workbook.add_format({
            'bold': True, 'text_wrap': False, 'valign': 'top', 
            'fg_color': '#D7E4BC', 'border': 1, 'align': 'left'
        })
        num_fmt = workbook.add_format({'num_format': '#,##0.00'})
        text_wrap_fmt = workbook.add_format({'text_wrap': False, 'valign': 'top'})
        
        # --- 2. กำหนด Formats สำหรับ Highlight และ Border ---
        highlight_link = workbook.add_format({'bg_color': '#FFEB9C', 'font_color': '#9C0006'}) 
        highlight_error = workbook.add_format({'bg_color': '#FFC7CE', 'font_color': '#9C0006'}) 

        # Border Formats (สังเกต: เรา set แค่ top เพื่อขีดเส้นคั่นบรรทัด)
        border_thick  = workbook.add_format({'top': 2}) # 2 = เส้นหนา
        border_thin   = workbook.add_format({'top': 1}) # 1 = เส้นบาง
        border_dashed = workbook.add_format({'top': 9}) # 9 = เส้นประ

        # --- 3. จัดรูปแบบ Headers และ Freeze Panes ---
        for col_num, value in enumerate(df_exweb.columns.values):
            worksheet.write(0, col_num, value, header_fmt)
        
        # Freeze Panes
        freeze_col = 4 
        if 'CODE7' in df_exweb.columns:
            freeze_col = df_exweb.columns.get_loc('CODE7') + 1
        worksheet.freeze_panes(1, freeze_col) 
        
        # --- 4. จัดความกว้างคอลัมน์ (Auto Width) ---
        for i, col in enumerate(df_exweb.columns):
            col_len = max(df_exweb[col].astype(str).map(len).max(), len(col)) + 2
            col_len = min(col_len, 50) 
            
            if col in ['COMM1', 'COMM2', 'DESCRIPTION', 'SHOP_NAME', 'Action_Log']:
                worksheet.set_column(i, i, 30, text_wrap_fmt)
            elif col in ['LAST_PRICE', 'AVG', 'COMPARE_PRICE', 'REL', 'ราคากลาง', 'REL2']:
                worksheet.set_column(i, i, 12, num_fmt)
            else:
                worksheet.set_column(i, i, col_len)

        # --- 5. Conditional Formatting (Highlight สี + เส้นขอบ Grouping) ---
        (max_row, max_col) = df_exweb.shape
        
        # ฟังก์ชันช่วยหาชื่อคอลัมน์ Excel (A, B, C...)
        def get_col_letter(name):
            if name in df_exweb.columns:
                return xl_col_to_name(df_exweb.columns.get_loc(name))
            return None

        # หาตำแหน่งคอลัมน์ที่ใช้คำนวณ
        let_link   = get_col_letter('LINK')
        let_comm2  = get_col_letter('COMM2')
        let_action = get_col_letter('Action_Log')
        
        # คอลัมน์สำหรับ Logic เส้นขอบ
        let_code7 = get_col_letter('CODE7')
        let_desc  = get_col_letter('DESCRIPTION')
        let_shop  = get_col_letter('ประเภทร้านค้า')

        # 5.1 Highlight LINK (สีส้ม)
        if let_link:
            # ใช้ range ตั้งแต่ row 1 (คือ excel row 2) ถึง row สุดท้าย
            worksheet.conditional_format(1, df_exweb.columns.get_loc('LINK'), max_row, df_exweb.columns.get_loc('LINK'), {
                'type': 'cell', 'criteria': '!=', 'value': 0, 'format': highlight_link
            })

        # 5.2 Highlight Error/Delete (สีแดง)
        for col_name in ['COMM2', 'Action_Log']:
            idx = df_exweb.columns.get_loc(col_name) if col_name in df_exweb.columns else None
            if idx is not None:
                for kw in ['เก็บผิด', 'ลบ']:
                    worksheet.conditional_format(1, idx, max_row, idx, {
                        'type': 'text', 'criteria': 'containing', 'value': kw, 'format': highlight_error
                    })

        # 5.3 Grouping Borders Logic (สำคัญ!)
        # หลักการ: ใช้ Formula เช็คค่าบรรทัดปัจจุบัน เทียบกับ บรรทัดก่อนหน้า (Row-1)
        # ต้องเรียงลำดับความสำคัญจาก "ย่อยสุด" ไป "ใหญ่สุด" (Dashed -> Thin -> Thick)
        # หรือใช้สูตร AND เพื่อระบุเงื่อนไขให้ชัดเจน

        # เริ่มต้นที่ Row 2 (Excel Row 3) เพื่อเทียบกับ Row 2 (Excel Row 2)
        # แต่เพื่อให้ครอบคลุมบรรทัดแรกสุดด้วย เราเริ่มที่ Row 1 (Excel Row 2) เทียบกับ Header (ซึ่งค่าต่างกันแน่นอน -> ได้เส้นขอบปิดหัวตารางพอดี)
        
        # เงื่อนไข 1: เส้นประ (Dashed) -> เมื่อ Code7 เท่าเดิม, Desc เท่าเดิม, แต่ ShopType เปลี่ยน
        if let_code7 and let_desc and let_shop:
            formula_dashed = f'=AND(${let_code7}2=${let_code7}1, ${let_desc}2=${let_desc}1, ${let_shop}2<>${let_shop}1)'
            worksheet.conditional_format(1, 9, max_row, max_col - 1, {
                'type': 'formula',
                'criteria': formula_dashed,
                'format': border_dashed
            })

        # เงื่อนไข 2: เส้นบาง (Thin) -> เมื่อ Code7 เท่าเดิม, แต่ Desc เปลี่ยน
        if let_code7 and let_desc:
            formula_thin = f'=AND(${let_code7}2=${let_code7}1, ${let_desc}2<>${let_desc}1)'
            worksheet.conditional_format(1, 0, max_row, max_col - 1, {
                'type': 'formula',
                'criteria': formula_thin,
                'format': border_thin
            })

        # เงื่อนไข 3: เส้นหนา (Thick) -> เมื่อ Code7 เปลี่ยน (ใหญ่สุด ทับทุกเงื่อนไข)
        if let_code7:
            formula_thick = f'=${let_code7}2<>${let_code7}1'
            worksheet.conditional_format(1, 0, max_row, max_col - 1, {
                'type': 'formula',
                'criteria': formula_thick,
                'format': border_thick
            })
        def _ext(name, width):
                    if name in df_exweb.columns:
                        # หา Index ของคอลัมน์จากชื่อ (0, 1, 2...)
                        idx = df_exweb.columns.get_loc(name)
                        # สั่ง set_column(start_col, end_col, width)
                        worksheet.set_column(idx, idx, width)

        # เรียกใช้งานตามต้องการ
        _ext('R_FLAG', 3)
        _ext('NR_FLAG', 3)
        _ext('ภาค', 5)
        _ext('GROUP', 9)
        _ext('SHOP_NAME', 7)
        _ext('ประเภทร้านค้า', 5)
        _ext('CODE7', 7)
        _ext('CODE9', 9)
        _ext('COMMODITY_CODE', 16)	
        _ext('DILLER_CODE', 10)	
        _ext('COMM1', 10)	
        _ext('COMM2', 10)	
        _ext('LAST_PRICE', 6)	
        _ext('AVG', 6)	
        _ext('COMPARE_PRICE', 6)	
        _ext('REL', 6)	
        _ext('ราคากลาง', 6)	
        _ext('ผลต่างราคากับราคากลาง', 6)	
        _ext('นับตัวซ้ำ', 6)	
        _ext('Online', 6)
        _ext('AVGtail_stable', 6)
        
        		


				

            
        # print(f"✅ บันทึกไฟล์พร้อมจัดรูปแบบและเส้นขอบเรียบร้อยที่: {output_path}")

except Exception as e:
    print(f"⚠️ Error ในการจัดรูปแบบ ({e}) กำลังบันทึกแบบปกติ...")
    df_exweb.to_excel(output_path, index=False)
    print(f"✅ บันทึกไฟล์ (แบบปกติ) เรียบร้อยที่: {output_path}")

In [30]:
# df_sorted[columns_to_keep]

In [31]:

normal = df_sorted.copy()
normal['Action_Log'] = "None"
normal = normal[columns_to_keep]
normal

output_path = f"{filename}_MainBefore_exweb.xlsx"

try:
    with pd.ExcelWriter(output_path, engine='xlsxwriter') as writer:
        sheet_name = 'Result'
        normal.to_excel(writer, sheet_name=sheet_name, index=False)
        
        workbook  = writer.book
        worksheet = writer.sheets[sheet_name]
        
        # --- 1. กำหนด Formats พื้นฐาน ---
        header_fmt = workbook.add_format({
            'bold': True, 'text_wrap': False, 'valign': 'top', 
            'fg_color': '#D7E4BC', 'border': 1, 'align': 'left'
        })
        num_fmt = workbook.add_format({'num_format': '#,##0.00'})
        text_wrap_fmt = workbook.add_format({'text_wrap': False, 'valign': 'top'})
        
        # --- 2. กำหนด Formats สำหรับ Highlight และ Border ---
        highlight_link = workbook.add_format({'bg_color': '#FFEB9C', 'font_color': '#9C0006'}) 
        highlight_error = workbook.add_format({'bg_color': '#FFC7CE', 'font_color': '#9C0006'}) 

        # Border Formats (สังเกต: เรา set แค่ top เพื่อขีดเส้นคั่นบรรทัด)
        border_thick  = workbook.add_format({'top': 2}) # 2 = เส้นหนา
        border_thin   = workbook.add_format({'top': 1}) # 1 = เส้นบาง
        border_dashed = workbook.add_format({'top': 9}) # 9 = เส้นประ

        # --- 3. จัดรูปแบบ Headers และ Freeze Panes ---
        for col_num, value in enumerate(normal.columns.values):
            worksheet.write(0, col_num, value, header_fmt)
        
        # Freeze Panes
        freeze_col = 4 
        if 'CODE7' in normal.columns:
            freeze_col = normal.columns.get_loc('CODE7') + 1
        worksheet.freeze_panes(1, freeze_col) 
        
        # --- 4. จัดความกว้างคอลัมน์ (Auto Width) ---
        for i, col in enumerate(normal.columns):
            col_len = max(normal[col].astype(str).map(len).max(), len(col)) + 2
            col_len = min(col_len, 50) 
            
            if col in ['COMM1', 'COMM2', 'DESCRIPTION', 'SHOP_NAME', 'Action_Log']:
                worksheet.set_column(i, i, 30, text_wrap_fmt)
            elif col in ['LAST_PRICE', 'AVG', 'COMPARE_PRICE', 'REL', 'ราคากลาง', 'REL2']:
                worksheet.set_column(i, i, 12, num_fmt)
            else:
                worksheet.set_column(i, i, col_len)

        # --- 5. Conditional Formatting (Highlight สี + เส้นขอบ Grouping) ---
        (max_row, max_col) = normal.shape
        
        # ฟังก์ชันช่วยหาชื่อคอลัมน์ Excel (A, B, C...)
        def get_col_letter(name):
            if name in normal.columns:
                return xl_col_to_name(normal.columns.get_loc(name))
            return None

        # หาตำแหน่งคอลัมน์ที่ใช้คำนวณ
        let_link   = get_col_letter('LINK')
        let_comm2  = get_col_letter('COMM2')
        let_action = get_col_letter('Action_Log')
        
        # คอลัมน์สำหรับ Logic เส้นขอบ
        let_code7 = get_col_letter('CODE7')
        let_desc  = get_col_letter('DESCRIPTION')
        let_shop  = get_col_letter('ประเภทร้านค้า')

        # 5.1 Highlight LINK (สีส้ม)
        if let_link:
            # ใช้ range ตั้งแต่ row 1 (คือ excel row 2) ถึง row สุดท้าย
            worksheet.conditional_format(1, normal.columns.get_loc('LINK'), max_row, normal.columns.get_loc('LINK'), {
                'type': 'cell', 'criteria': '!=', 'value': 0, 'format': highlight_link
            })

        # 5.2 Highlight Error/Delete (สีแดง)
        for col_name in ['COMM2', 'Action_Log']:
            idx = normal.columns.get_loc(col_name) if col_name in normal.columns else None
            if idx is not None:
                for kw in ['เก็บผิด', 'ลบ']:
                    worksheet.conditional_format(1, idx, max_row, idx, {
                        'type': 'text', 'criteria': 'containing', 'value': kw, 'format': highlight_error
                    })

        # 5.3 Grouping Borders Logic (สำคัญ!)
        # หลักการ: ใช้ Formula เช็คค่าบรรทัดปัจจุบัน เทียบกับ บรรทัดก่อนหน้า (Row-1)
        # ต้องเรียงลำดับความสำคัญจาก "ย่อยสุด" ไป "ใหญ่สุด" (Dashed -> Thin -> Thick)
        # หรือใช้สูตร AND เพื่อระบุเงื่อนไขให้ชัดเจน

        # เริ่มต้นที่ Row 2 (Excel Row 3) เพื่อเทียบกับ Row 2 (Excel Row 2)
        # แต่เพื่อให้ครอบคลุมบรรทัดแรกสุดด้วย เราเริ่มที่ Row 1 (Excel Row 2) เทียบกับ Header (ซึ่งค่าต่างกันแน่นอน -> ได้เส้นขอบปิดหัวตารางพอดี)
        
        # เงื่อนไข 1: เส้นประ (Dashed) -> เมื่อ Code7 เท่าเดิม, Desc เท่าเดิม, แต่ ShopType เปลี่ยน
        if let_code7 and let_desc and let_shop:
            formula_dashed = f'=AND(${let_code7}2=${let_code7}1, ${let_desc}2=${let_desc}1, ${let_shop}2<>${let_shop}1)'
            worksheet.conditional_format(1, 9, max_row, max_col - 1, {
                'type': 'formula',
                'criteria': formula_dashed,
                'format': border_dashed
            })

        # เงื่อนไข 2: เส้นบาง (Thin) -> เมื่อ Code7 เท่าเดิม, แต่ Desc เปลี่ยน
        if let_code7 and let_desc:
            formula_thin = f'=AND(${let_code7}2=${let_code7}1, ${let_desc}2<>${let_desc}1)'
            worksheet.conditional_format(1, 0, max_row, max_col - 1, {
                'type': 'formula',
                'criteria': formula_thin,
                'format': border_thin
            })

        # เงื่อนไข 3: เส้นหนา (Thick) -> เมื่อ Code7 เปลี่ยน (ใหญ่สุด ทับทุกเงื่อนไข)
        if let_code7:
            formula_thick = f'=${let_code7}2<>${let_code7}1'
            worksheet.conditional_format(1, 0, max_row, max_col - 1, {
                'type': 'formula',
                'criteria': formula_thick,
                'format': border_thick
            })
        def _ext(name, width):
                    if name in normal.columns:
                        # หา Index ของคอลัมน์จากชื่อ (0, 1, 2...)
                        idx = normal.columns.get_loc(name)
                        # สั่ง set_column(start_col, end_col, width)
                        worksheet.set_column(idx, idx, width)

        # เรียกใช้งานตามต้องการ
        _ext('R_FLAG', 3)
        _ext('NR_FLAG', 3)
        _ext('ภาค', 5)
        _ext('GROUP', 9)
        _ext('SHOP_NAME', 7)
        _ext('ประเภทร้านค้า', 5)
        _ext('CODE7', 7)
        _ext('CODE9', 9)
        _ext('COMMODITY_CODE', 16)	
        _ext('DILLER_CODE', 10)	
        _ext('COMM1', 10)	
        _ext('COMM2', 10)	
        _ext('LAST_PRICE', 6)	
        _ext('AVG', 6)	
        _ext('COMPARE_PRICE', 6)	
        _ext('REL', 6)	
        _ext('ราคากลาง', 6)	
        _ext('ผลต่างราคากับราคากลาง', 6)	
        _ext('นับตัวซ้ำ', 6)	
        _ext('Online', 6)
        _ext('AVGtail_stable', 6)
        
        		


				

            
        # print(f"✅ บันทึกไฟล์พร้อมจัดรูปแบบและเส้นขอบเรียบร้อยที่: {output_path}")

except Exception as e:
    print(f"⚠️ Error ในการจัดรูปแบบ ({e}) กำลังบันทึกแบบปกติ...")
    normal.to_excel(output_path, index=False)
    print(f"✅ บันทึกไฟล์ (แบบปกติ) เรียบร้อยที่: {output_path}")

In [32]:
# def apply_excel_formatting(wb, df, sheet_name="Result_Formatted"):
#     """
#     ฟังก์ชันสำหรับเขียน DataFrame ลง Worksheet ใหม่และจัด Format ตามเงื่อนไข
#     """
    
#     # 1. สร้าง Sheet ใหม่
#     if sheet_name in wb.sheetnames:
#         del wb[sheet_name] # ลบของเก่าถ้ามี
#     ws = wb.create_sheet(sheet_name)

#     # 2. เขียนข้อมูล (Write Data)
#     for r in dataframe_to_rows(df, index=False, header=True):
#         ws.append(r)

#     # --- เตรียม Style (Defines Styles) เพื่อลดการสร้าง object ซ้ำๆ ---
#     # สี (Fills)
#     fill_red = PatternFill(start_color="FFCCCC", end_color="FFCCCC", fill_type="solid")    # สีแดงอ่อน
#     fill_pink = PatternFill(start_color="FFD1DC", end_color="FFD1DC", fill_type="solid")   # สีชมพู
#     fill_grey = PatternFill(start_color="D3D3D3", end_color="D3D3D3", fill_type="solid")   # สีเทา
#     fill_yellow = PatternFill(start_color="FFFFCC", end_color="FFFFCC", fill_type="solid") # สีเหลือง
#     fill_green = PatternFill(start_color="CCFFCC", end_color="CCFFCC", fill_type="solid")  # สีเขียว
#     fill_blue_highlight = PatternFill(start_color="E0F7FA", end_color="E0F7FA", fill_type="solid") # สีฟ้าสำหรับร้านใหญ่

#     # เส้นขอบ (Borders)
#     thin_side = Side(style='thin')
#     thick_side = Side(style='thick')
#     dashed_side = Side(style='dashed')

#     # --- ค้นหา Index ของคอลัมน์ที่ต้องใช้ (Column Mapping) ---
#     # หมายเหตุ: openpyxl เริ่มนับที่ 1, pandas เริ่มที่ 0
#     headers = {col: i+1 for i, col in enumerate(df.columns)}
    
#     # ระบุชื่อคอลัมน์สำคัญ (แก้ไขชื่อให้ตรงกับ Data จริงของคุณ)
#     col_code_7 = headers.get('CODE_7_DIGITS')
#     col_desc = headers.get('DESCRIPTION')
#     col_store_type = headers.get('ประเภทร้านค้า')
#     col_status = headers.get('สถานะ')
#     col_rel = headers.get('REL')
#     col_avg = headers.get('AVG')
#     col_check_other = headers.get('ผลการตรวจสอบร้านอื่นๆ')
#     col_check_modern = headers.get('ผลการตรวจสอบModernTrade')
#     col_is_large_store = headers.get('IS_LARGE_STORE') # สมมติว่ามี flag บอกว่าเป็นร้านใหญ่

#     # 3. เริ่มวนลูปจัดรูปแบบรายบรรทัด (Iterate Rows)
#     # เริ่มที่ row 2 เพราะ row 1 คือ Header
#     max_row = ws.max_row
#     max_col = ws.max_column

#     prev_code = None
#     prev_desc = None
#     prev_store_type = None

#     for row_idx in range(2, max_row + 1):
#         # ดึงค่าเพื่อใช้ตรวจสอบเงื่อนไข
#         curr_code = ws.cell(row=row_idx, column=col_code_7).value if col_code_7 else None
#         curr_desc = ws.cell(row=row_idx, column=col_desc).value if col_desc else None
#         curr_store_type = ws.cell(row=row_idx, column=col_store_type).value if col_store_type else None
        
#         # --- A. การจัดกลุ่มด้วยเส้นขอบ (Borders) ---
#         # Logic: ตรวจสอบการเปลี่ยนค่าจากบรรทัดก่อนหน้า
#         top_border_style = None

#         if row_idx > 2: # ข้ามบรรทัดแรกของข้อมูล
#             if curr_code != prev_code:
#                 top_border_style = thick_side # เปลี่ยน Code 7 หลัก -> เส้นหนา
#             elif curr_desc != prev_desc:
#                 top_border_style = thin_side  # เปลี่ยน Description -> เส้นบาง
#             elif curr_store_type != prev_store_type:
#                 top_border_style = dashed_side # เปลี่ยนประเภทร้าน -> เส้นประ

#         # วาดเส้นขอบ (ถ้ามีการเปลี่ยนแปลง)
#         if top_border_style:
#             for col_idx in range(1, max_col + 1):
#                 cell = ws.cell(row=row_idx, column=col_idx)
#                 current_border = cell.border
#                 # สร้าง Border ใหม่โดยคงค่า side อื่นๆ ไว้ เปลี่ยนแค่ top
#                 new_border = Border(
#                     left=current_border.left, 
#                     right=current_border.right, 
#                     top=top_border_style, 
#                     bottom=current_border.bottom
#                 )
#                 cell.border = new_border

#         # Update previous values
#         prev_code = curr_code
#         prev_desc = curr_desc
#         prev_store_type = curr_store_type

#         # --- B. การเน้นสี (Conditional Formatting) ---
        
#         # 1. ร้านขนาดใหญ่ (Highlight สีฟ้า คอลัมน์ 26-30)
#         # ตรวจสอบว่าร้านนี้เป็นร้านใหญ่หรือไม่ (สมมติเช็คจากคอลัมน์ flag หรือ logic ที่คุณมี)
#         is_large = False
#         if col_is_large_store:
#              val = ws.cell(row=row_idx, column=col_is_large_store).value
#              if val == 'Y' or val == True: # ปรับเงื่อนไขตามจริง
#                  is_large = True
        
#         if is_large:
#             for c_idx in range(26, 31): # คอลัมน์ 26 ถึง 30
#                 ws.cell(row=row_idx, column=c_idx).fill = fill_blue_highlight

#         # 2. สถานะ (Red/Pink/Grey)
#         if col_status:
#             status_cell = ws.cell(row=row_idx, column=col_status)
#             status_val = str(status_cell.value).strip() if status_cell.value else ""
            
#             if "ผิดปกติ" in status_val:
#                 status_cell.fill = fill_red
#             elif "น่าสงสัย" in status_val:
#                 status_cell.fill = fill_pink
#             elif "ไม่มีราคา" in status_val:
#                 status_cell.fill = fill_grey

#         # 3. REL (Color Scale)
#         if col_rel:
#             rel_cell = ws.cell(row=row_idx, column=col_rel)
#             try:
#                 rel_val = float(rel_cell.value)
#                 if rel_val < 100:
#                     rel_cell.fill = fill_yellow
#                 elif rel_val > 100:
#                     rel_cell.fill = fill_green
#             except (ValueError, TypeError):
#                 pass # ข้ามถ้าไม่ใช่ตัวเลข

#         # 4. การตรวจสอบราคา (Logic ข้ามคอลัมน์)
#         # เช็คผลการตรวจสอบร้านอื่น และ ModernTrade เพื่อระบายสี AVG และ REL
#         check_passed = True # สมมติว่าปกติ
        
#         # ตัวอย่าง Logic: ถ้ามีข้อความเตือนในคอลัมน์ตรวจสอบ ให้ถือว่าผิดปกติ
#         val_check_other = ws.cell(row=row_idx, column=col_check_other).value if col_check_other else None
#         val_check_mt = ws.cell(row=row_idx, column=col_check_modern).value if col_check_modern else None

#         if val_check_other or val_check_mt: 
#             # ปรับเงื่อนไข if ตามข้อมูลจริง เช่น if val_check_other == 'Diff':
#             # ถ้าเข้าเงื่อนไข ให้ระบายสี AVG และ REL
#             if col_avg: ws.cell(row=row_idx, column=col_avg).fill = fill_yellow # หรือแดงตามต้องการ
#             if col_rel: ws.cell(row=row_idx, column=col_rel).fill = fill_yellow

#     # 4. ซ่อนคอลัมน์ (Hide Columns)
#     # ลิสต์คอลัมน์จำนวนมากที่ต้องการซ่อน (Hardcoded List)
#     COLUMNS_TO_HIDE_LIST = [
#         "INTERNAL_ID", "CREATED_DATE", "UPDATED_BY", "RAW_DATA_SOURCE", 
#         "TEMP_CALC_1", "TEMP_CALC_2", # ... ใส่รายชื่อคอลัมน์จริงที่นี่ยาวๆ
#     ]

#     for col_name, col_idx in headers.items():
#         # ถ้าอยู่ใน List ที่ต้องซ่อน และ ไม่ได้ถูกสั่งให้แสดง (ในที่นี้ซ่อนหมดตาม list)
#         if col_name in COLUMNS_TO_HIDE_LIST:
#             col_letter = get_column_letter(col_idx)
#             ws.column_dimensions[col_letter].hidden = True

#     print(f"Formatting completed for sheet: {sheet_name}")
#     return wb

# # --- วิธีการเรียกใช้งาน ---
# from openpyxl import load_workbook
# wb = load_workbook('.xlsx')
# df = pd.read_excel('data_source.xlsx') # หรือ dataframe ที่ process มาแล้ว
# apply_excel_formatting(wb, df)
# wb.save('final_report.xlsx')

In [33]:
Comment = df_exweb[['ComKeyBasic','Action_Log']] 

In [34]:
# df_exweb



#### สรุป

In [35]:
all = df_sorted.copy()

# if 'ComKeyBasic' in PD.columns: PD.drop(columns=['ComKeyBasic'], inplace=True)
# PD.insert(insert_idx, 'ComKeyBasic', val_to_insert) # 3. Insert ลงไปที่ตำแหน่งนั้น
all = all.merge(Comment, on = 'ComKeyBasic')

# 3. แก้ไข Syntax replace (เปลี่ยนเป็น Dictionary)
# เปลี่ยนคำที่ระบุ ให้เป็นค่าว่าง (None หรือ '')

all['Action_Log'] = all['Action_Log'].replace({
    'เติมราคากลาง': None, 
    'เติมAVG ด้วยCompare': None,
    'เติมCompare ด้วยLastPrice':None
})



# insert_idx = PD.columns.get_loc('DILLER_CODE')+1 # 1. หาตำแหน่ง Index ของคอลัมน์ 'CODE_7_DIGITS'
# all.insert()



In [36]:
df_sorted= all.copy()


In [37]:
# ==========================================
# 1. ส่วนคำนวณ Final_Master (AVG, REL, Count, Prop, Contrib)
# ==========================================
# (สมมติว่ามี df_sorted อยู่แล้ว)
# df_sorted  = df_sorted[(df_sorted['ผู้ดูแล']=='เฟิน') | (df_sorted['ผู้ดูแล']=='วิรัตน์')]
df_sorted = df_sorted[df_sorted['TOTAL_FLAGS']=='คำนวณ']
# df_sorted.loc[(df_sorted['Online']=='Online') & (df_sorted['ราคากลาง'].isna()),'ราคากลาง'] = 'หมด'

allcenter_clean_applied = allcenter_clean.copy()
allcenter_clean_applied['CODE_7_DIGITS'] = allcenter_clean_applied['COMMODITY_CODE'].astype(str).str[0:7]
allcenter_clean_applied = allcenter_clean_applied.merge(RM, on='CODE_7_DIGITS', how='left')
allcenter_clean_applied.columns
allcenter_clean_applied = allcenter_clean_applied.reindex(columns=['ผู้ดูแล','CODE_7_DIGITS','COMMODITY_CODE','DESCRIPTION','ประเภทร้านค้า','ราคากลาง','ภาวะ','Online'])
# allcenter_clean_applied
# df_sorted = df_sorted[df_sorted['ผู้ดูแล']=='คำนวณ']
# --- 1.1 AVG ---
ResultAVG_All = df_sorted.groupby('CODE_7_DIGITS')['AVG'].mean().reset_index(name='Average_All')
ResultAVG_0 = df_sorted[df_sorted['LINK'] == 0].groupby('CODE_7_DIGITS')['AVG'].mean().reset_index(name='Average_L0')
ResultAVG_1 = df_sorted[df_sorted['LINK'] == 1].groupby('CODE_7_DIGITS')['AVG'].mean().reset_index(name='Average_L1')
ResultAVG_2 = df_sorted[df_sorted['LINK'] == 2].groupby('CODE_7_DIGITS')['AVG'].mean().reset_index(name='Average_L2')
ResultAVG_3 = df_sorted[df_sorted['LINK'] == 3].groupby('CODE_7_DIGITS')['AVG'].mean().reset_index(name='Average_L3')

dfs_avg_merge = [
    ResultAVG_All.set_index('CODE_7_DIGITS'), ResultAVG_0.set_index('CODE_7_DIGITS'),
    ResultAVG_1.set_index('CODE_7_DIGITS'), ResultAVG_2.set_index('CODE_7_DIGITS'), ResultAVG_3.set_index('CODE_7_DIGITS')
]
Result_final_AVG = pd.concat(dfs_avg_merge, axis=1, join='outer').reset_index()

# --- 1.2 REL ---
ResultREL_All = df_sorted.groupby('CODE_7_DIGITS')['REL'].mean().reset_index(name='REL_All')
ResultREL_0 = df_sorted[df_sorted['LINK'] == 0].groupby('CODE_7_DIGITS')['REL'].mean().reset_index(name='RelL0')
ResultREL_1 = df_sorted[df_sorted['LINK'] == 1].groupby('CODE_7_DIGITS')['REL'].mean().reset_index(name='RelL1')
ResultREL_2 = df_sorted[df_sorted['LINK'] == 2].groupby('CODE_7_DIGITS')['REL'].mean().reset_index(name='RelL2')
ResultREL_3 = df_sorted[df_sorted['LINK'] == 3].groupby('CODE_7_DIGITS')['REL'].mean().reset_index(name='RelL3')

dfs_rel_merge = [
    ResultREL_All.set_index('CODE_7_DIGITS'), ResultREL_0.set_index('CODE_7_DIGITS'),
    ResultREL_1.set_index('CODE_7_DIGITS'), ResultREL_2.set_index('CODE_7_DIGITS'), ResultREL_3.set_index('CODE_7_DIGITS')
]
Result_final_REL = pd.concat(dfs_rel_merge, axis=1, join='outer').reset_index()

# --- 1.3 Count ---
ResultCount_All = df_sorted.groupby('CODE_7_DIGITS').size().reset_index(name='Count_All')
ResultCount_0 = df_sorted[df_sorted['LINK'] == 0].groupby('CODE_7_DIGITS').size().reset_index(name='CountL0')
ResultCount_1 = df_sorted[df_sorted['LINK'] == 1].groupby('CODE_7_DIGITS').size().reset_index(name='CountL1')
ResultCount_2 = df_sorted[df_sorted['LINK'] == 2].groupby('CODE_7_DIGITS').size().reset_index(name='CountL2')
ResultCount_3 = df_sorted[df_sorted['LINK'] == 3].groupby('CODE_7_DIGITS').size().reset_index(name='CountL3')

dfs_count_merge = [
    ResultCount_All.set_index('CODE_7_DIGITS'), ResultCount_0.set_index('CODE_7_DIGITS'),
    ResultCount_1.set_index('CODE_7_DIGITS'), ResultCount_2.set_index('CODE_7_DIGITS'), ResultCount_3.set_index('CODE_7_DIGITS')
]
Result_final_Count = pd.concat(dfs_count_merge, axis=1, join='outer').reset_index()

# --- 1.4 Merge All & Text ---
Result_Owner = df_sorted.groupby('CODE_7_DIGITS')['ผู้ดูแล'].first().reset_index()
# ใช้ 'DESCRIPTION' แล้วเปลี่ยนชื่อเป็น 'รายการ'
# Result_Item = df_sorted.groupby('CODE_7_DIGITS')['DESCRIPTION'].first().reset_index(name='รายการ')
Result_Item = df_sorted.groupby('CODE_7_DIGITS')['รายการ'].first().reset_index()

Final_Master = pd.merge(Result_final_AVG, Result_final_REL, on='CODE_7_DIGITS', how='outer')
Final_Master = pd.merge(Final_Master, Result_final_Count, on='CODE_7_DIGITS', how='outer')
Final_Master = pd.merge(Final_Master, Result_Owner, on='CODE_7_DIGITS', how='left')
Final_Master = pd.merge(Final_Master, Result_Item, on='CODE_7_DIGITS', how='left')

# Fill NaN with 0 for calculation
cols_to_fix = ['CountL0', 'CountL1', 'CountL2', 'CountL3', 'RelL0', 'RelL1', 'RelL2', 'RelL3']
for col in cols_to_fix:
    if col in Final_Master.columns:
        Final_Master[col] = Final_Master[col].fillna(0)

# --- 1.5 Calculation (Prop & Contrib) ---
# Proportion
Final_Master['PropL0'] = Final_Master['CountL0'] / Final_Master['Count_All'] * 100
Final_Master['PropL1'] = Final_Master['CountL1'] / Final_Master['Count_All'] * 100
Final_Master['PropL2'] = Final_Master['CountL2'] / Final_Master['Count_All'] * 100
Final_Master['PropL3'] = Final_Master['CountL3'] / Final_Master['Count_All'] * 100

# Contribute
Final_Master['ContribL0'] = Final_Master['RelL0'] * Final_Master['PropL0'] / 100
Final_Master['ContribL1'] = Final_Master['RelL1'] * Final_Master['PropL1'] / 100
Final_Master['ContribL2'] = Final_Master['RelL2'] * Final_Master['PropL2'] / 100
Final_Master['ContribL3'] = Final_Master['RelL3'] * Final_Master['PropL3'] / 100

# Sum Contribute
Final_Master['Sum_Contrib'] = (
    Final_Master['ContribL0'] + Final_Master['ContribL1'] + 
    Final_Master['ContribL2'] + Final_Master['ContribL3']
)

# --- 1.6 Reorder Columns ---
cols_order = [
    'CODE_7_DIGITS',  'รายการ', 'ผู้ดูแล',
    'REL_All', 'RelL0', 'RelL1', 'RelL2', 'RelL3',
    'Count_All', 'CountL0', 'CountL1', 'CountL2', 'CountL3',
    'PropL0', 'PropL1', 'PropL2', 'PropL3',
    'ContribL0', 'ContribL1', 'ContribL2', 'ContribL3',
    'Sum_Contrib'
]
    # 'Average_All', 'Average_L0', 'Average_L1', 'Average_L2', 'Average_L3',
cols_order = [c for c in cols_order if c in Final_Master.columns]
Final_Master = Final_Master[cols_order]


# ==========================================
# 2. ฟังก์ชันสร้าง Sheet สรุป (Formatting Logic)
# ==========================================
def create_summary_sheet(wb, df):
    sheet_name = 'Summary'
    # ลบชีตเดิมถ้ามีซ้ำ
    if sheet_name in wb.sheetnames:
        del wb[sheet_name]
    ws = wb.create_sheet(sheet_name)

    # --- Config ---
    divider_cols = [
        'Average_All', 'REL_All', 'Count_All', 
        'PropL0', 'ContribL0', 'Sum_Contrib'
    ]

    custom_widths = {
        'CODE_7_DIGITS': 8, 'ผู้ดูแล': 6, 'รายการ': 10,
        'Average_All': 3, 'Average_L0': 3,
        'Average_L1': 3, 'Average_L2': 3, 'Average_L3': 3,
        'REL_All': 9, 'RelL0': 9, 'RelL1': 9, 'RelL2': 9, 'RelL3': 9,
        'Count_All': 10, 'CountL0': 10, 'CountL1': 10, 'CountL2': 10, 'CountL3': 10,
        'PropL0': 10, 'PropL1': 10, 'PropL2': 10, 'PropL3': 10,
        'ContribL0': 12, 'ContribL1': 12, 'ContribL2': 12, 'ContribL3': 12,
        'Sum_Contrib': 12
    }

    colors = {
        'Average': 'CCE5FF', 'REL': 'FFCC99', 'Count': 'CCFFCC',
        'Prop': 'E0E0E0', 'Contrib': 'FFCCCC', 'Sum': 'FFFF00',
        'Sum_Highlight': 'FFFF99', 'Header': 'EFEFEF',
        # [NEW] Conditional Colors for REL
        'REL_Bad': 'FFC7CE',   # แดงอ่อน
        'REL_Good': 'C6EFCE'   # เขียวอ่อน
    }

    # --- Write Data ---
    # Header
    for col_idx, col_name in enumerate(df.columns, 1):
        ws.cell(row=1, column=col_idx, value=col_name)
    # Rows
    for r_idx, row in enumerate(dataframe_to_rows(df, index=False, header=False), 2):
        for c_idx, value in enumerate(row, 1):
            ws.cell(row=r_idx, column=c_idx, value=value)

    # --- Styles Definition ---
    thin_border = Side(style='thin')
    left_border_style = Border(left=thin_border)
    full_border_style = Border(left=thin_border, right=thin_border, top=thin_border, bottom=thin_border)
    no_border = Border()
    
    font_bold = Font(bold=True)
    font_sum = Font(bold=True, color='FF0000')
    align_center = Alignment(horizontal='center', vertical='center')

    ws.freeze_panes = 'D2' # Freeze 3 Columns

    # --- Formatting Loop ---
    for col_idx, col_name in enumerate(df.columns, 1):
        col_letter = get_column_letter(col_idx)
        
        # 1. Width
        if col_name in custom_widths:
            ws.column_dimensions[col_letter].width = custom_widths[col_name]
        else:
            max_length = len(col_name)
            for cell in ws[col_letter]:
                try:
                    if len(str(cell.value)) > max_length:
                        max_length = len(str(cell.value))
                except: pass
            ws.column_dimensions[col_letter].width = min(max_length + 2, 50)

        # 2. Identify Column Type
        is_divider = col_name in divider_cols
        is_sum_col = 'Sum_Contrib' in col_name
        is_count = 'Count' in col_name
        is_rel = 'REL' in col_name # [NEW]
        
        # 3. Apply Header Style
        header_fill_color = colors['Header']
        if 'Average' in col_name: header_fill_color = colors['Average']
        elif 'REL' in col_name:   header_fill_color = colors['REL']
        elif 'Count' in col_name: header_fill_color = colors['Count']
        elif 'Prop' in col_name:  header_fill_color = colors['Prop']
        elif 'Contrib' in col_name: header_fill_color = colors['Contrib']
        elif 'Sum' in col_name:   header_fill_color = colors['Sum']
        
        header_cell = ws.cell(row=1, column=col_idx)
        header_cell.font = font_bold
        header_cell.alignment = align_center
        header_cell.fill = PatternFill(start_color=header_fill_color, end_color=header_fill_color, fill_type='solid')
        
        if is_sum_col:
            header_cell.border = full_border_style
        elif is_divider:
            header_cell.border = left_border_style
        else:
            header_cell.border = no_border

        # 4. Apply Data Style
        for cell in ws[col_letter][1:]: 
            # --- Number Format ---
            if col_idx > 3: 
                if is_count: 
                    # จำนวนเต็ม, ซ่อน 0
                    cell.number_format = '#,##0;-#,##0;""'
                else: 
                    # ถ้า 100 ให้แสดงจำนวนเต็ม, ถ้า 0 ให้ว่าง, นอกนั้นทศนิยม 2 ตำแหน่ง
                    cell.number_format = '[=100]0;[=0]"";#,##0.00;""'
            
            # --- Initialize Fill & Border ---
            cell_fill = None
            cell_border = no_border

            # --- Conditional Logic (REL) ---
            if is_rel and isinstance(cell.value, (int, float)):
                val = cell.value
                if val != 0: # ไม่ใส่สีถ้าเป็น 0
                    if val < 80:
                        cell_fill = PatternFill(start_color=colors['REL_Bad'], end_color=colors['REL_Bad'], fill_type='solid')
                    elif val > 120: # [UPDATED] ใส่สีเขียวเฉพาะที่มากกว่า 102
                        cell_fill = PatternFill(start_color=colors['REL_Good'], end_color=colors['REL_Good'], fill_type='solid')

            # --- Special Columns Overwrite ---
            if is_sum_col:
                cell_fill = PatternFill(start_color=colors['Sum_Highlight'], end_color=colors['Sum_Highlight'], fill_type='solid')
                cell.font = font_sum
                cell_border = full_border_style
            elif is_divider:
                cell_border = left_border_style
            
            # --- Apply Final Styles ---
            if cell_fill:
                cell.fill = cell_fill
            cell.border = cell_border

# ==========================================
# 3. Main Execution
# ==========================================

# เตรียม Dataframes แยก Link
# data_mouthG = df.copy() # (ถ้ามี)
link1_df = df_sorted[df_sorted['LINK'] == 1].copy()
link2_df = df_sorted[df_sorted['LINK'] == 2].copy()
link3_df = df_sorted[df_sorted['LINK'] == 3].copy()

# สร้าง detect_df ขึ้นมาก่อน
detect_df = df_sorted[(df_sorted['ผลการตรวจสอบ'] == 'ให้ตรวจสอบ ราคา(ModernTrade)') | 
                      (df_sorted['ผลการตรวจสอบ'] == 'ให้ตรวจสอบ REL(ModernTrade)')].copy()

# --- จุดที่แก้ไข (เปลี่ยน df_sorted เป็น detect_df ในวงเล็บ) ---
# เพื่อความปลอดภัย ใส่ if เช็คว่ามีข้อมูลไหมก่อนทำ
if not detect_df.empty:
    detect_df.loc[detect_df['ผลการตรวจสอบ'] == 'ให้ตรวจสอบ ราคา(ModernTrade)', 'ผลการตรวจสอบ AVG'] = 'ตรวจสอบ ราคา(ModernTrade)'
    detect_df.loc[detect_df['ผลการตรวจสอบ'] == 'ให้ตรวจสอบ REL(ModernTrade)', 'ผลการตรวจสอบ REL'] = 'ตรวจสอบ REL(ModernTrade)'

# ลบคอลัมน์ตามเดิม
detect_df.drop(columns=['ผลการตรวจสอบ'], inplace=True)

nonPro = all.copy()
pat = r'(?<!หมด)โปร|เเถม|แถม|1\+1|ซื้อ ?2'
mask = (
    nonPro['COMM1'].notna() &
    ~nonPro['COMM1'].str.contains(pat, na=False) & 
    # ~nonPro['COMM2'].str.contains(pat, na=False) & 
    ~nonPro['COMM2'].astype(str).str.contains(pat, na=False) &
    ((nonPro['REL'] > 200) | (nonPro['REL'] < 50))
)
nonPro = nonPro[mask].drop_duplicates(subset='ComKeyAdv', keep=False).copy()
nonPro = nonPro[(nonPro['REL']==0)|(nonPro['ผลการตรวจสอบ']).notna()]

# detect_df.drop(columns=['ผลการตรวจสอบ'], inplace=True)
# middle_df = all[(all['ผลต่างราคากับราคากลาง'] > 0)|((all['Online']=='Online') & (all['REL']==0))].copy()
# middle_df = middle_df[middle_df['ราคากลาง']>0]
middle_df = all.copy()
middle_df = middle_df[(middle_df['ผลต่างราคากับราคากลาง'] > 0)|((middle_df['Online']=='Online') & (middle_df['AVG'].isna()))]
middle_df = middle_df[middle_df['LINK']!=3]


repetition = all.copy()
repetition = repetition[repetition['นับตัวซ้ำ'] != 1]

Impossible = all.copy()
Impossible = Impossible[Impossible['ผลการตรวจสอบ'] == 'ให้ตรวจสอบการเปลี่ยนแปลงราคาในอดีต']

RelZeros = all.copy()
RelZeros = RelZeros[(RelZeros['REL'] == 0)|(RelZeros['REL'].isna())]




# ตั้งชื่อไฟล์ Output
output_file = filename + "_multi_sheet.xlsx"

# สร้าง Workbook หลัก
wb = Workbook()
# ลบชีตเริ่มต้นทิ้ง
if 'Sheet' in wb.sheetnames:
    del wb['Sheet']

# ใส่ข้อมูล Sheet ต่างๆ
try:
    Excel_Condition3(all, wb, 'ข้อมูล')
    Excel_Condition3(allcenter_clean_applied, wb, 'ราคากลาง')
    Excel_Condition3(middle_df, wb, 'ราคากลางไม่ตรง')
    Excel_Condition3(repetition, wb, 'ซ้ำ')
    Excel_Condition3(RelZeros, wb, 'ศูนย์') 
    Excel_Condition3(nonPro, wb, 'RELเกิน')

    Excel_Condition3(Impossible, wb, 'เปลี่ยนแปลงราคา')    
    Excel_Condition3(detect_df, wb, 'ModernTrade')
    Excel_Condition3(link1_df, wb, 'l1')
    Excel_Condition3(link2_df, wb, 'l2')
    Excel_Condition3(link3_df, wb, 'l3')

except NameError:
    print("Warning: Excel_Condition3 function not found. Skipping individual sheets.")

# เรียกใช้ฟังก์ชันสร้าง Sheet สรุป
print("Creating Summary Sheet...")
create_summary_sheet(wb, Final_Master)

# บันทึกไฟล์
wb.save(output_file)
print(f"✅ Successfully saved DataFrames to '{output_file}' with Summary sheet.")



Creating Summary Sheet...
✅ Successfully saved DataFrames to 'cpig_month_2569-2_multi_sheet.xlsx' with Summary sheet.


In [38]:
# middle_df = all[(all['ผลต่างราคากับราคากลาง'] > 0)|((all['Online']=='Online') & (all['REL']==0))].copy()
# middle_df = middle_df[middle_df['ราคากลาง']>0]
# middle_df = middle_df[middle_df['LINK']!=3]
middle_df = all.copy()
middle_df = middle_df[(middle_df['ผลต่างราคากับราคากลาง'] > 0)|((middle_df['Online']=='Online') & (middle_df['AVG'].isna()))]
# [(all['ผลต่างราคากับราคากลาง'] > 0)]


print(len(middle_df[middle_df['ผู้ดูแล']=='C']))

middle_df[middle_df['ผู้ดูแล']=='C']['DESCRIPTION']

3


284     ขนมปังปอนด์ ชนิดแผ่น\ บรรจุถุงพลาสติก  480 กรั...
717     อาหารธัญพืช   บรรจุกล่อง\  น้ำหนัก 150 กรัม \ ...
1081    ขนมขบเคี้ยว \ บรรจุห่อ  26 กรัม\ ตราฮานามิ ข้า...
Name: DESCRIPTION, dtype: object

In [39]:
# import pandas as pd
# from scipy.stats import gmean
# import math

# # ฟังก์ชันปัดเศษขึ้นตามที่คุณกำหนด
# def round_up(n, decimals=0):
#     if pd.isna(n): return n
#     multiplier = 10 ** decimals
#     return math.ceil(n * multiplier) / multiplier

# # 1. เตรียมข้อมูล (สมมติว่ามี df_sorted อยู่แล้ว)
# # 2. ทำการ Group by 'CODE7' และคำนวณ gmean สำหรับ 'AVG' และ 'REL'


# report = df.groupby(['CODE7','TYPE'])[['AVG','COMPARE_PRICE', 'REL','LAST_PRICE']].agg(gmean).reset_index()

# # 3. ปัดเศษขึ้น 2 ตำแหน่งตามต้องการ
# report['AVG'] = report['AVG'].apply(lambda x: round_up(x, 2))
# report['REL'] = report['REL'].apply(lambda x: round_up(x, 2))
# report['COMPARE_PRICE'] = report['COMPARE_PRICE'].apply(lambda x: round_up(x, 2))
# report['LAST_PRICE'] = report['LAST_PRICE'].apply(lambda x: round_up(x, 2))

# # 4. เรียงลำดับตาม CODE7 (ถ้ายังไม่ได้เรียง)
# report = report.sort_values('CODE7').reset_index(drop=True)

# # แสดงผลลัพธ์
# report.to_excel('Test_report.xlsx')

In [40]:
# type(nonPro['COMM1'].loc[1303])

In [41]:
# df_sorted[columns_to_keep].to_clipboard()
# หาราคากลางที่ยังผิด
df_sorted[(df_sorted['ราคากลาง']>0)&(df_sorted['COMM1'].isna())&(df_sorted['ผลต่างราคากับราคากลาง']!=0)][columns_to_keep].to_clipboard(index=False)
df_sorted[(df_sorted['ราคากลาง']>0)&(df_sorted['COMM1'].isna())&(df_sorted['ผลต่างราคากับราคากลาง']!=0)][columns_to_keep]

,G_FLAG,L_FLAG,N_FLAG,CODE7,CODE9,ภาค,GROUP,COMMODITY_CODE,DESCRIPTION,DILLER_CODE,...,COMM1,COMM2,Online,ราคากลาง,ผลต่างราคากับราคากลาง,นับตัวซ้ำ,AVGtail_stable,ผลการตรวจสอบ,ComKeyBasic,Action_Log


<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [42]:
xxx
777777777777

NameError: name 'xxx' is not defined

In [ ]:
# ห้ามลบ
# link1_df = df_sorted[df_sorted['LINK'] == 1].copy()
# link2_df = df_sorted[df_sorted['LINK'] == 2].copy()
# link3_df = df_sorted[df_sorted['LINK'] == 3].copy()

# output_file = filename + "_multi_sheet_dataG.xlsx"

# wb = Workbook()

# # ลบชีตเริ่มต้น (Sheet) ทิ้งก่อน
# default_ws = wb.active
# wb.remove(default_ws)

# Excel_Condition3(df_sorted,     wb, 'ข้อมูลดิบ')
# Excel_Condition3(allcenter_clean, wb, 'ราคากลาง')
# Excel_Condition3(link1_df,        wb, 'link1')
# Excel_Condition3(link2_df,        wb, 'link2')
# Excel_Condition3(link3_df,        wb, 'link3')
# summay
# wb.save(output_file)
# print(f"✅ Successfully saved DataFrames to '{output_file}' with 5 sheets.")

#### จัดการ ขนาด ตรา

In [ ]:
# ==========================================
# 1. Configuration & Dictionaries
# ==========================================

# 1.1 คำที่ใช้ตัดจบชื่อสินค้า (Stop Words for GP Name)
# เจอคำพวกนี้ปุ๊บ ให้ถือว่าจบชื่อสินค้าทันที
STOP_WORDS_GP = [
    'บรร', 'น้ำหนัก', 'จำนวน', 'ขนาด', 'กว้าง', 'ชนิด', 
    'กล่อง', 'ใส่', 'คุณภาพ', 'กำ', 'เสียบ', 'ตรา', 'สูตร', 'ปริมาณ'
]

# 1.2 คลังคำศัพท์ภาชนะ (Packaging)
CONTAINERS = [
    'บรรจุ', 'ใส่', 'ชนิด', 
    'ขวดแก้ว', 'ขวดพลาสติก', 'ขวด', 'กระป๋อง', 'กระปุก', 'กล่อง', 'ซอง', 'ถุง', 
    'แพ็ค', 'แพค', 'แกลลอน', 'ปี๊บ', 'ถัง', 'ตลับ', 'หลอด', 'ห่อ', 'กระสอบ', 'ตระกร้า', 'เข่ง',
    'BOTTLE', 'CAN', 'BOX', 'BAG', 'POUCH', 'PACK', 'GALLON', 'JAR', 'TUBE', 'SACHET', 'TIN'
]

# 1.3 คลังหน่วยนับ (Universal Units)
UNIT_REGEX_PARTS = [
    # Weight
    r'กิโลกรัม', r'กรัม', r'มิลลิกรัม', r'กก\.?', r'ก\.ก\.?', r'gm', r'gms', r'kg', r'kgs', r'mg', r'lb', r'lbs', r'pound', r'ounce', r'oz',
    # Volume
    r'มิลลิลิตร', r'ลิตร', r'ซีซี', r'ม\.ล\.?', r'มล\.?', r'ml', r'ltr', r'liters', r'liter', r'cc', r'fl\s*oz', r'gallon', r'gal',
    # Length
    r'เซนติเมตร', r'เมตร', r'นิ้ว', r'มิลลิเมตร', r'cm', r'mm', r'meter', r'inch', r'ft',
    # Count / Special
    r'ลูก', r'เม็ด', r'ชิ้น', r'ม้วน', r'กำ', r'แผ่น', r'แท่ง', r'ฟอง', r'capsule', r'tablet', r'sheet', r'roll', r'pc', r'pcs'
]
UNITS_REGEX = r'(?:' + r'|'.join(UNIT_REGEX_PARTS) + r')'

# 1.4 Regex ตัวเลข (รองรับ , และ .)
NUM_REGEX = r'(?:\d{1,3}(?:,\d{3})*|\d+)(?:\.\d+)?'

# ==========================================
# 2. Helper Functions
# ==========================================

def clean_text(text):
    if pd.isna(text): return ""
    return str(text).replace('\xa0', ' ').strip()

def extract_brand_universal(text):
    text = clean_text(text)
    # 1. หาหลังคำว่า "ตรา"
    match = re.search(r'ตรา\s*([ก-๙a-zA-Z0-9\.\-]+)', text)
    if match: return match.group(1).strip()
    # 2. หาในวงเล็บภาษาอังกฤษตัวใหญ่ (กันเหนียว)
    match = re.search(r'\(([A-Z][a-zA-Z\s]+)\)', text)
    if match and 'RED CUP' not in match.group(1): 
        return match.group(1).strip()
    return None

def standardize_units_universal(series):
    mapping = {
        r'(?i)(มล\.?|ม\.ล\.?|\.ม\.ล\.?|ซีซี\.?|cc\.?|ml\.?|mls\.?)$': 'มิลลิลิตร',
        r'(?i)(ลิตร|ltr|liter|liters|l\.?)$': 'ลิตร',
        r'(?i)(fl\s*oz|oz\s*fl)$': 'ออนซ์',
        r'(?i)(gallon|gal)$': 'แกลลอน',
        r'(?i)(กก\.?|ก\.ก\.?|k\.?g\.?|kgs\.?|kilogram)$': 'กิโลกรัม',
        r'(?i)(กรัม|g\.?|gm\.?|gms\.?|gram)$': 'กรัม',
        r'(?i)(mg\.?|mgs\.?|milligram)$': 'มิลลิกรัม',
        r'(?i)(lb|lbs|pound)$': 'ปอนด์',
        r'(?i)(oz|ounce)$': 'ออนซ์',
        r'(?i)(pcs|pc|ชิ้น)$': 'ชิ้น',
        r'(?i)(pack|packs|แพ็ค|แพค)$': 'แพ็ค',
        r'(?i)(นิ้ว|inch|\")$': 'นิ้ว',
    }
    s = series.astype(str).str.lower().str.strip()
    for pat, repl in mapping.items():
        s = s.str.replace(pat, repl, regex=True)
    return s.str.replace(r'\.$', '', regex=True)

# ==========================================
# 3. Main Logic
# ==========================================
# เตรียมข้อมูล
df_spec = df[['CODE_7_DIGITS', 'COMMODITY_CODE', 'DESCRIPTION']].copy()
df_spec = df_spec.replace("-", None).dropna(subset=['DESCRIPTION'])
df_spec['COMMODITY_CODE'] = df_spec['COMMODITY_CODE'].astype(str).str.zfill(16)
df_spec['Clean_Desc'] = df_spec['DESCRIPTION'].apply(clean_text)

# --- 3.1 GP_name Extraction (NEW!) ---
# Regex: จับทุกอย่างตั้งแต่ต้น จนกว่าจะเจอคำใน STOP_WORDS_GP
gp_pattern = fr'(.*?)(?={"|".join(STOP_WORDS_GP)})'
df_spec['GP_name'] = df_spec['Clean_Desc'].str.extract(gp_pattern)[0]
# Fallback: ถ้าไม่เจอคำหยุดเลย ให้เอาทั้งประโยคเป็นชื่อ
df_spec['GP_name'] = df_spec['GP_name'].fillna(df_spec['Clean_Desc'])
# Clean: ลบเครื่องหมาย \ หรือวงเล็บเปิดทิ้งท้าย
df_spec['GP_name'] = df_spec['GP_name'].str.replace(r'[\\/(\s]+$', '', regex=True).str.strip()


# --- 3.2 Brand Extraction ---
df_spec['Band'] = df_spec['Clean_Desc'].apply(extract_brand_universal)
df_spec['Bandรายละเอียด'] = df_spec['DESCRIPTION'].str.extract(r'(ตรา\s*\S.*)')

# --- 3.3 Packaging Extraction (Smart Search) ---
# Explicit search
pack_regex_explicit = r'(?:บรรจุ|ใส่|ชนิด)\s*([ก-๙a-zA-Z\s\-]+?)(?=\s*\d|\\|$)'
df_spec['การบรรจุ'] = df_spec['Clean_Desc'].str.extract(pack_regex_explicit)[0].str.strip()

# Keyword search (Fallback)
container_pattern = r'|'.join(CONTAINERS)
mask_no_pack = df_spec['การบรรจุ'].isna() | (df_spec['การบรรจุ'] == '')
df_spec.loc[mask_no_pack, 'การบรรจุ'] = df_spec.loc[mask_no_pack, 'Clean_Desc'].str.extract(f'({container_pattern})', flags=re.IGNORECASE)[0]

# --- 3.4 Quantity & Units (Multipack Logic) ---
# Logic A: Multipack (3 x 100 ml)
multipack_regex = fr'({NUM_REGEX})\s*[xX*]\s*({NUM_REGEX})\s*({UNITS_REGEX})'
multi_match = df_spec['Clean_Desc'].str.extract(multipack_regex)

# Logic B: Single (500 ml)
single_regex = fr'({NUM_REGEX})\s*({UNITS_REGEX})'
single_match = df_spec['Clean_Desc'].str.extract(single_regex)

# Merge Logic
df_spec['ปริมาณ'] = single_match[0]
df_spec['หน่วย'] = single_match[1]
df_spec['จำนวนต่อหน่วยบรรจุ'] = 1
df_spec['จำนวนต่อหน่วยบรรจุ'] = df_spec['จำนวนต่อหน่วยบรรจุ'].astype('object') # แปลงเป็น object ก่อน เพื่อให้รับค่า String จาก Regex ได้

mask_multi = multi_match[0].notna()
df_spec.loc[mask_multi, 'จำนวนต่อหน่วยบรรจุ'] = multi_match.loc[mask_multi, 0]
df_spec.loc[mask_multi, 'ปริมาณ'] = multi_match.loc[mask_multi, 1]
df_spec.loc[mask_multi, 'หน่วย'] = multi_match.loc[mask_multi, 2]

# Cleaning Numbers
df_spec['ปริมาณ'] = df_spec['ปริมาณ'].str.replace(',', '', regex=False)
df_spec['ปริมาณ'] = pd.to_numeric(df_spec['ปริมาณ'], errors='coerce')

df_spec['จำนวนต่อหน่วยบรรจุ'] = df_spec['จำนวนต่อหน่วยบรรจุ'].astype(str).str.replace(',', '', regex=False)
df_spec['จำนวนต่อหน่วยบรรจุ'] = pd.to_numeric(df_spec['จำนวนต่อหน่วยบรรจุ'], errors='coerce').fillna(1)

# Standardize Units
df_spec['หน่วย'] = standardize_units_universal(df_spec['หน่วย'])

# --- 3.5 Defaults ---
defaults_map = {
    r'ขนาด\s*8\s*[\"นิ้ว]': (8, 'นิ้ว', 1),
    r'สำหรับบริโภค 4 คน': (1, 'ชุด', 1),
    r'1\s*เล่ม': (1, 'เล่ม', 1),
    r'1\s*ชิ้น': (1, 'ชิ้น', 1)
}
for pat, (qty, unit, count) in defaults_map.items():
    mask = df_spec['ปริมาณ'].isna() & df_spec['Clean_Desc'].str.contains(pat, regex=True, na=False)
    df_spec.loc[mask, 'ปริมาณ'] = qty
    df_spec.loc[mask, 'หน่วย'] = unit
    df_spec.loc[mask, 'จำนวนต่อหน่วยบรรจุ'] = count

# ==========================================
# 4. Final Output
# ==========================================
final_cols = ['COMMODITY_CODE', 'DESCRIPTION', 'GP_name', 'Band', 'การบรรจุ', 'จำนวนต่อหน่วยบรรจุ', 'ปริมาณ', 'หน่วย']
df_result = df_spec[final_cols].drop_duplicates()

df_result.to_clipboard()
print("Universal Parsing Completed.")
print(df_result.head())

In [ ]:
# ==========================================
# 5. Refine Brand (ซ่อมแซมชื่อตราสินค้า)
# ==========================================

def refine_brand_name(val):
    """ฟังก์ชันจัดระเบียบชื่อแบรนด์ แก้คำผิด และรวมกลุ่ม"""
    if not isinstance(val, str): return val
    
    # แปลงเป็นตัวพิมพ์เล็กบางส่วนเพื่อเช็คง่าย (แต่ตอน return คืนค่าปกติ)
    # ใช้หลักการ: เช็คคำเฉพาะเจาะจงก่อน -> คำทั่วไปทีหลัง
    
    # --- กลุ่มผงซักฟอก/น้ำยาปรับผ้านุ่ม ---
    if any(x in val for x in ["Fineline", "ไฟน์ไลน์", "ไฟไลน์"]): return "ตราไฟน์ไลน์"
    if any(x in val for x in ["แอคแทค", "แอทแทค"]): return "ตราแอคแทค"
    if any(x in val for x in ["ดาวน์นี่", "ดาาวน์นี่"]): return "ตราดาวน์นี่"
    if any(x in val for x in ["เปา", "เปาซิลเวอร์", "เปาวินวอช"]): return "ตราเปา"
    if any(x in val for x in ["โอโม", "โอโมพลัส"]): return "ตราโอโม"
    if any(x in val for x in ["คอมฟอร์ท"]): return "ตราคอมฟอร์ท"
    
    # Breeze (แยก Excel กับ ธรรมดา ตาม Logic เดิม)
    if any(x in val for x in ["บรีสเอ็กเซล", "บรีสเอกเซล"]): return "ตราบรีสเอ็กเซล"
    if any(x in val for x in ["บรีส", "บรีสพาวเวอร์"]): return "ตราบรีส"
    
    # Essence (เขียนผิดบ่อยมาก)
    if any(x in val for x in ["เอสเซ้น", "เอสเซนต์", "เอสเซน", "เอสเซ็น"]): return "ตราเอสเซ้นซ์"

    # --- กลุ่มสบู่/แชมพู ---
    if any(x in val for x in ["Be", "be", "บีไนซ์"]): return "ตราBeNice" # ระวังคำว่า Be อาจกว้างไป แต่ตาม Logic เดิม
    if any(x in val for x in ["ซัลซิล", "ซันซิล"]): return "ตราซันซิล"
    if any(x in val for x in ["โชกุบุซึ", "โชกุบุส", "Shokubutsu"]): return "ตราโชกุบุซึ"
    if any(x in val for x in ["โพรเทค", "โพเทค"]): return "ตราโพรเทค"
    if any(x in val for x in ["แพนทีน", "Pantene"]): return "ตราแพนทีน"
    if any(x in val for x in ["Feather", "แฟซ่า"]): return "ตราแฟซ่า"
    if any(x in val for x in ["เฮด", "Head"]): return "ตราเฮดแอนด์โชว์เดอร์"
    if any(x in val for x in ["ลักซ์", "ลักส์", "Lux"]): return "ตราลักส์"

    # --- แก้ไขจุดที่น่าจะ Copy ผิด ---
    # เดิม: "ตราแพรอทพฤกษา" map ไป "ตราแพนทีน" (น่าจะผิด)
    # แก้ไข: map ไป "ตรานกแก้ว" (Parrot) หรือ "ตราพฤกษา"
    if any(x in val for x in ["แพรอท", "พฤกษา", "Parrot"]): return "ตราพฤกษานกแก้ว"

    # --- กลุ่มร้านค้า (Store Brands) ---
    if any(x in val for x in ["อีเลฟเว่น", "ELEVEN", "7 - 11", "7-11", "เซเว่น", "seven"]): return "Seven Eleven"

    return val

# เรียกใช้ฟังก์ชันซ่อมแบรนด์
df_spec['Band'] = df_spec['Band'].apply(refine_brand_name)

# ==========================================
# 6. Merge & Export
# ==========================================

# สมมติว่า Test คือ DataFrame หลักที่มีอยู่แล้วตามโจทย์
# เลือกเฉพาะคอลัมน์ที่ต้องการจาก df_spec เพื่อไปแปะ
cols_to_merge = ['COMMODITY_CODE', 'Band', 'Bandรายละเอียด', 'การบรรจุ', 'จำนวนต่อหน่วยบรรจุ', 'ปริมาณ', 'หน่วย']

# Merge กับ Test
# หมายเหตุ: ใช้ COMMODITY_CODE เป็น Key
# ถ้า df_spec มี COMMODITY_CODE ซ้ำ ต้องระวังจำนวนแถวเพิ่ม (ใช้ drop_duplicates ช่วย)
df_clean_unique = df_spec[cols_to_merge].drop_duplicates(subset=['COMMODITY_CODE'])

df_final = df.merge(df_clean_unique, on=['COMMODITY_CODE'], how='left')

# ส่งออก
df_final['ราคาต่อหน่วย'] = df_final['AVG']/(df_final['จำนวนต่อหน่วยบรรจุ']*df_final['ปริมาณ'])
df_final.to_clipboard()
print("Process Completed: Brand Refined & Merged.")
print(df_final[['COMMODITY_CODE', 'Band', 'การบรรจุ']].head())

#### Z-score and IQR

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import warnings

# ==========================================
# 1. ฟังก์ชันคำนวณ Box-Cox & Z-Score (Robust Version)
# ==========================================
def calculate_boxcox_zscore(x):
    # กรณีข้อมูลไม่เพียงพอ หรือ เท่ากันหมด (Std=0)
    if len(x) < 2 or x.nunique() == 1:
        return pd.Series(0, index=x.index)
    
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        try:
            # Box-Cox Transformation
            transformed_data, _ = stats.boxcox(x)
            transformed_series = pd.Series(transformed_data, index=x.index)
            
            # Check Infinity/NaN
            if not np.isfinite(transformed_series).all():
                return pd.Series(0, index=x.index)
            
            # Calculate Z-Score
            z_scores = (transformed_series - transformed_series.mean()) / transformed_series.std(ddof=1)
            return z_scores.fillna(0)
            
        except Exception:
            return pd.Series(0, index=x.index)

# ==========================================
# 2. เตรียมข้อมูล (Data Preparation)
# ==========================================
# แยกข้อมูลเป็น 2 ส่วน: ส่วนที่คำนวณได้ (AVG > 0) และส่วนที่คำนวณไม่ได้
mask_valid = (df_final['AVG'] > 0) & (df_final['AVG'].notna())
process_df = df_final[mask_valid].copy()
ignored_df = df_final[~mask_valid].copy()

# ==========================================
# 3. คำนวณทางสถิติ (Statistical Calculation)
# ==========================================
# 3.1 คำนวณ Z-Score ผ่าน Box-Cox
process_df['Z_Score_Global'] = process_df.groupby(['CODE_7_DIGITS', 'DESCRIPTION'])['AVG'].transform(calculate_boxcox_zscore)
# process_df['Z_Score_Local'] = process_df.groupby(['CODE_7_DIGITS', 'ประเภทร้านค้า', 'DESCRIPTION'])['AVG'].transform(calculate_boxcox_zscore) # (Optional: ถ้าไม่ได้ใช้ปิดไว้ประหยัดเวลา)

# Fillna & Absolute & Round
process_df['Sum_Z-Score'] = process_df['Z_Score_Global'].fillna(0).abs().round(2)

# 3.2 คำนวณสัดส่วน % (Vectorized)
grouper = process_df.groupby("DESCRIPTION")['AVG']
process_df['cnt_items'] = grouper.transform('count')
process_df['pct_share'] = (process_df['AVG'] / grouper.transform('sum') * 100).round(2)

# ==========================================
# 4. ตรวจจับความผิดปกติ (Anomaly Detection Rules)
# ==========================================

# --- Rule 1: Basic Z-Score > 2 ---
conditions = [ process_df['Sum_Z-Score'] > 2 ]
choices = [ 'ผิดปกติ(Z)' ]

# --- Rule 2: Special Distribution Rules (Count & % & Z-Score) ---
# นิยามเงื่อนไขพิเศษตามจำนวนสินค้าในกลุ่ม (3, 4, 5 ชิ้น)
c_cnt = process_df['cnt_items']
c_pct = process_df['pct_share']
c_z   = process_df['Sum_Z-Score']

mask_rule3 = (c_cnt == 3) & ((c_pct < 19) | (c_pct > 51)) & (c_z > 1.0)
mask_rule4 = (c_cnt == 4) & ((c_pct < 14) | (c_pct > 41)) & (c_z > 1.3)
mask_rule5 = (c_cnt == 5) & ((c_pct < 12) | (c_pct > 32)) & (c_z > 1.5)

# รวมเงื่อนไขพิเศษ
process_df['Flag_Special'] = np.select(
    [mask_rule3, mask_rule4, mask_rule5], 
    ['ผิดปกติ(Z)', 'ผิดปกติ(Z)', 'ผิดปกติ(Z)'], 
    default=''
)

# --- Combine Flags ---
# ใช้ Rule 1 เป็นหลัก ถ้า Rule 1 ว่าง ให้ใช้ Rule 2
process_df['Report_Z>=2'] = np.select(conditions, choices, default='')
mask_empty_report = (process_df['Report_Z>=2'] == '') | (process_df['Report_Z>=2'].isna())
process_df.loc[mask_empty_report, 'Report_Z>=2'] = process_df.loc[mask_empty_report, 'Flag_Special']

# ==========================================
# 5. รวมข้อมูลและทำความสะอาด (Finalize)
# ==========================================
# ลบคอลัมน์ชั่วคราวทิ้งก่อนรวม
cols_to_drop = ['cnt_items', 'pct_share', 'Flag_Special']
process_df = process_df.drop(columns=cols_to_drop, errors='ignore')

# รวมข้อมูลกลับ
df_final = pd.concat([process_df, ignored_df], ignore_index=True)

# --- Final Filters ---
# 1. ล้าง Flag ถ้าไม่ใช่ "ร้านอื่นๆ"
df_final.loc[df_final['ประเภทร้านค้า'] != 'ร้านอื่นๆ', 'Report_Z>=2'] = ''

# 2. ล้าง Flag ถ้ามีราคาซ้ำกัน >= 2 ครั้ง (เฉพาะร้านอื่นๆ)
dup_counts = df_final.groupby(['COMMODITY_CODE', 'AVG'])['AVG'].transform('count')
mask_dup_safe = (df_final['ประเภทร้านค้า'] == 'ร้านอื่นๆ') & (dup_counts >= 2)
df_final.loc[mask_dup_safe, 'Report_Z>=2'] = ''


# # 3. สร้างผลลัพธ์สุดท้าย
# # (สมมติว่ามีคอลัมน์ 'ผล' อยู่แล้ว ถ้าไม่มีให้ลบ .fillna('') ตัวหลังออก)
# if 'ผล' in df_final.columns:
#     df_final['ผลลัพธ์'] = df_final['Report_Z>=2'].replace('', np.nan).fillna(df_final['ผล'].fillna(''))
# else:
#     df_final['ผลลัพธ์'] = df_final['Report_Z>=2']

# 4. จัดเรียง
df_final = df_final.sort_values(
    by=['COMMODITY_CODE', 'DESCRIPTION', 'ประเภทร้านค้า', 'AVG'], 
    ascending=[True, True, True, False]
).reset_index(drop=True)

df_final.to_clipboard()
# แสดงผล
# print(df_final.head())

In [ ]:
import pandas as pd
import numpy as np

# ==============================================================================
# สร้าง Column ใหม่: Report_IQR (ใช้หลักการ Interquartile Range)
# ==============================================================================

# 1. เตรียมข้อมูลเฉพาะส่วนที่คำนวณได้ (AVG > 0)
mask_calc = (df_final['AVG'] > 0) & (df_final['AVG'].notna())

# สร้าง DataFrame ชั่วคราวเพื่อคำนวณ (ใช้ Index เดิมเพื่อ Mapping กลับ)
calc_df = df_final[mask_calc].copy()

# 2. คำนวณ Q1, Q3 และ IQR แยกตามกลุ่มสินค้า
# Group by CODE_7_DIGITS และ DESCRIPTION
grouper = calc_df.groupby(['CODE_7_DIGITS', 'DESCRIPTION'])['AVG']

# ใช้ transform เพื่อให้ค่ากลับมาเท่ากับจำนวนแถวเดิม
Q1 = grouper.transform(lambda x: x.quantile(0.25))
Q3 = grouper.transform(lambda x: x.quantile(0.75))
IQR = Q3 - Q1

# คำนวณขอบเขต (Bounds)
Lower_Bound = Q1 - (1.5 * IQR)
Upper_Bound = Q3 + (1.5 * IQR)

# 3. สร้างคอลัมน์ใหม่ใน df_final (ค่าเริ่มต้นเป็นว่าง)
df_final['Report_IQR'] = ''

# 4. ตรวจจับ Outlier
# เงื่อนไข: น้อยกว่าขอบล่าง OR มากกว่าขอบบน
mask_outlier_iqr = (calc_df['AVG'] < Lower_Bound) | (calc_df['AVG'] > Upper_Bound)

# Map ค่ากลับไปที่ df_final โดยอ้างอิง Index
# (เฉพาะแถวที่เข้าเงื่อนไข Outlier จะถูกกาว่า 'ผิดปกติ(IQR)')
df_final.loc[mask_outlier_iqr[mask_outlier_iqr].index, 'Report_IQR'] = 'ผิดปกติ(IQR)'

# ==============================================================================
# 5. ใส่กฎยกเว้น (Business Rules) *สำคัญ*
# ==============================================================================

# กฎที่ 1: ล้าง Flag ถ้าไม่ใช่ "ร้านอื่นๆ" (ร้านมาตรฐานไม่เช็ค)
df_final.loc[df_final['ประเภทร้านค้า'] != 'ร้านอื่นๆ', 'Report_IQR'] = ''

# กฎที่ 2: ล้าง Flag ถ้ามีราคาซ้ำกัน >= 2 ครั้ง (เฉพาะร้านอื่นๆ)
# (ถ้ามีคนขายราคานี้เหมือนกัน 2 ร้าน แสดงว่าเป็นราคาตลาดจริง ไม่ใช่ Outlier)
dup_counts = df_final.groupby(['COMMODITY_CODE', 'AVG'])['AVG'].transform('count')
mask_dup_safe = (df_final['ประเภทร้านค้า'] == 'ร้านอื่นๆ') & (dup_counts >= 2)
df_final.loc[mask_dup_safe, 'Report_IQR'] = ''

# ==============================================================================
# 6. จัดเรียงและส่งออก
# ==============================================================================
# จัดเรียงให้ดูง่าย
df_final = df_final.sort_values(
    by=['COMMODITY_CODE', 'DESCRIPTION', 'ประเภทร้านค้า', 'AVG'], 
    ascending=[True, True, True, False]
).reset_index(drop=True)

# ส่งออก
df_final.to_clipboard(index=False)
# print("เพิ่มคอลัมน์ Report_IQR เรียบร้อยแล้ว")
# print(df_final[['DESCRIPTION', 'AVG', 'Report>=2', 'Report_IQR']].head(10)) # โชว์เทียบกัน
df_final.to_clipboard()

#### Condition_time_series

#### สรุป Condition

#### คัดเลือก และ ยกเลิกการตรวจสินค้า

#### จัดการภาวะ

In [ ]:
# ---------------------------
# 1. เตรียม DataFrame (เปลี่ยนชื่อตัวแปรเป็น df_actions)
# ---------------------------
df_actions = df_final.copy() # หรือ df_actions = data_mouthG.copy()

# จัดการ Missing Value
df_actions["COMM1"] = df_actions["COMM1"].fillna("").astype(str)
df_actions["Action_Label"] = None

# ---------------------------
# 2. นิยาม Regex Patterns
# ---------------------------

# กลุ่ม "การเปลี่ยน"
REGEX_CHANGE_SHOP = r"ร้าน|อีกสาขา|บิ๊กซี|โลตัส|ท็อป|แม็คโคร|7-11|เซเว่น|เปลี่ยนร้าน|สาขา|แหล่ง|ชุปเปอร์ชิป"
REGEX_CHANGE_KEYWORD = r"เปลี่ยน|เปลียน" 
REGEX_CHANGE_PRODUCT_SPECS = (
    r"Spec|สเปค|เปลี่ยนสินค้า|รสชาติใหม่|รสใหม่|สูตรใหม่|สูตรเดิม|ตัวอื่นแทน|ยี่ห้อ|"
    r"สี|ข้าว|ตรา|ยี่ห้อ|ชนิด|แบบ|แพ๊ค|แพ็กเกจ|รส|กลิ่น|ขนาด|รุ่น|รูปแบบ|น้ำหนัก|ปริมาตร|เมนู|"
    r"เม็ด|ขนาด|ชิ้น|กรัม|มิลลิลิตร|ลิตร|กิโล|กก\.|แพ็ค|แพค|ถุง|ซอง|กล่อง|ขวด|กระป๋อง|ก้อน|ลูก|แท่ง|แผง|ห่อ|ลัง|ตู้|"
    r"คู่|อายุ|ฝา|แบรนด์|ตัว|มล\.|ml|mg|g|kg|ขีด|มก|ซีซี|cc|"
    r"ผืน|ส้มตำ|ใบมีด|จาน|ชุด|กระทะ|หม้อ|เตา|กะละมัง|ตะกร้า|ตะหลิว|ทัพพี|กระบวย|ตะแกรง|"
    r"กิ๊ฟบ็อก|พื้น|ด้าม|จีบ|ปูน|ผ้า|ผ้าเช็ดตัว|ผ้าขนหนู|ผ้าปูที่นอน|ปลอก|หมอน|ไซส์|size|เบอร์|แผ่น"
)

# คำยกเว้น
REGEX_IGNORE = r"เปลี่ยนโปร|โปรใหม่|โปรโมชั่นใหม่|เปลี่ยน promotion" 

# กลุ่มอื่นๆ
REGEX_ADD = r"ใหม่|เพืม|เพิ่มสินค้า|เพื่ม|เพิ่มสาขา|รายการเพิ่ม|เพิ่ม|ขอเพิ่ม|อยากได้เพิ่ม|เพิ่มจำนวน"
REGEX_PRICE_DOWN = r"ปรับราคา|ปรับตามราคา|จาก|เดิม|ปรับลง|ลดล้าง|ลดราคา|ปรับลดลง|ลดลง"
REGEX_PRICE_UP = r"ปรับขึ้น|ปรับราคาขึ้น|ปรับเพิ่ม|เพิ่มขึ้น"
REGEX_PRICE_CENTER = r"ราคากลาง|ส่วนกลาง|ไฟล์กลาง"
REGEX_WRONG = r"ผิด|แก้ไข|ขอโทษ|ซ้ำ"
REGEX_NO_STOCK = r"มีเร็วๆนี้|พักจำหน่าย|เดือนถัดไป|เดือนหน้า|หาย|ขาด|หมด|ไม่้มี|ไม่มี|ไม่พบ|ของหมด|สต๊อกหมด|ของไม่เข้า|หมดสินค้า|สินค้าหมด|หมดแล้ว|\bC[1-5]\b|\bC\.[1-5]\b"
REGEX_DELETE = r"ปิดกิจการ|ตัด|ลบออก|ลบสินค้า|เลิกขาย|ไม่ได้เก็บแล้ว|ยกเลิกสินค้า|เลิกจำหน่าย|ถอดสินค้า|ยกเลิก|นำรายการออก"
REGEX_PROMO = r"จัดโปี|จัดโปร|จัดโปรโมชั่น|พิเศษ|โปร|pro|PRO|โปรโมชั่น|ลดราคา|ส่วนลด|สมาชิก"
REGEX_NOT_PROMO = r"ยืนยัน|Normal|Nomal|ปกิ|ปกติ|ไม่มีโปร|หมดโปร|ไม่โปร"
REGEX_FREE = r"1\+1|1 เเถม 1|ซื้อ 2 จ่าย 1|แถม|ของแถม|ฟรี"

# ---------------------------
# 3. Apply Rules (Priority System)
# ---------------------------

# สร้าง Mask
mask_shop = df_actions["COMM1"].str.contains(REGEX_CHANGE_SHOP, na=False)
mask_change_gen = df_actions["COMM1"].str.contains(REGEX_CHANGE_KEYWORD, na=False)
mask_change_spec = df_actions["COMM1"].str.contains(REGEX_CHANGE_PRODUCT_SPECS, na=False)
mask_ignore = df_actions["COMM1"].str.contains(REGEX_IGNORE, na=False)

# =========================================================
# STEP 1: Focus หา "การเปลี่ยน" ก่อน (High Priority)
# =========================================================

# 1.1 เปลี่ยนทั้งสินค้าและร้าน
mask_both = mask_shop & mask_change_spec & ~mask_ignore
df_actions.loc[mask_both, "Action_Label"] = "เปลี่ยนทั้งสินค้าและร้าน"

# 1.2 เปลี่ยนร้าน (ใช้ isnull เพื่อไม่ทับข้อ 1.1)
df_actions.loc[mask_shop & df_actions["Action_Label"].isnull(), "Action_Label"] = "เปลี่ยนร้าน"

# 1.3 เปลี่ยนสินค้า
mask_product = (mask_change_gen | mask_change_spec) & ~mask_shop & ~mask_ignore
df_actions.loc[mask_product & df_actions["Action_Label"].isnull(), "Action_Label"] = "เปลี่ยนสินค้า"

# =========================================================
# STEP 2: เติมคำในช่องว่าง (Fill in the rest)
# =========================================================

def fill_label(regex, label_name):
    # เช็คว่าเจอ Regex และ Label ต้องยังว่างอยู่
    mask = df_actions["COMM1"].str.contains(regex, na=False) & df_actions["Action_Label"].isnull()
    df_actions.loc[mask, "Action_Label"] = label_name

# เรียงลำดับการเติม
fill_label(REGEX_ADD, "เพิ่มสินค้า")
fill_label(REGEX_PRICE_UP, "ปรับราคาเพิ่ม")
fill_label(REGEX_PRICE_DOWN, "ปรับราคาลง")
fill_label(REGEX_PRICE_CENTER, "ราคากลาง")
fill_label(REGEX_WRONG, "แจ้งผิดพลาด")
fill_label(REGEX_DELETE, "ลบสินค้า")
fill_label(REGEX_NO_STOCK, "ไม่มีสินค้า")

# --- Logic โปรโมชั่น ---
mask_remaining = df_actions["Action_Label"].isnull()
mask_p = df_actions["COMM1"].str.contains(REGEX_PROMO, na=False)
mask_np = df_actions["COMM1"].str.contains(REGEX_NOT_PROMO, na=False)
mask_f = df_actions["COMM1"].str.contains(REGEX_FREE, na=False)

# Combo (โปร+แถม)
df_actions.loc[mask_remaining & mask_p & mask_f, "Action_Label"] = "โปร+แถม"
df_actions.loc[mask_remaining & mask_np & mask_f, "Action_Label"] = "หมดโปร+แถม"

# Single (โปร / หมด / แถม)
mask_remaining = df_actions["Action_Label"].isnull()  # อัปเดต mask อีกรอบ
df_actions.loc[mask_remaining & mask_p, "Action_Label"] = "โปร"
df_actions.loc[mask_remaining & mask_np, "Action_Label"] = "หมดโปร"
df_actions.loc[mask_remaining & mask_f, "Action_Label"] = "แถม"

# =========================================================
# 4. สรุปผล
# =========================================================
print("จำนวน label จาก rule-based (New Logic):")
print(df_actions["Action_Label"].value_counts(dropna=False))

df_actions = df_actions.reset_index(drop=True)
df_actions.to_clipboard(index=False)

In [ ]:
import pandas as pd
import os
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# ==============================================================================
# PART 2: Machine Learning (Training with Excel & Prediction)
# ==============================================================================

print("--- เริ่มกระบวนการ Machine Learning ---")

# ใช้ตัวแปร df_actions จากขั้นตอนก่อนหน้า
if 'df_actions' not in locals():
    print("⚠️ ไม่พบตัวแปร df_actions กรุณารัน Code ส่วน Rule-based ก่อน")
    # df_actions = df.copy() # ปลลดล็อกบรรทัดนี้ถ้าคุณใช้ชื่อตัวแปร df

# ---------------------------------------------------------
# 1. รวบรวมข้อมูลสอน (Training Data Collection)
# ---------------------------------------------------------

# 1.1 ข้อมูลจาก Rule-based (เอาเฉพาะที่มั่นใจ และมีข้อความ)
df_train_rules = df_actions[df_actions["Action_Label"].notna() & (df_actions["COMM1"].str.strip() != "")][["COMM1", "Action_Label"]].copy()
print(f"-> จำนวนข้อมูลสอนจาก Rule-based: {len(df_train_rules)} แถว")

# 1.2 ข้อมูลจาก Excel ภายนอก (Manual Training Data)
EXTERNAL_FILE_NAME = "Training_Data.xlsx" 
df_train_excel = pd.DataFrame()

if os.path.exists(EXTERNAL_FILE_NAME):
    try:
        print(f"-> พบไฟล์ {EXTERNAL_FILE_NAME} กำลังอ่านข้อมูล...")
        # อ่านไฟล์ (Assumption: Col 0 = Text, Col 1 = Label)
        df_ext = pd.read_excel(EXTERNAL_FILE_NAME)
        df_ext = df_ext.iloc[:, 0:2] 
        df_ext.columns = ["COMM1", "Action_Label"]
        
        # Clean Data
        df_ext = df_ext.dropna()
        df_ext["COMM1"] = df_ext["COMM1"].astype(str)
        df_ext["Action_Label"] = df_ext["Action_Label"].astype(str)
        
        df_train_excel = df_ext
        print(f"-> เพิ่มข้อมูลสอนจาก Excel สำเร็จ: {len(df_train_excel)} แถว")
        
    except Exception as e:
        print(f"⚠️ เกิดข้อผิดพลาดในการอ่านไฟล์ Excel: {e}")
else:
    print(f"-> ไม่พบไฟล์ {EXTERNAL_FILE_NAME} (ใช้เฉพาะข้อมูลจาก Rule-based)")

# 1.3 รวมร่าง (Rule + Excel)
df_train_final = pd.concat([df_train_rules, df_train_excel], ignore_index=True)

# ---------------------------------------------------------
# 2. เริ่มกระบวนการ Train
# ---------------------------------------------------------

if len(df_train_final) < 10:
    print("⚠️ ข้อมูลสำหรับสอนรวมกันแล้วน้อยเกินไป (<10) ML อาจทำงานได้ไม่ดี")
else:
    print(f"-> รวมข้อมูลสอนทั้งหมด: {len(df_train_final)} แถว")
    
    X = df_train_final["COMM1"]
    y = df_train_final["Action_Label"]

    # แบ่งข้อมูลสอบ (Test Split)
    try:
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    except ValueError:
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # สร้าง Pipeline (TF-IDF + LinearSVC)
    model = Pipeline([
        ('tfidf', TfidfVectorizer(analyzer='char', ngram_range=(2, 5), min_df=1)), 
        ('clf', LinearSVC(class_weight='balanced', random_state=42))
    ])

    # Train
    model.fit(X_train, y_train)

    # Evaluate
    y_pred_test = model.predict(X_test)
    print("\n=== ผลการทดสอบความแม่นยำ (Rule + Excel) ===")
    print(classification_report(y_test, y_pred_test, zero_division=0))

    # ---------------------------------------------------------
    # 3. ทำนายข้อมูลจริง (Action_Label_2)
    # ---------------------------------------------------------
    print("กำลังทำนายผลลงใน Action_Label_2 ...")
    
    df_actions["Action_Label_2"] = None
    
    # สร้าง Mask สำหรับแถวที่มีข้อความ (ไม่ว่าง)
    mask_has_text = df_actions["COMM1"].str.strip() != ""

    if mask_has_text.sum() > 0:
        # ทำนาย
        predicted_values = model.predict(df_actions.loc[mask_has_text, "COMM1"])
        df_actions.loc[mask_has_text, "Action_Label_2"] = predicted_values
    else:
        print("ไม่พบข้อมูลคอมเมนต์ที่จะทำนายเพิ่ม")

    # เปรียบเทียบผลลัพธ์ (จุดที่ Rule พลาด แต่ AI เก็บตกได้)
    mask_new_prediction = df_actions["Action_Label"].isna() & df_actions["Action_Label_2"].notna()
    print(f"\nจำนวนแถวที่ Rule-based ทายไม่ได้ แต่ AI ทายให้ใหม่: {mask_new_prediction.sum()} แถว")
    
    if mask_new_prediction.sum() > 0:
        print("ตัวอย่างผลลัพธ์ที่ AI ช่วยเติมให้:")
        print(df_actions.loc[mask_new_prediction, ["COMM1", "Action_Label", "Action_Label_2"]].head(10))

# ==============================================================================
# ส่งออกผลลัพธ์สุดท้าย
# ==============================================================================
df_actions.to_clipboard(index=False)
print("Process Completed. Data copied to clipboard.")

In [ ]:
# df = df[['ผู้ดูแล','นับตัวซ้ำ','TOTAL_FLAGS','TYPE','ภาค','PROVINCE_NAME','กลุ่ม','GROUP', 'MONTH', 'YEAR',  'GROUP_KEY','TYPE_CODE', 
#              'CODE_7_DIGITS', 'COMMODITY_CODE', 
#              'DESCRIPTION','Band','Bandรายละเอียด','การบรรจุ','จำนวนต่อหน่วยบรรจุ','ปริมาณ','หน่วย', 'จำนวนสินค้า',
#              'DILLER_CODE','SHOP_NAME','ประเภทร้านค้า', 'จำนวนร้าน(ประเภทร้านค้า)', 'ขนาดร้านค้า',
#             'จำนวนร้าน(ขนาดร้านค้า)','ราคาที่นิยมเฉพาะร้าน', 'C_FLAG', 'G_FLAG','L_FLAG', 'N_FLAG', 'R_FLAG', 'NR_FLAG', 'P_FLAG', 'USERID',
#             'LAST_PRICE','ราคากลาง','AVG','LINK','COMPARE_PRICE','REL','COMM1', 'COMM2','AVG_ฐานนิยม','REL_ฐานนิยม','สถานะ']]


####  เรื่องไรดี

In [ ]:
# df = df_final[['ผู้ดูแล','นับตัวซ้ำ','TOTAL_FLAGS', 'TYPE','ภาค','COMMODITY_CODE', 'MONTH', 'YEAR', 'DILLER_CODE', 'GROUP_KEY',
#        'TYPE_CODE', 'AVG', 'ราคาที่นิยมเฉพาะร้าน', 'C_FLAG', 'G_FLAG',
#        'L_FLAG', 'N_FLAG', 'R_FLAG', 'NR_FLAG', 'LINK', 'COMPARE_PRICE',
#        'LAST_PRICE', 'REL', 'COMM1', 'COMM2', 'P_FLAG', 'USERID', 'SHOP_NAME',
#        'PROVINCE_NAME', 'DESCRIPTION','Band','Bandรายละเอียด','การบรรจุ','จำนวนต่อหน่วยบรรจุ','ปริมาณ','หน่วย', 'จำนวนสินค้า', 'CODE_7_DIGITS',
#        'ประเภทร้านค้า', 'จำนวนร้าน(ประเภทร้านค้า)', 'ขนาดร้านค้า',
#        'จำนวนร้าน(ขนาดร้านค้า)','สถานะ']]

df.to_clipboard()

# Excel_Condition2(df)


In [ ]:
dg = df.copy()
dg.columns

In [ ]:


# link1_df = data_mouthG[data_mouthG['LINK'] == 1].copy()
# link2_df = data_mouthG[data_mouthG['LINK'] == 2].copy()
# link3_df = data_mouthG[data_mouthG['LINK'] == 3].copy()

# # 2. Define the output file path
# output_file = "multi_sheet_dataG.xlsx"

# # # 3. Use ExcelWriter to write multiple DataFrames to separate sheets
# # with pd.ExcelWriter(output_file, engine='xlsxwriter') as writer:
# #     # Sheet 1: data_mouthG (Your merged data)
# #     data_mouthG.to_excel(writer, sheet_name='ข้อมูลดิบ', index=False)
# #     allcenter_clean.to_excel(writer, sheet_name='ราคากลาง', index=False)
# #     link1_df.to_excel(writer, sheet_name='link1', index=False)
# #     link2_df.to_excel(writer, sheet_name='link2', index=False)
# #     link3_df.to_excel(writer, sheet_name='link3', index=False)
    
# # print(f"✅ Successfully saved DataFrames to '{output_file}' with 5 sheets.")



In [ ]:
# df
# x
# df.head(10)

In [ ]:
# data_mouthG = df.copy()
# link1_df = data_mouthG[data_mouthG['LINK'] == 1].copy()
# link2_df = data_mouthG[data_mouthG['LINK'] == 2].copy()
# link3_df = data_mouthG[data_mouthG['LINK'] == 3].copy()

# output_file = "multi_sheet_dataG.xlsx"

# # Excel_Condition2(data_mouthG,"Series2.xlsx")

# from openpyxl import Workbook

# data_mouthG = df.copy()
# link1_df = data_mouthG[data_mouthG['LINK'] == 1].copy()
# link2_df = data_mouthG[data_mouthG['LINK'] == 2].copy()
# link3_df = data_mouthG[data_mouthG['LINK'] == 3].copy()

# output_file = "multi_sheet_dataG.xlsx"

# wb = Workbook()

# # ลบชีตเริ่มต้น (Sheet) ทิ้งก่อน
# default_ws = wb.active
# wb.remove(default_ws)

# Excel_Condition3(data_mouthG,     wb, 'ข้อมูลดิบ')
# Excel_Condition3(allcenter_clean, wb, 'ราคากลาง')
# Excel_Condition3(link1_df,        wb, 'link1')
# Excel_Condition3(link2_df,        wb, 'link2')
# Excel_Condition3(link3_df,        wb, 'link3')

# wb.save(output_file)
# print(f"✅ Successfully saved DataFrames to '{output_file}' with 5 sheets.")


In [ ]:
# df

In [ ]:
# data_mouthG = df.copy()

# data_mouthG['ราคาต่อหน่วย'] = data_mouthG['AVG']/(data_mouthG['จำนวนต่อหน่วยบรรจุ']*data_mouthG['ปริมาณ'])




# # data_mouthG = df.copy()
# link1_df = data_mouthG[data_mouthG['LINK'] == 1].copy()
# link2_df = data_mouthG[data_mouthG['LINK'] == 2].copy()
# link3_df = data_mouthG[data_mouthG['LINK'] == 3].copy()

# output_file = "multi_sheet_dataG.xlsx"

# wb = Workbook()

# # ลบชีตเริ่มต้น (Sheet) ทิ้งก่อน
# default_ws = wb.active
# wb.remove(default_ws)

# Excel_Condition3(data_mouthG,     wb, 'ข้อมูลดิบ')
# Excel_Condition3(allcenter_clean, wb, 'ราคากลาง')
# Excel_Condition3(link1_df,        wb, 'link1')
# Excel_Condition3(link2_df,        wb, 'link2')
# Excel_Condition3(link3_df,        wb, 'link3')

# wb.save(output_file)
# print(f"✅ Successfully saved DataFrames to '{output_file}' with 5 sheets.")

# data_mouthG



In [ ]:
# data_mouthG

In [ ]:
# detect_df = data_mouthG.copy()
# # detect_df = detect_df[(detect_df['AVG'] > 0) | (~detect_df['AVG'].isna())] #*****ค่าหาย
# detect_df = detect_df[detect_df['AVG']!=0] # กำจัดค่า 0
# detect_df = detect_df[~detect_df['AVG'].isna()] # กำจัดค่าว่าง
# detect_df[detect_df['DESCRIPTION']=='น้ำส้ม (เขียวหวาน)  บรรจุกล่อง \น้ำหนักสุทธิ 200 กรัม\  ตรามาลี']['AVG']

### คัดสรรค์

In [ ]:
DataDetection = shop_popular.copy()

# #เลือกตัวไม่ส่งไป
#ไม่คำนวณ
Delist  = [1112101,1112102,1112103,1112201,1112206,1112208,1112209,1121102,1121201,1121202,1121206,1122101,1122201,1122203,1123105,1123201,1123205,1123206,1123207,1123208,1123209,1123303,1123308,1123405,1123502,1132007,1132008,1141120,1141202,1141206,1141209,1142109,1142201,1142203,1151103,1151105,1151108,1152102,1152103,1152106,1152208,1152214,1152215,1160003,1210007,1210009,1210013,1220003,1220005,1421102,2111001,2113006,2113007,2113011,2113022,2113044,2113302,2113304,2113306,2113322,2113344,2113366,2114111,2114401,2114402,2114411,2114441,2115001,2115500,2115504,2115550,2121001,2121002,2123005,2123007,2124004,2125002,2125004,2131001,2131102,2131222,2132004,2132022,2132202,2132203,2132222,2133002,2141001,2141003,2141006,2142001,2142002,2211001,2211002,2213001,2213002,2221001,2221002,2222001,2223001,2223002,2232001,2233003,2241002,2242001,3130004,3130009,3220003,3240001,3240003,3300001,3300002,3300003,3300007,3300008,3400001,3400003,3400004,3400005,3400007,3400008,3510004,3520006,3520007,3520012,3520013,3600002,3600006,3600007,3600015,4111004,4111005,4111006,4111007,4111009,4111010,4111013,4111015,4111017,4112001,4121101,4121102,4121402,4121501,4210006,4210018,4210025,4210026,4220001,4220002,4220003,4220004,4220005,4222001,4300001,5110005,5110007,5110009,5120001,5120003,5120004,5120005,5120008,5120009,5120010,5212003,5220002,5220012,5230003,5230006,5230010,5330002,5410001,5410004,5410005,5410007,5420001,5420004,5420005,6110003,6110004,6110005,6110006,6121001,6122002,6123001,6123002,6123003,6124001,6124002,6124003,6124004,6124005,6125002,6130001,6130005,6130008,6130009,6140002,6140003,6140004,6140005,6221101,6221102,6221104,6221105,6221201,6221202,6221204,6221205]
DataDetection['CODE_7_DIGITS'] = DataDetection['CODE_7_DIGITS'].astype(int)

# งดวิเคราะห์
DataDetection.loc[DataDetection['CODE_7_DIGITS'].isin(Delist), 'คัดสรรค์'] = 'ไม่คำนวณ'
DataDetection.loc[DataDetection['CODE_7_DIGITS'].astype(str).str.startswith("4111"), 'คัดสรรค์'] = 'ยกเว้นการวิเคราะห์' #พี่จา
DataDetection.loc[DataDetection['CODE_7_DIGITS'].astype(str).str.startswith("412"), 'คัดสรรค์'] = 'ยกเว้นการวิเคราะห์' #พี่ปุ้ม
DataDetection.loc[DataDetection['CODE_7_DIGITS'].astype(str).str.startswith("4221"), 'คัดสรรค์'] = 'ยกเว้นการวิเคราะห์' #พี่ปุ้ม
DataDetection.loc[DataDetection['CODE_7_DIGITS'].isin([1250001, 1250002, 1240008]), 'คัดสรรค์'] = 'ยกเว้นการวิเคราะห์' #พี่จา
DataDetection.loc[DataDetection['CODE_7_DIGITS'].isin([5110001,5110008,5120002,6110002,6110005,6110007,6124005,6150006,6211001, 6211002, 6212001,6212002,6213001,6213002,6214001,6214002]), 'คัดสรรค์'] = 'ยกเว้นการวิเคราะห์' #พี่ปุ้ม

DataDetection.loc[DataDetection['CODE_7_DIGITS'].isin([1160010]), 'คัดสรรค์'] = 'ยืนยันความถูกต้อง' #เป๋า
DataDetection.loc[DataDetection['ประเภทร้านค้า']=='CJ EXPRESS', 'คัดสรรค์'] = 'ยืนยันความถูกต้อง' #เป๋า
DataDetection.loc[DataDetection['COMMODITY_CODE'].isin(['1112210211011601']), 'คัดสรรค์'] = 'ยกเว้นการวิเคราะห์'
# DataDetection.loc[DataDetection['สถานะ']=='น่าสงสัยกลุ่ม', 'คัดสรรค์'] = 'ยกเว้นการวิเคราะห์'

# ราคากลาง
DataDetection.loc[DataDetection['CODE_7_DIGITS'].isin([3120001, 3210001, 4300002, 4300003, 5110010, 5110011, 5110013, 5210007, 5120012, 5230008, 5310001, 5310002, 5320001, 5320002, 5330001, 5330004, 5410001, 5410002, 6124006, 6150004, 6150005, 6221103]), 'คัดสรรค์'] = 'ยกเว้นการวิเคราะห์'

# ยืนยันว่าถูกต้อง
DataDetection['KEY'] = DataDetection['COMMODITY_CODE'].astype(str) + "_" + DataDetection['DILLER_CODE'].astype(str)+ "_" + DataDetection['AVG'].fillna(0).astype(float).astype(int).astype(str)
dfr['KEY'] = dfr['COMMODITY_CODE'].astype(str) + "_" + dfr['DILLER_CODE'].astype(str)+ "_" + dfr['AVG'].fillna(0).astype(float).astype(int).astype(str)
DataDetection.loc[DataDetection['KEY'].isin(dfr['KEY']), 'คัดสรรค์'] = 'ยืนยันความถูกต้อง'
DataDetection = DataDetection.drop(columns=['KEY'])

DataDetection.loc[DataDetection['CODE_7_DIGITS'].isin([1160015]), 'คัดสรรค์'] = 'ยืนยันความถูกต้อง' #เป๋า
# DataDetection.loc[DataDetection['สถานะ']=='ผิดปกติ + ร้านใหญ่ร้านเดียว', 'คัดสรรค์'] = 'บังคับถูกต้อง' #พี่จา
# DataDetection.loc[DataDetection['สถานะ']=='ผิดปกติ + ร้านใหญ่2ร้านราคาต่างกัน', 'คัดสรรค์'] = 'บังคับถูกต้อง' #พี่จา

# data_mouthG = df.copy()
link1_df = DataDetection[DataDetection['LINK'] == 1].copy()
link2_df = DataDetection[DataDetection['LINK'] == 2].copy()
link3_df = DataDetection[DataDetection['LINK'] == 3].copy()

output_file = filename + "_multi_sheet_dataG.xlsx"

wb = Workbook()

# ลบชีตเริ่มต้น (Sheet) ทิ้งก่อน
default_ws = wb.active
wb.remove(default_ws)

Excel_Condition3(DataDetection,     wb, 'ข้อมูลดิบ')
Excel_Condition3(allcenter_clean, wb, 'ราคากลาง')
Excel_Condition3(link1_df,        wb, 'link1')
Excel_Condition3(link2_df,        wb, 'link2')
Excel_Condition3(link3_df,        wb, 'link3')

wb.save(output_file)
print(f"✅ Successfully saved DataFrames to '{output_file}' with 5 sheets.")


In [ ]:
SR = dfs_ .copy()
MF = DataDetection.copy()
print(DataDetection.columns)
SR.columns
SR = SR[['COMMODITY_CODE', 'DILLER_CODE', 'CountAVG',
       'AVGtail_stable','AVG_Max', 'AVG_Min', 'AVG_maxdiff', 'AVG_mindiff', 
       'Rel_Max', 'Rel_Min', 
       'Reltrail_eq100']].copy()
# SR
SR['COMMODITY_CODE'] = SR['COMMODITY_CODE'].astype(str).str.zfill(16)   
SR['DILLER_CODE'] = SR['DILLER_CODE'].astype(str).str.zfill(10)

SR = SR.drop_duplicates(subset=['COMMODITY_CODE','DILLER_CODE']).reset_index(drop=True)
MF = MF.merge(SR,on=['COMMODITY_CODE','DILLER_CODE'], how='left')

# len(MF)
MF['ต่อเนื่องเกิน12เดือน'] = np.where(
    MF['AVGtail_stable'] >= 12, MF['AVGtail_stable'], np.nan
)

# 3) คอลัมน์ “ราคาเกินช่วงสูงต่ำ”
cond1 = (MF['AVG'] > MF['AVG_Max']) | (MF['AVG'] < MF['AVG_Min'])
MF['ราคาเกินช่วงสูงต่ำ'] = pd.Series(
    np.where(cond1, 'เกิน', None), index=MF.index, dtype='object'  # ใช้ object จะได้ None ไม่ใช่ <NA>
)


# # 4) คอลัมน์ “ราคากระโดดไกล”
# cond2 = ((MF['AVG'] - MF['COMPARE_PRICE']) > MF['AVG_maxdiff']) | \
#         ((MF['AVG'] - MF['COMPARE_PRICE']) < MF['AVG_mindiff'])
# MF['ราคากระโดดไกล'] = pd.Series(
#     np.where(cond2, 'เกิน', None), index=MF.index, dtype='object'
# )
#  da[['ราคาปกติ', 'ราคาโปร']].min(axis=1)
MF['MAX_DIFF'] = MF[['AVG_maxdiff', 'AVG_mindiff']].max(axis=1)
cond2 = abs(MF['AVG'] - MF['COMPARE_PRICE']) > abs(MF['MAX_DIFF'])
MF['ราคากระโดดไกล'] = np.where(cond2, 'เกิน', None)

# 5) สำคัญ: แปลงค่าที่หายไปทั้งหมดให้เป็น None (openpyxl เขียนได้)
MF = MF.astype(object).where(pd.notna(MF), None)
# MF['ราคาเกินช่วงสูงต่ำ'] = "" 
# MF['ราคากระโดดไกล'] = ""

MF['C diff A'] = MF['AVG'] - MF['COMPARE_PRICE']
MF.loc[MF['C diff A'] == 0, 'ราคากระโดดไกล'] = ''
# MF
# Excel_Condition(MF, filename +"_ตรวจราคา__final_Series1.xlsx")
# TestMF = MF.loc[:500].copy()
TestMF = MF.copy()
print(TestMF.columns)

# 1) แทนค่าตามเงื่อนไขเบื้องต้น
TestMF['นับตัวซ้ำ_']      = TestMF['นับตัวซ้ำ'].replace({1: '', 2: 'ซ้ำ2ตัว'})

TestMF.loc[TestMF['ผลต่างราคากับราคากลาง'] > 0, 'แสดงผล'] = 'ราคาผิด'
# TestMF[[TestMF['ผลต่างราคากับราคากลาง']>0]],'แสดงผล' == 'ราคาผิด'

# TestMF['แสดงผล']        = TestMF['แสดงผล'].replace({'ราคาถูกต้อง': ''})
TestMF['ราคากระโดดไกล'] = TestMF['ราคากระโดดไกล'].replace({'เกิน': 'ไกล'})

# ---------- สรุปผลกระโดด ----------
# 2) รวมข้อความแบบปลอดภัย (ทุกคอลัมน์ -> str, NaN -> '')
cols_jump = ['ราคากระโดดไกล', 'COMMODITY_CODE', 'ขนาดร้านค้า',
             'COMPARE_PRICE', 'AVG', 'นับตัวซ้ำ_', 'แสดงผล']

TestMF['สรุปผลกระโดด'] = (
    TestMF[cols_jump]
    .astype(str)        # ทำให้แน่ใจว่าเป็น str ทั้งหมด (ถ้ามี NaN จะกลายเป็น "nan")
    .replace('nan', '') # เคลียร์ "nan" ที่เกิดจาก NaN
    .agg(''.join, axis=1)
)

# เคลียร์เคสพิเศษ: REL == 100 ให้เป็นค่าว่าง
TestMF.loc[TestMF['REL'] == 100, 'สรุปผลกระโดด'] = ''

# 3) คำนวณ mask "หลัง" ได้คอลัมน์สรุปผลจริง
mask_jump = TestMF['สรุปผลกระโดด'].ne('')

# 4) นับจำนวนซ้ำเฉพาะแถวที่ไม่ว่าง
counts_jump = TestMF.loc[mask_jump].groupby('สรุปผลกระโดด')['สรุปผลกระโดด'].transform('size')

# เติมกลับเข้า Series ให้ index ตรงกับ DataFrame (ที่ไม่ว่างได้ค่า counts, ที่เหลือ NaN)
TestMF['นับ_สรุปผลกระโดด'] = 0
TestMF.loc[mask_jump, 'นับ_สรุปผลกระโดด'] = counts_jump
TestMF['นับ_สรุปผลกระโดด'] = TestMF['นับ_สรุปผลกระโดด'].fillna(0).astype(int)


TestMF.loc[TestMF['ราคากระโดดไกล'].isna(), 'สรุปผลกระโดด'] = ''

# ถ้านับได้ 2 ให้เคลียร์ข้อความ
TestMF.loc[TestMF['นับ_สรุปผลกระโดด'] == 2, 'สรุปผลกระโดด'] = ''

TestMF.loc[TestMF['นับ_สรุปผลกระโดด'].isna(), 'สรุปผลกระโดด'] = ''
TestMF.loc[TestMF['นับ_สรุปผลกระโดด'] == 2, 'สรุปผลกระโดด'] = ''
TestMF.loc[TestMF['นับ_สรุปผลกระโดด'] >= 2, 'สรุปผลกระโดด'] = ''
TestMF.loc[TestMF['นับ_สรุปผลกระโดด'] >= 2, 'นับ_สรุปผลกระโดด'] = ''

# ---------- สรุปผลเกินขอบ ----------
cols_edge = ['ราคาเกินช่วงสูงต่ำ', 'COMMODITY_CODE', 'ขนาดร้านค้า',
             'COMPARE_PRICE', 'AVG', 'นับตัวซ้ำ_', 'แสดงผล']

TestMF['สรุปผลเกินขอบ'] = (
    TestMF[cols_edge]
    .astype(str)
    .replace('nan', '')
    .agg(''.join, axis=1)
)

# เคลียร์เคสพิเศษ: REL == 100 ให้เป็นค่าว่าง
TestMF.loc[TestMF['REL'] == 100, 'สรุปผลเกินขอบ'] = ''

mask_edge = TestMF['สรุปผลเกินขอบ'].ne('')

counts_edge = TestMF.loc[mask_edge].groupby('สรุปผลเกินขอบ')['สรุปผลเกินขอบ'].transform('size')

TestMF['นับ_สรุปผลเกินขอบ'] = 0
TestMF.loc[mask_edge, 'นับ_สรุปผลเกินขอบ'] = counts_edge
TestMF['นับ_สรุปผลเกินขอบ'] = TestMF['นับ_สรุปผลเกินขอบ'].fillna(0).astype(int)

TestMF.loc[TestMF['ราคาเกินช่วงสูงต่ำ'].isna(), 'สรุปผลเกินขอบ'] = ''
TestMF.loc[TestMF['นับ_สรุปผลเกินขอบ'] == 2, 'สรุปผลเกินขอบ'] = ''
TestMF.loc[TestMF['นับ_สรุปผลเกินขอบ'] >= 2, 'สรุปผลเกินขอบ'] = ''
TestMF.loc[TestMF['นับ_สรุปผลเกินขอบ'] >= 2, 'นับ_สรุปผลเกินขอบ'] = ''

# ---------- Clear NONE ----------

# ---------- RELเกิน50 ----------
# ทำให้แน่ใจว่า REL เป็นตัวเลข
TestMF['REL'] = pd.to_numeric(TestMF['REL'], errors='coerce')

# เริ่มด้วยค่าว่าง
TestMF['สรุปผลREL'] = ''

# mask นอกช่วง 90–110
mask_edge = (TestMF['REL'] <= 50) | (TestMF['REL'] >= 150)

# ต่อสตริง: ใส่วงเล็บและกัน NaN ด้วย fillna('')
TestMF.loc[mask_edge, 'สรุปผลREL'] = (
    'REL'
    + TestMF['COMMODITY_CODE'].fillna('').astype(str)
    + TestMF['ขนาดร้านค้า'].fillna('').astype(str)
    + TestMF['COMPARE_PRICE'].fillna('').astype(str)
    + TestMF['AVG'].fillna('').astype(str)
)

vc = TestMF.loc[mask_edge, 'สรุปผลREL'].value_counts()
TestMF['นับ_สรุปผลREL'] = 0
TestMF.loc[mask_edge, 'นับ_สรุปผลREL'] = TestMF.loc[mask_edge, 'สรุปผลREL'].map(vc).astype(int)
TestMF.loc[TestMF['นับ_สรุปผลREL'] >= 2, 'สรุปผลREL'] = ''

# ---------- รวมสรุปผล ----------
TestMF['สรุปผล'] = (
    TestMF['สรุปผลกระโดด'].fillna('') +
    TestMF['สรุปผลเกินขอบ'].fillna('')+
    TestMF['สรุปผลREL'].fillna('')
)


# กรณีต้องเช็ค AVG|REL ว่างหรือศูนย์
mask_avg_blank_or_zero = TestMF['AVG'].isna() | (pd.to_numeric(TestMF['AVG'], errors='coerce').fillna(np.nan).eq(0))
TestMF.loc[(pd.to_numeric(TestMF['AVG'], errors='coerce').fillna(0) == 0) & mask_avg_blank_or_zero,'สรุปผล'] = 'ข้อมูลไม่ครบ'

mask_avg_blank_or_zero = TestMF['REL'].isna() | (pd.to_numeric(TestMF['REL'], errors='coerce').fillna(np.nan).eq(0))
TestMF.loc[(pd.to_numeric(TestMF['REL'], errors='coerce').fillna(0) == 0) & mask_avg_blank_or_zero,'สรุปผล'] = 'ข้อมูลไม่ครบ'





# ---------- AVG ว่าง AVG 0 REL ว่าง REL 0 ----------
s = TestMF['สรุปผล'].astype('string')

m_far  = s.str.startswith('ไกล', na=False)
m_over = s.str.contains('เกิน',  na=False)
m_rel  = s.str.contains('REL',   na=False)
m_emtry  = s.str.contains('ข้อมูลไม่ครบ',   na=False)

out = (
    np.where(m_far,  'กระโดดไกล',  '') + ' ' +
    np.where(m_over, 'เกินขอบเขต', '') + ' ' +
    np.where(m_rel,  'REL',        '') + ' ' +
    np.where(m_emtry,  'ข้อมูลไม่ครบ',        '')
)

# รวมคำ, ตัดช่องว่างซ้ำ/หัว-ท้าย และแทนค่าว่างเป็น NaN
TestMF['สรุปผล_จัดหมวด'] = (
    pd.Series(out, index=TestMF.index).str.split().str.join(' ').replace('', np.nan)
)


TestMF.loc[
    (TestMF['สถานะ'].isin(['specเดียว', 'ร้านใหญ่2ร้านราคาต่างกัน', 'ร้านใหญ่ร้านเดียว'])) & 
    (TestMF['สรุปผล_จัดหมวด'].notna()),
    'ผลลัพธ์'
] = 'ผิด'

TestMF.loc[TestMF['สถานะ']=="ผิดปกติ + ร้านใหญ่2ร้านราคาต่างกัน",'ผลลัพธ์'] = "ผิด"


TestMF.loc[
    (
        (TestMF['จำนวนร้าน(ประเภทร้านค้า)'] >= 3) & 
        (TestMF['สถานะ'].isna()) &
        (TestMF['สถานะ'].astype(str).str.strip() == '') &  # ตรวจสอบว่าไม่ว่างเปล่า
        (TestMF['สรุปผล_จัดหมวด'] == 'เกินขอบเขต')
    ),
    'ผลลัพธ์'
] = ''
TestMF.loc[
    (
        (TestMF['จำนวนร้าน(ประเภทร้านค้า)'] < 3) & 
        (TestMF['สถานะ'].notna()) &
        (TestMF['สถานะ'].astype(str).str.strip() != '') &
        (TestMF['สรุปผล_จัดหมวด'] == 'เกินขอบเขต')
    ), 
    'ผลลัพธ์'
] = 'ผิด'

s = TestMF['สรุปผล_จัดหมวด'].astype('string')
m_rel = s.str.contains('REL', na=False)
TestMF.loc[m_rel, 'ผลลัพธ์'] = 'ผิด'


TestMF.loc[
    (
        ((TestMF['สถานะ'] == 'ผิดปกติ') |  (TestMF['สถานะ'] == 'น่าสงสัย'))&
        ((TestMF['สรุปผล_จัดหมวด'].notna()))
    ), 
    'ผลลัพธ์'
] = 'ผิด'

TestMF.loc[TestMF['สรุปผล_จัดหมวด']=="ข้อมูลไม่ครบ",'ผลลัพธ์'] = "ข้อมูลไม่ครบ"

    
TestMF.loc[
    (
        (TestMF['ประเภทร้านค้า'] != 'ร้านอื่นๆ')&
        (TestMF['CountAVG'] <= 2)&
        (TestMF['สรุปผล_จัดหมวด'].isna()) &
        (TestMF['สถานะ'] == 'ผิดปกติ')
    ), 
    'ผลลัพธ์'
] = 'ผิด'

TestMF.loc[
    (
        (TestMF['ประเภทร้านค้า'] != 'ร้านอื่นๆ')&
        (TestMF['จำนวนร้าน(ประเภทร้านค้า)']>=3) &
        (TestMF['สรุปผล_จัดหมวด'].isna()) &
        ((TestMF['สถานะ'] == 'ผิดปกติ') |  (TestMF['สถานะ'] == 'น่าสงสัย'))
    ), 
    'ผลลัพธ์'
] = 'มีโอกาสผิด'





# เคลียร์เมื่อ "คัดสรรค์" = ยืนยันความถูกต้อง
TestMF.loc[TestMF['คัดสรรค์'].eq('ยืนยันความถูกต้อง'), 'ผลลัพธ์'] = ''

TestMF.loc[
    (TestMF['ประเภทร้านค้า'].eq('CJ EXPRESS')) &
    (TestMF['สถานะ'].astype(str).str.contains('ผิดปกติ|น่าสงสัย', na=False)),
    'ผลลัพธ์'
] = 'มีโอกาสผิด CJ'


TestMF.loc[
    (
        (TestMF['AVGtail_stable'] >= 12)&
        (TestMF['ประเภทร้านค้า']=='ร้านอื่นๆ') &
        (TestMF['สรุปผล_จัดหมวด'].isna()) &
        (TestMF['REL']==100) &
        ((TestMF['สถานะ'] == 'ผิดปกติ') |  (TestMF['สถานะ'] == 'น่าสงสัย'))
    ), 
    'ผลลัพธ์'
] = 'ให้ติดตามสินค้า'

# ให้คอลัมน์เป็นชนิดสตริงก่อน และแทน NaN ด้วยค่าว่างสำหรับฝั่งผลลัพธ์
TestMF['ผลลัพธ์'] = TestMF.get('ผลลัพธ์', '').astype('string').fillna('')

# ต่อสตริงจากคอลัมน์ 'นับตัวซ้ำ' โดยให้ NaN กลายเป็นค่าว่าง (ไม่ใส่ 0)
TestMF['ผลลัพธ์'] = TestMF['ผลลัพธ์'].str.cat(
    TestMF['นับตัวซ้ำ_'].astype('string'),
    na_rep=''   # ถ้า 'นับตัวซ้ำ' เป็น NaN จะต่อด้วยค่าว่าง
)


TestMF.loc[TestMF['สรุปผล_จัดหมวด']=="ข้อมูลไม่ครบ",'ผลลัพธ์'] = "ข้อมูลไม่ครบ"

TestMF.loc[TestMF['ผลต่างราคากับราคากลาง']>0,'ผลลัพธ์'] = "ราคากลางผิด"
TestMF.loc[(TestMF['ราคากลาง']>0)&(TestMF['AVG'].isna()),'ผลลัพธ์'] = "ราคากลางผิด"

# 'LINK', 'LAST_PRICE', 'COMPARE_PRICE', 'AVG', 'ราคาแนะนำ', 'REL', 

TestMF = TestMF[['ผู้ดูแล', 'นับตัวซ้ำ', 'TOTAL_FLAGS', 'TYPE', 'ภาค', 'PROVINCE_NAME',
       'กลุ่ม', 'GROUP', 'MONTH', 'YEAR', 'GROUP_KEY', 'TYPE_CODE',
       'CODE_7_DIGITS', 'COMMODITY_CODE', 'DESCRIPTION', 'Band',
       'Bandรายละเอียด', 'การบรรจุ', 'จำนวนต่อหน่วยบรรจุ', 'ปริมาณ', 'หน่วย',
       'จำนวนสินค้า', 'Rank', 'DILLER_CODE', 'SHOP_NAME', 'ประเภทร้านค้า',
       'จำนวนร้าน(ประเภทร้านค้า)', 'ขนาดร้านค้า', 'จำนวนร้าน(ขนาดร้านค้า)',
       'ราคาที่นิยมเฉพาะร้าน', 'C_FLAG', 'G_FLAG', 'L_FLAG', 'N_FLAG',
       'R_FLAG', 'NR_FLAG', 'P_FLAG', 'USERID', 'LAST_PRICE', 'ราคากลาง',
       'AVG', 'LINK', 'COMPARE_PRICE', 'REL', 'COMM1', 'COMM2', 'AVG_ฐานนิยม',
       'REL_ฐานนิยม', 'Action_Label', 'ผลต่างราคากับราคากลาง',
       'ราคาต่อหน่วย', 'AVG_ฐานนิยม_ใหม่', 'ผลต่างAVG', 'REL_ฐานนิยม_ใหม่',
       'ผลต่างREL', 'ผล', 'Z-Score_ทั้งสินค้า', 'Z-Score_เฉพาะแหล่งเดียวกัน',
       'Sum_Z-Score', 'Report>=2', 'จำนวนราคาเหมือน', 
       'รายงานจังหวัดที่ขาดส่ง', 'คัดสรรค์',
       'AVG_Max', 'AVG_Min', 'AVG_maxdiff', 'AVG_mindiff',
       'Rel_Max', 'Rel_Min', 'Reltrail_eq100', 'ต่อเนื่องเกิน12เดือน',
        'MAX_DIFF', 'C diff A',
       'นับตัวซ้ำ_', 'ราคากระโดดไกล', 'ราคาเกินช่วงสูงต่ำ','CountAVG','AVGtail_stable',
       'สรุปผลกระโดด','นับ_สรุปผลกระโดด','สรุปผลเกินขอบ','นับ_สรุปผลเกินขอบ','สรุปผลREL','นับ_สรุปผลREL','สรุปผล','ผลลัพธ์','สถานะ','สรุปผล_จัดหมวด'
            ]]


# TestMF = TestMF[['ใช้คำนวณ', 'MONTH', 'YEAR', 'TYPE', 'C_FLAG', 'G_FLAG',
#        'L_FLAG', 'N_FLAG', 'R_FLAG', 'NR_FLAG', 'CODE_7_DIGITS',
#        'COMMODITY_CODE',  'TYPE_CODE', 'ข้อความ0', 'ข้อความ1',
#        'ข้อความ2', 'DESCRIPTION', 'จำนวนสินค้า', 'Rank', 'ขนาดร้านค้า',
#        'จำนวนร้าน(ขนาดร้านค้า)','DILLER_CODE', 'SHOP_NAME', 'ประเภทร้านค้า',
#        'จำนวนร้าน(ประเภทร้านค้า)', 'PROVINCE_NAME', '[ภาค]', 'จังหวัด',
#        'นับรายการเก็บต่อจังหวัด',
#         'LAST_PRICE',  'AVG','LINK', 'COMPARE_PRICE', 'REL', 'ราคาแนะนำ', 'ราคากลาง','แสดงผล',
#        'ราคาที่นิยมเฉพาะร้าน',
#        'ผลต่าง_ราคาที่นิยมเฉพาะร้าน', 'ฐานนิยม(ในสินค้า)',
#        'ฐานนิยม(ในขนาดร้านค้า)', 'ฐานนิยม(ในประเภทร้านค้า)', 'ฐานนิยม(รวมค่า)',
#        'ราคาอยู่ในช่วง', 'Z-Score_ทั้งสินค้า', 'Z-Score_เฉพาะแหล่งเดียวกัน',
#        'Sum_Z-Score', 'Report Z >=2',  'ตีความComment1', 'COMM1',
#        'COMM2', 'check_max_ฐานนิยม(ในประเภทร้านค้า)', 'max', 'min',
#        'รายงานจังหวัดที่ขาดส่ง', 'ราคาปกติ', 'ราคาโปร', 'ภาวะ', 
#        'นับการการแก้ไขตามจังหวัด', 'คัดสรรค์', 'temp_diff',
#        'AVG_Max', 'AVG_Min', 'AVG_maxdiff', 'AVG_mindiff',
#        'Rel_Max', 'Rel_Min', 'Re
# ltrail_eq100', 'ต่อเนื่องเกิน12เดือน',
#         'MAX_DIFF', 'C diff A',
#        'นับตัวซ้ำ', 'ราคากระโดดไกล', 'ราคาเกินช่วงสูงต่ำ','CountAVG','AVGtail_stable',
#        'สรุปผลกระโดด','นับ_สรุปผลกระโดด','สรุปผลเกินขอบ','นับ_สรุปผลเกินขอบ','สรุปผลREL','นับ_สรุปผลREL','สรุปผล','ผลลัพธ์','สถานะ','สรุปผล_จัดหมวด']]
# # 456987

    #   'สรุปผลกระโดด','นับ_สรุปผลกระโดด','สรุปผลเกินขอบ','นับ_สรุปผลเกินขอบ','สรุปผลREL','นับ_สรุปผลREL','สรุปผล','สรุปผล_จัดหมวด','ผลลัพธ์']]



Excel_Condition2(TestMF, filename +"_ตรวจราคา__final.xlsx")


In [ ]:
dff = pd.read_excel('cpig_month_2568-11_ตรวจราคา__final.xlsx')

In [ ]:
myDet = dff.copy()

myDet["COMMODITY_CODE"] = myDet["COMMODITY_CODE"].astype(str).str.zfill(16)
myDet["DILLER_CODE"] = myDet["DILLER_CODE"].astype(str).str.zfill(10)
myDet["CODE_7_DIGITS"] = myDet["COMMODITY_CODE"].astype(str).str[:7]

myDet["CODE7"] = myDet["COMMODITY_CODE"].str[0:7]
myDet["CODE9"] = myDet["COMMODITY_CODE"].str[7:16]
myDet = myDet[["G_FLAG","L_FLAG","N_FLAG","CODE7","CODE9","COMMODITY_CODE","DESCRIPTION","DILLER_CODE","SHOP_NAME","ภาค","GROUP","ประเภทร้านค้า","LAST_PRICE","AVG","LINK","COMPARE_PRICE","REL","COMM1","COMM2","ราคากลาง","นับตัวซ้ำ","CountAVG","AVGtail_stable","ผลลัพธ์"]]
myDet.to_excel('Test.xlsx',index=False)

Excel_Condition2(myDet, filename +"_final_Detection.xlsx")



In [ ]:
myDet.info()

In [ ]:
# # แก้้
# result_df1 = result_df.copy()


# result_df1["Sum_group"] = result_df1.groupby(["DESCRIPTION"])['AVG'].transform("sum")
# result_df1["percentage"] = (result_df1["AVG"]/result_df1["Sum_group"]*100).round(2)
# result_df1 = result_df1.reset_index(drop=True)
# result_df1.loc[(result_df1['จำนวนสินค้า'] == 3) & ((result_df1['percentage']<19) | (result_df1['percentage']>51)), 'สถานะพิเศษ'] = 'ผิดปกติ3'
# result_df1.loc[(result_df1['จำนวนสินค้า'] == 4) & ((result_df1['percentage']<14) | (result_df1['percentage']>41)), 'สถานะพิเศษ'] = 'ผิดปกติ4'
# result_df1.loc[(result_df1['จำนวนสินค้า'] == 5) & ((result_df1['percentage']<12) | (result_df1['percentage']>32)), 'สถานะพิเศษ'] = 'ผิดปกติ5'

# result_df1.loc[(result_df1['สถานะพิเศษ'] == 'ผิดปกติ3') & (result_df1['Z-Score_ทั้งสินค้า']>1) , 'สถานะ_'] = 'ผิดปกติ'
# result_df1.loc[(result_df1['สถานะพิเศษ'] == 'ผิดปกติ4') & (result_df1['Z-Score_ทั้งสินค้า']>1.3) , 'สถานะ_'] = 'ผิดปกติ'
# result_df1.loc[(result_df1['สถานะพิเศษ'] == 'ผิดปกติ5') & (result_df1['Z-Score_ทั้งสินค้า']>1.5) , 'สถานะ_'] = 'ผิดปกติ'

# # result_df1.loc[result_df1['สถานะ'].isna(), 'สถานะ'] = result_df1.loc[result_df1['สถานะ'].isna(), 'สถานะ_']

# condition = result_df1['Report>=2'].isin(['', 'ยังว่าง?', None])
# result_df1.loc[condition, 'Report>=2'] = result_df1.loc[condition, 'สถานะ_']

# # result_df1 = result_df1.drop(columns=["Sum_group", "percentage", "สถานะพิเศษ", "สถานะ_"])
# result_df1.to_clipboard()
# # df_shop_cj
# # Excel_Condition(df_shop_cj, filename +"_ตรวจราคา__2.xlsx")
# Excel_Condition2(result_df1 ,"Series2.xlsx")


In [ ]:
len(result_df2)

In [ ]:
# data_mouthG = data_mouthG.copy()
# Replace explicit zeros with NaN first
rdf = result_df2.copy()
import numpy as np
# data_mouthG['AVG'] = data_mouthG['AVG'].replace(0, np.nan)
# data_mouthG['REL'] = data_mouthG['REL'].replace(0, np.nan)

# Filter the entire DataFrame to keep only valid rows
rdf = rdf[rdf['AVG'].notna() & rdf['REL'].notna()].copy()
# 3600001	3600003	4210001	4210003

# rdf = rdf[rdf['CODE_7_DIGITS']=='4210001']

# Final output
rdf.reset_index(drop=True)
# display(rdf)
# (Your calculation steps remain the same)


mask = rdf['ประเภทร้านค้า'] != 'ร้านอื่นๆ'
big = (mask).sum()
big_link1 = ((mask) & (rdf['LINK'] == 1)).sum()
big_link2 = ((mask) & (rdf['LINK'] == 2)).sum()
big_link3 = ((mask) & (rdf['LINK'] == 3)).sum()
Rel_plus_Big =  ((mask)& (rdf['REL']>100)).sum()
Rel_equal_Big = ((mask)& (rdf['REL']==100)).sum()
Rel_minus_Big = ((mask)& (rdf['REL']<100)).sum()

rdf['ผลต่างราคากับราคากลาง']=abs(rdf['ราคากลาง']-rdf['AVG'])
mask = rdf['ผลต่างราคากับราคากลาง'] >= 0
mask_right = rdf['ผลต่างราคากับราคากลาง'] == 0
mask_wrong = rdf['ผลต่างราคากับราคากลาง'] > 0

master_price = (mask).sum()

master_price_right = (mask_right).sum()
Rel_plus =  ((mask_right)& (rdf['REL']>100)).sum()
Rel_equal = ((mask_right)& (rdf['REL']==100)).sum()
Rel_minus = ((mask_right)& (rdf['REL']<100)).sum()

master_price_wrong = (mask_wrong).sum()
Rel_plus_wrong=  ((mask_wrong )& (rdf['REL']>100)).sum()
Rel_equal_wrong = ((mask_wrong )& (rdf['REL']==100)).sum()
Rel_minus_wrong = ((mask_wrong )& (rdf['REL']<100)).sum()

Normal_shop = ((rdf['ประเภทร้านค้า'] != 'ร้านอื่นๆ')&(~mask )).sum()
Rel_plus_Normal=  ((rdf['ประเภทร้านค้า'] != 'ร้านอื่นๆ')&(~mask )& (rdf['REL']>100)).sum()
Rel_equal_Normal = ((rdf['ประเภทร้านค้า'] != 'ร้านอื่นๆ')&(~mask )& (rdf['REL']==100)).sum()
Rel_minus_Normal = ((rdf['ประเภทร้านค้า'] != 'ร้านอื่นๆ')&(~mask )& (rdf['REL']<100)).sum()


mask = rdf['ประเภทร้านค้า'] == 'ร้านอื่นๆ'
small = (mask).sum()
small_link1 = ((mask) & (rdf['LINK'] == 1)).sum()
small_link2 = ((mask) & (rdf['LINK'] == 2)).sum()
small_link3 = ((mask) & (rdf['LINK'] == 3)).sum()

Rel_plus_small =  ((mask)& (rdf['REL']>100)).sum()
Rel_equal_small = ((mask)& (rdf['REL']==100)).sum()
Rel_minus_small = ((mask)& (rdf['REL']<100)).sum()

# ⭐️ Print using a Pandas Series
results = pd.Series({
    'Toal ร้านทั้งหมด': big+small,
    '-------------' : '',
    'Total ModernTrade': big,
    '==> ModernTrade (LINK=1)': big_link1,
    '==> ModernTrade (LINK=2)': big_link2,
    '==> ModernTrade (LINK=3)': big_link3,
    '===> sumสินค้าเดิมไม่เปลี่ยนแปลง_MT': big-sum([big_link1, big_link2, big_link3]),
    '===> sumเปลี่ยนสินค้าหรือเปลี่ยนร้าน_MT': sum([big_link1, big_link2, big_link3]),


    '==>>>>> Rel_Big > 100': Rel_plus_Big,
    '==>>>>> Rel_Big = 100': Rel_equal_Big ,
    '==>>>>> Rel_Big < 100': Rel_minus_Big,   
    '------------------' : '',
    'Total ModernTrade ราคากลาง': master_price,
    '==> ราคากลางถูกต้อง': master_price_right,
    '==>>>>> Rel_ราคากลาง_Big > 100': Rel_plus,
    '==>>>>> Rel_ราคากลาง_Big = 100': Rel_equal,
    '==>>>>> Rel_ราคากลาง_Big < 100': Rel_minus,

    '==>Total ModernTrade ราคากลางไม่ถูกต้อง': master_price_wrong,
    '==>>>>> Rel_ราคากลางผิด_Big > 100': Rel_plus_wrong,
    '==>>>>> Rel_ราคากลางผิด_Big = 100': Rel_equal_wrong,
    '==>>>>> Rel_ราคากลางผิด_Big< 100': Rel_minus_wrong,
    
    'Total ModernTrade ไม่ใช่ราคากลาง_Big': Normal_shop,
    '==>>>>> Rel_ไม่ใช่ราคากลาง_Big > 100': Rel_plus_Normal,
    '==>>>>> Rel_ไม่ใช่ราคากลาง_Big = 100': Rel_equal_Normal,
    '==>>>>> Rel_ไม่ใช่ราคากลาง_Big < 100': Rel_minus_Normal,
    '--------------------' : '',
    'Total ร้านอื่นๆ': small,
    '==> ร้านอื่นๆ (LINK=1)': small_link1,
    '==> ร้านอื่นๆ (LINK=2)': small_link2,
    '==> ร้านอื่นๆ (LINK=3)': small_link3, 
    '===> sumสินค้าเดิมไม่เปลี่ยนแปลง_ร้านอื่นๆ ': small-sum([small_link1, small_link2, small_link3]),
    '===> sumเปลี่ยนสินค้าหรือเปลี่ยนร้าน_ร้านอื่นๆ ': sum([small_link1, small_link2, small_link3]),    

    # 'Total Rel_Normal_small': Normal_shop,
    '==>>>>> Rel_small > 100': Rel_plus_small,
    '==>>>>> Rel_small = 100': Rel_equal_small ,
    '==>>>>> Rel_small < 100': Rel_minus_small,  

    'ไม่ใช่ราคากลางทั้งหมด' : Rel_plus_Normal+Rel_plus_small+Rel_equal_Normal+Rel_equal_small+Rel_minus_Normal+Rel_minus_small,
    '==>>>>> Rel > 100':Rel_plus_Normal+Rel_plus_small,
    '==>>>>> Rel= 100':Rel_equal_Normal+Rel_equal_small,
    '==>>>>> Rel < 100':Rel_minus_Normal+Rel_minus_small

    # 'ไม่ใช่ราคากลาง': non_master_price 
})

# mask = data_mouthG['ราคากลาง'] >= 0
# master_price = (mask).sum()
# # master_price_right = (mask & (data_mouthG['ราคากลาง'] == data_mouthG['AVG'])).sum()
# # non_master_price_right = (mask & (data_mouthG['ราคากลาง'] != data_mouthG['AVG'])).sum()
# mark_right = mask & ((data_mouthG['ราคากลาง'] == data_mouthG['AVG'])&(data_mouthG['REL'] == data_mouthG['REL_ฐานนิยม'])) 
# master_price_right = (mark_right).sum()

# mark_wrong = mask & ((data_mouthG['ราคากลาง'] != data_mouthG['AVG'])|(data_mouthG['REL'] != data_mouthG['REL_ฐานนิยม'])) 
# master_price_wrong = (mark_wrong).sum()

# mask_mini = data_mouthG['REL'] 
# Rel_plus =  (mark_right & (mask_mini >100)).sum()
# Rel_equal = (mark_right & (mask_mini==100)).sum()
# Rel_minus = (mark_right & (mask_mini <100)).sum()

# Rel_plus_wrong=  (mark_wrong & (mask_mini >100)).sum()
# Rel_equal_wrong = (mark_wrong & (mask_mini==100)).sum()
# Rel_minus_wrong = (mark_wrong & (mask_mini <100)).sum()


# Normal_shop = (~mask).sum()
# Rel_plus_Normal=  (~mask & (mask_mini >100)).sum()
# Rel_equal_Normal = (~mask & (mask_mini==100)).sum()
# Rel_minus_Normal = (~mask & (mask_mini <100)).sum()


results.to_clipboard()
results


In [ ]:
# data_mouthG['ผลต่างราคากับราคากลาง']=abs(data_mouthG['ราคากลาง']-data_mouthG['AVG'])
# len(data_mouthG[data_mouthG['ผลต่างราคากับราคากลาง']==0])

1071+707

In [ ]:

# RM = Real_master[['รหัส','ผู้ดูแล']].copy()
# RM = RM.rename(columns={'รหัส':'CODE_7_DIGITS'})
# RM["CODE_7_DIGITS"] = RM["CODE_7_DIGITS"].astype(str).str.zfill(7)
# data_mouthG = data_mouthG.merge(RM, on =['CODE_7_DIGITS'], how = 'left')

# data_mouthG
result_df.to_clipboard()

In [ ]:
result_df

In [ ]:
Final_Data = result_df.copy()
Final_Data = Final_Data[Final_Data['LINK']==0]
# rdf = rdf[rdf['CODE_7_DIGITS']=='4210003']
# ,'จำนวนร้าน(ประเภทร้านค้า)'
Final_Data = Final_Data[['CODE_7_DIGITS','DESCRIPTION','Band','ปริมาณ','ประเภทร้านค้า','ราคากลาง','AVG','REL','AVG_ฐานนิยม_ใหม่','REL_ฐานนิยม_ใหม่']]


Final_Data ['Band'] = Final_Data ['Band'].apply(
    lambda x: "ตราไฟน์ไลน์" if isinstance(x, str) and ("ตราFineline" in x or "ตราไฟน์ไลน์" in x) 
              else "ตราBeNice" if isinstance(x, str) and ("ตราBe" in x)
              else "Seven Eleven" if isinstance(x, str) and ("อีเลฟเว่น" in x or "ELEVEN" in x or "7 - 11" in x or "7-11" in x or "7-ELEVEN" in x or "เซเว่น" in x or "7-Eleven" in x or "7 - eleven" in x or "7- eleven" in x or "7 SEVENELEVAN" in x or "seven" in x or "SEVEN" in x or "Eleven" in x or "เซ่เว่น" in x) 
              else "ตราไฟน์ไลน์" if isinstance(x, str) and ("ตราไฟน์ไลน์" in x or "ตราไฟไลน์" in x)
              else "ตราแอคแทค" if isinstance(x, str) and ("ตราแอคแทค" in x or "ตราแอทแทค" in x)
              else "ตราซัลซิล" if isinstance(x, str) and ("ตราซัลซิล" in x or "ตราซันซิล" in x)
              else x
)


Final_Data = Final_Data[Final_Data['ประเภทร้านค้า']!='ร้านอื่นๆ']
Final_Data = Final_Data[Final_Data['ราคากลาง']>=0.01]
Final_Data = Final_Data.sort_values(by = ['Band','ประเภทร้านค้า','ปริมาณ'],ascending=[True,True,True])


Final_Data['จำนวน'] = Final_Data.groupby(['CODE_7_DIGITS','Band','ปริมาณ','ประเภทร้านค้า'])['ประเภทร้านค้า'].transform('count')

Final_Data['ผลต่างAVG'] = abs(Final_Data['AVG']-Final_Data['AVG_ฐานนิยม_ใหม่'])
Final_Data['ผลต่างREL'] = abs(Final_Data['REL']-Final_Data['REL_ฐานนิยม_ใหม่'])




# Final_Data['MAPPING'] = (
#     Final_Data['Band'].astype(str) + '|' +
#     Final_Data['ปริมาณ'].astype(str) + '|' +
#     Final_Data['ประเภทร้านค้า'].astype(str) + '|' +
#     Final_Data['AVG'].astype(str) + '|' +
#     Final_Data['REL'].astype(str)
# )

# Full = Final_Data[['CODE_7_DIGITS','DESCRIPTION','Band','ปริมาณ','ประเภทร้านค้า','จำนวน','ราคาและ relative ตรงกับราคากลาง','AVG','REL','AVG_ฐานนิยม_ใหม่','REL_ฐานนิยม_ใหม่']]
# Full = Full.sort_values(by = ['CODE_7_DIGITS','Band','ปริมาณ','ประเภทร้านค้า'],ascending=[True,True,False,True])
# Full = Full.reset_index(drop=True)

# เก็บเฉพาะแถวที่ผลต่างทั้งสอง = 0
Final_Data = Final_Data[
    (Final_Data['ผลต่างAVG'] == 0) &
    (Final_Data['ผลต่างREL'] == 0)
].reset_index(drop=True)

Final_Data['ราคาและ relative ตรงกับราคากลาง'] = Final_Data.groupby(['CODE_7_DIGITS','Band','ปริมาณ','ประเภทร้านค้า'])['ประเภทร้านค้า'].transform('count')


# Final_Data['MAPPING2'] = Final_Data.groupby(['MAPPING'])['MAPPING'].transform('count')

Final_Data = Final_Data[['CODE_7_DIGITS','Band','ปริมาณ','ประเภทร้านค้า','จำนวน','ราคาและ relative ตรงกับราคากลาง']]
Final_Data = Final_Data.sort_values(by = ['CODE_7_DIGITS','Band','ปริมาณ','ประเภทร้านค้า'],ascending=[True,True,True,True])

Final_Data = Final_Data.drop_duplicates()

# Final_Data['นับจำนวนร้าน2'] = Final_Data.groupby(['DESCRIPTION','ประเภทร้านค้า'])['ประเภทร้านค้า'].transform('count')
Final_Data = Final_Data.reset_index(drop=True)

Final_Data.to_clipboard()




In [ ]:
result_df.columns
x

In [ ]:
Final_Data = result_df.copy()
# rdf = rdf[rdf['CODE_7_DIGITS']=='4210003']
# ,'จำนวนร้าน(ประเภทร้านค้า)'
Final_Data = Final_Data[Final_Data['LINK']==0]
Final_Data = Final_Data[['CODE_7_DIGITS','DESCRIPTION','GROUP', 'LINK','Band','ปริมาณ','ประเภทร้านค้า','ราคากลาง','AVG','REL','AVG_ฐานนิยม_ใหม่','REL_ฐานนิยม_ใหม่']]


Final_Data ['Band'] = Final_Data ['Band'].apply(
    lambda x: "ตราไฟน์ไลน์" if isinstance(x, str) and ("ตราFineline" in x or "ตราไฟน์ไลน์" in x) 
              else "ตราBeNice" if isinstance(x, str) and ("ตราBe" in x)
              else "Seven Eleven" if isinstance(x, str) and ("อีเลฟเว่น" in x or "ELEVEN" in x or "7 - 11" in x or "7-11" in x or "7-ELEVEN" in x or "เซเว่น" in x or "7-Eleven" in x or "7 - eleven" in x or "7- eleven" in x or "7 SEVENELEVAN" in x or "seven" in x or "SEVEN" in x or "Eleven" in x or "เซ่เว่น" in x) 
              else "ตราไฟน์ไลน์" if isinstance(x, str) and ("ตราไฟน์ไลน์" in x or "ตราไฟไลน์" in x)
              else "ตราแอคแทค" if isinstance(x, str) and ("ตราแอคแทค" in x or "ตราแอทแทค" in x)
              else "ตราซัลซิล" if isinstance(x, str) and ("ตราซัลซิล" in x or "ตราซันซิล" in x)
              else x
)


Final_Data = Final_Data[Final_Data['ประเภทร้านค้า']!='ร้านอื่นๆ']
Final_Data = Final_Data[Final_Data['ราคากลาง']>=0.01]
Final_Data = Final_Data.sort_values(by = ['Band','ประเภทร้านค้า','ปริมาณ'],ascending=[True,True,True])


Final_Data['จำนวน'] = Final_Data.groupby(['CODE_7_DIGITS','Band','ปริมาณ','ประเภทร้านค้า'])['ประเภทร้านค้า'].transform('count')

Final_Data['ผลต่างAVG'] = abs(Final_Data['AVG']-Final_Data['AVG_ฐานนิยม_ใหม่'])
Final_Data['ผลต่างREL'] = abs(Final_Data['REL']-Final_Data['REL_ฐานนิยม_ใหม่'])



Final_Data.to_clipboard()




In [ ]:
Full.groupby(key_cols)['จำนวน'].first()


In [ ]:
# grouped_avg_list = (df[mask].groupby(['DESCRIPTION', 'ประเภทร้านค้า'])['AVG'].apply(clean_avg_list).reset_index().rename(columns={'AVG': 'ราคาแนะนำ'}))
# df = df.merge(grouped_avg_list, on=['DESCRIPTION', 'ประเภทร้านค้า'], how='left')

# def format_avg_list(val):
#     if isinstance(val, list):
#         return ', '.join(str(int(v)) if v == int(v) else str(v) for v in val)
#     return val  # สำหรับ NaN หรือค่าอื่น

# df['ราคาแนะนำ'] = df['ราคาแนะนำ'].apply(format_avg_list)
def get_mode_as_list_or_single(series):
    modes = series.mode()
    if modes.empty:
        return None
    elif len(modes) > 1:
        return modes.tolist() 
    else:
        return modes.iloc[0] 

# Apply the aggregation
y = data_mouthG.groupby(['COMMODITY_CODE', 'ประเภทร้านค้า'])['REL'].agg(get_mode_as_list_or_single).reset_index()
y.to_clipboard(index=False)

        # Return the single mode value

y = data_mouthG.groupby(['COMMODITY_CODE','ประเภทร้านค้า'])['REL'].agg(lambda x: x.mode().mean() if len(x.mode()) > 1 else (x.mode().iloc[0] if not x.mode().empty else None)).reset_index()
y.rename(columns={'REL': 'REL_ฐานนิยม'}, inplace=True)
data_mouthGx = data_mouthG.merge(y, on=['COMMODITY_CODE','ประเภทร้านค้า'], how='left')
data_mouthGx.to_clipboard(index=False)

In [ ]:
def get_mode_as_list_or_single(series):
    modes = series.mode()
    if modes.empty:
        return None
    elif len(modes) > 1:
        return modes.tolist() 
    else:
        return modes.iloc[0] 

# Apply the aggregation
y = data_mouthG.groupby(['COMMODITY_CODE', 'ประเภทร้านค้า'])['REL'].agg(get_mode_as_list_or_single).reset_index()
y.to_clipboard(index=False)

In [ ]:

# Excel_Condition2(data_mouthGx.head(1000),"SeriesX.xlsx")


